In [1]:
# Cell 1 — Stage 4 clean project setup

from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb

from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, find_peaks

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# PICS PhysioNet configuration
# ------------------------------------------------------------

PICS_DB = "picsdb"
PICS_VERSION = "1.0.0"

# ------------------------------------------------------------
# Final Stage 4 configuration
# ------------------------------------------------------------

PRECURSOR_DURATION = 15.0
FILTER_CONTEXT = 60.0

ECG_LOW = 0.5
ECG_HIGH = 40.0
ECG_FILTER_ORDER = 4

NOTCH_FREQ = 50.0
NOTCH_Q = 30.0

RESP_LOW = 0.03
RESP_HIGH = 2.0
RESP_FILTER_ORDER = 4

ECG_MIN_PEAK_DISTANCE = 0.30
RR_MIN = 0.30
RR_MAX = 2.00

RESP_MIN_PEAK_DISTANCE = 0.80
RESP_INTERVAL_MIN = 0.80
RESP_INTERVAL_MAX = 10.0

print("=" * 70)
print("STAGE 4 — CLEAN PRECURSOR/RISK DATASET REBUILD")
print("=" * 70)

print()
print("Project root       :", PROJECT_ROOT)
print("PICS database      :", PICS_DB)
print("PICS version       :", PICS_VERSION)
print("Random seed        :", RANDOM_SEED)

print()
print("Window configuration:")
print("Precursor duration :", PRECURSOR_DURATION, "s")
print("Filtering context  :", FILTER_CONTEXT, "s each side")

print()
print("ECG filter         :", f"{ECG_LOW}–{ECG_HIGH} Hz, order {ECG_FILTER_ORDER}")
print("ECG notch          :", f"{NOTCH_FREQ} Hz, Q={NOTCH_Q}")

print()
print("Respiration filter :", f"{RESP_LOW}–{RESP_HIGH} Hz, order {RESP_FILTER_ORDER}")

print()
print("=" * 70)
print("PASS: CLEAN STAGE 4 SETUP COMPLETE")
print("=" * 70)

STAGE 4 — CLEAN PRECURSOR/RISK DATASET REBUILD

Project root       : c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor
PICS database      : picsdb
PICS version       : 1.0.0
Random seed        : 42

Window configuration:
Precursor duration : 15.0 s
Filtering context  : 60.0 s each side

ECG filter         : 0.5–40.0 Hz, order 4
ECG notch          : 50.0 Hz, Q=30.0

Respiration filter : 0.03–2.0 Hz, order 4

PASS: CLEAN STAGE 4 SETUP COMPLETE


In [2]:
# Cell 2 — Verify PICS ECG and respiration records

print("=" * 70)
print("PICS MULTI-INFANT RECORD VERIFICATION")
print("=" * 70)

pics_inventory = []

for infant in range(1, 11):

    ecg_name = f"infant{infant}_ecg"
    resp_name = f"infant{infant}_resp"

    # Read headers only — do NOT download the full signals
    ecg_header = wfdb.rdheader(
        ecg_name,
        pn_dir=f"{PICS_DB}/{PICS_VERSION}"
    )

    resp_header = wfdb.rdheader(
        resp_name,
        pn_dir=f"{PICS_DB}/{PICS_VERSION}"
    )

    ecg_fs = float(ecg_header.fs)
    resp_fs = float(resp_header.fs)

    ecg_duration = float(ecg_header.sig_len) / ecg_fs
    resp_duration = float(resp_header.sig_len) / resp_fs

    pics_inventory.append({
        "infant": infant,
        "ecg_fs": ecg_fs,
        "resp_fs": resp_fs,
        "ecg_duration_s": ecg_duration,
        "resp_duration_s": resp_duration,
        "ecg_channels": ecg_header.n_sig,
        "resp_channels": resp_header.n_sig
    })

pics_inventory = pd.DataFrame(pics_inventory)

print()
print(pics_inventory.to_string(index=False))

print()
print("=" * 70)

assert len(pics_inventory) == 10
assert set(pics_inventory["infant"]) == set(range(1, 11))
assert np.all(pics_inventory["ecg_fs"] > 0)
assert np.all(pics_inventory["resp_fs"] > 0)
assert np.all(pics_inventory["ecg_duration_s"] > 0)
assert np.all(pics_inventory["resp_duration_s"] > 0)

print("PASS: ALL 10 PICS INFANTS VERIFIED")
print("STATUS: DEVELOPMENT DATASET")
print("=" * 70)

PICS MULTI-INFANT RECORD VERIFICATION

 infant  ecg_fs  resp_fs  ecg_duration_s  resp_duration_s  ecg_channels  resp_channels
      1   250.0    500.0      164208.764        164244.00             1              1
      2   500.0     50.0      157832.000        157828.66             1              1
      3   500.0     50.0      157369.228        157365.92             1              1
      4   500.0     50.0      168416.000        168416.00             1              1
      5   250.0     50.0      175514.892        179111.96             1              1
      6   500.0     50.0      174984.394        174981.14             1              1
      7   500.0     50.0       73209.000         73205.76             1              1
      8   500.0     50.0       88576.000         88572.26             1              1
      9   500.0     50.0      253138.176        253135.68             1              1
     10   500.0     50.0      170166.508        170163.08             1              1

PAS

In [3]:
# Cell 3 — Freeze the five development precursor events

print("=" * 70)
print("STAGE 4 — PRECURSOR EVENT DEFINITION")
print("=" * 70)

# Five project-derived neonatal event candidates selected during
# Stage 3 / Stage 3b validation for precursor development.
#
# IMPORTANT:
# These are NOT clinical apnea ground truth.
# They are project-derived event candidates used for development.

PRECURSOR_EVENTS = pd.DataFrame([
    {
        "infant": 1,
        "event_number": 2,
        "event_time_s": 3456.512
    },
    {
        "infant": 3,
        "event_number": 2,
        "event_time_s": 8143.846
    },
    {
        "infant": 3,
        "event_number": 3,
        "event_time_s": 9427.300
    },
    {
        "infant": 9,
        "event_number": 1,
        "event_time_s": 6900.050
    },
    {
        "infant": 9,
        "event_number": 2,
        "event_time_s": 10072.600
    }
])

PRECURSOR_DURATION_S = 15.0

PRECURSOR_EVENTS["precursor_start_s"] = (
    PRECURSOR_EVENTS["event_time_s"] - PRECURSOR_DURATION_S
)

PRECURSOR_EVENTS["precursor_end_s"] = (
    PRECURSOR_EVENTS["event_time_s"]
)

PRECURSOR_EVENTS["window_type"] = "precursor"
PRECURSOR_EVENTS["risk_label"] = 1

print()
print(PRECURSOR_EVENTS.to_string(index=False))

print()
print("=" * 70)

# Structural checks
assert len(PRECURSOR_EVENTS) == 5
assert set(PRECURSOR_EVENTS["infant"]) == {1, 3, 9}
assert list(PRECURSOR_EVENTS["event_number"]) == [2, 2, 3, 1, 2]

durations = (
    PRECURSOR_EVENTS["precursor_end_s"]
    - PRECURSOR_EVENTS["precursor_start_s"]
)

assert np.allclose(durations, PRECURSOR_DURATION_S)
assert np.all(PRECURSOR_EVENTS["risk_label"] == 1)
assert np.all(PRECURSOR_EVENTS["window_type"] == "precursor")

print("Number of precursor events :", len(PRECURSOR_EVENTS))
print("Unique infants             :", PRECURSOR_EVENTS["infant"].nunique())
print("Precursor duration         :", PRECURSOR_DURATION_S, "s")
print()
print("PASS: FIVE PRECURSOR EVENTS FROZEN")
print("STATUS: DEVELOPMENT DATASET — NOT CLINICAL APNEA GROUND TRUTH")
print("=" * 70)

STAGE 4 — PRECURSOR EVENT DEFINITION

 infant  event_number  event_time_s  precursor_start_s  precursor_end_s window_type  risk_label
      1             2      3456.512           3441.512         3456.512   precursor           1
      3             2      8143.846           8128.846         8143.846   precursor           1
      3             3      9427.300           9412.300         9427.300   precursor           1
      9             1      6900.050           6885.050         6900.050   precursor           1
      9             2     10072.600          10057.600        10072.600   precursor           1

Number of precursor events : 5
Unique infants             : 3
Precursor duration         : 15.0 s

PASS: FIVE PRECURSOR EVENTS FROZEN
STATUS: DEVELOPMENT DATASET — NOT CLINICAL APNEA GROUND TRUTH


In [4]:
# Cell 4 — Extract the five 15-second precursor windows

print("=" * 70)
print("PRECURSOR WINDOW EXTRACTION")
print("=" * 70)

precursor_windows = []

for _, event in PRECURSOR_EVENTS.iterrows():

    infant = int(event["infant"])
    event_number = int(event["event_number"])
    event_time = float(event["event_time_s"])
    start_time = float(event["precursor_start_s"])
    end_time = float(event["precursor_end_s"])

    ecg_fs = float(
        pics_inventory.loc[
            pics_inventory["infant"] == infant,
            "ecg_fs"
        ].iloc[0]
    )

    resp_fs = float(
        pics_inventory.loc[
            pics_inventory["infant"] == infant,
            "resp_fs"
        ].iloc[0]
    )

    # Read exactly the precursor interval.
    ecg_record = wfdb.rdrecord(
        f"infant{infant}_ecg",
        pn_dir=f"{PICS_DB}/{PICS_VERSION}",
        sampfrom=int(round(start_time * ecg_fs)),
        sampto=int(round(end_time * ecg_fs))
    )

    resp_record = wfdb.rdrecord(
        f"infant{infant}_resp",
        pn_dir=f"{PICS_DB}/{PICS_VERSION}",
        sampfrom=int(round(start_time * resp_fs)),
        sampto=int(round(end_time * resp_fs))
    )

    ecg_signal = np.asarray(ecg_record.p_signal[:, 0], dtype=float)
    resp_signal = np.asarray(resp_record.p_signal[:, 0], dtype=float)

    expected_ecg_samples = int(round(PRECURSOR_DURATION_S * ecg_fs))
    expected_resp_samples = int(round(PRECURSOR_DURATION_S * resp_fs))

    actual_ecg_duration = len(ecg_signal) / ecg_fs
    actual_resp_duration = len(resp_signal) / resp_fs

    assert len(ecg_signal) == expected_ecg_samples
    assert len(resp_signal) == expected_resp_samples

    assert np.all(np.isfinite(ecg_signal))
    assert np.all(np.isfinite(resp_signal))

    precursor_windows.append({
        "infant": infant,
        "event_number": event_number,
        "event_time_s": event_time,
        "precursor_start_s": start_time,
        "precursor_end_s": end_time,
        "ecg_fs": ecg_fs,
        "resp_fs": resp_fs,
        "ecg_samples": len(ecg_signal),
        "resp_samples": len(resp_signal),
        "ecg_duration_s": actual_ecg_duration,
        "resp_duration_s": actual_resp_duration,
        "ecg_signal": ecg_signal,
        "resp_signal": resp_signal,
        "risk_label": 1,
        "window_type": "precursor"
    })

    print(
        f"Infant {infant} | "
        f"Event {event_number} | "
        f"{start_time:.3f}–{end_time:.3f} s | "
        f"ECG {len(ecg_signal)} | "
        f"RESP {len(resp_signal)} | PASS"
    )

print()
print("=" * 70)

assert len(precursor_windows) == 5

print("Total precursor windows :", len(precursor_windows))
print("Window duration         :", PRECURSOR_DURATION_S, "s")
print()
print("PASS: ALL FIVE PRECURSOR WINDOWS EXTRACTED")
print("STATUS: DEVELOPMENT / RAW WINDOW EXTRACTION")
print("=" * 70)

PRECURSOR WINDOW EXTRACTION
Infant 1 | Event 2 | 3441.512–3456.512 s | ECG 3750 | RESP 7500 | PASS
Infant 3 | Event 2 | 8128.846–8143.846 s | ECG 7500 | RESP 750 | PASS
Infant 3 | Event 3 | 9412.300–9427.300 s | ECG 7500 | RESP 750 | PASS
Infant 9 | Event 1 | 6885.050–6900.050 s | ECG 7500 | RESP 750 | PASS
Infant 9 | Event 2 | 10057.600–10072.600 s | ECG 7500 | RESP 750 | PASS

Total precursor windows : 5
Window duration         : 15.0 s

PASS: ALL FIVE PRECURSOR WINDOWS EXTRACTED
STATUS: DEVELOPMENT / RAW WINDOW EXTRACTION


In [5]:
# Cell 4 — Extract the five 15-second precursor windows

print("=" * 70)
print("PRECURSOR WINDOW EXTRACTION")
print("=" * 70)

precursor_windows = []

for _, event in PRECURSOR_EVENTS.iterrows():

    infant = int(event["infant"])
    event_number = int(event["event_number"])
    event_time = float(event["event_time_s"])
    start_time = float(event["precursor_start_s"])
    end_time = float(event["precursor_end_s"])

    ecg_fs = float(
        pics_inventory.loc[
            pics_inventory["infant"] == infant,
            "ecg_fs"
        ].iloc[0]
    )

    resp_fs = float(
        pics_inventory.loc[
            pics_inventory["infant"] == infant,
            "resp_fs"
        ].iloc[0]
    )

    # Read exactly the precursor interval.
    ecg_record = wfdb.rdrecord(
        f"infant{infant}_ecg",
        pn_dir=f"{PICS_DB}/{PICS_VERSION}",
        sampfrom=int(round(start_time * ecg_fs)),
        sampto=int(round(end_time * ecg_fs))
    )

    resp_record = wfdb.rdrecord(
        f"infant{infant}_resp",
        pn_dir=f"{PICS_DB}/{PICS_VERSION}",
        sampfrom=int(round(start_time * resp_fs)),
        sampto=int(round(end_time * resp_fs))
    )

    ecg_signal = np.asarray(ecg_record.p_signal[:, 0], dtype=float)
    resp_signal = np.asarray(resp_record.p_signal[:, 0], dtype=float)

    expected_ecg_samples = int(round(PRECURSOR_DURATION_S * ecg_fs))
    expected_resp_samples = int(round(PRECURSOR_DURATION_S * resp_fs))

    actual_ecg_duration = len(ecg_signal) / ecg_fs
    actual_resp_duration = len(resp_signal) / resp_fs

    assert len(ecg_signal) == expected_ecg_samples
    assert len(resp_signal) == expected_resp_samples

    assert np.all(np.isfinite(ecg_signal))
    assert np.all(np.isfinite(resp_signal))

    precursor_windows.append({
        "infant": infant,
        "event_number": event_number,
        "event_time_s": event_time,
        "precursor_start_s": start_time,
        "precursor_end_s": end_time,
        "ecg_fs": ecg_fs,
        "resp_fs": resp_fs,
        "ecg_samples": len(ecg_signal),
        "resp_samples": len(resp_signal),
        "ecg_duration_s": actual_ecg_duration,
        "resp_duration_s": actual_resp_duration,
        "ecg_signal": ecg_signal,
        "resp_signal": resp_signal,
        "risk_label": 1,
        "window_type": "precursor"
    })

    print(
        f"Infant {infant} | "
        f"Event {event_number} | "
        f"{start_time:.3f}–{end_time:.3f} s | "
        f"ECG {len(ecg_signal)} | "
        f"RESP {len(resp_signal)} | PASS"
    )

print()
print("=" * 70)

assert len(precursor_windows) == 5

print("Total precursor windows :", len(precursor_windows))
print("Window duration         :", PRECURSOR_DURATION_S, "s")
print()
print("PASS: ALL FIVE PRECURSOR WINDOWS EXTRACTED")
print("STATUS: DEVELOPMENT / RAW WINDOW EXTRACTION")
print("=" * 70)

PRECURSOR WINDOW EXTRACTION
Infant 1 | Event 2 | 3441.512–3456.512 s | ECG 3750 | RESP 7500 | PASS
Infant 3 | Event 2 | 8128.846–8143.846 s | ECG 7500 | RESP 750 | PASS
Infant 3 | Event 3 | 9412.300–9427.300 s | ECG 7500 | RESP 750 | PASS
Infant 9 | Event 1 | 6885.050–6900.050 s | ECG 7500 | RESP 750 | PASS
Infant 9 | Event 2 | 10057.600–10072.600 s | ECG 7500 | RESP 750 | PASS

Total precursor windows : 5
Window duration         : 15.0 s

PASS: ALL FIVE PRECURSOR WINDOWS EXTRACTED
STATUS: DEVELOPMENT / RAW WINDOW EXTRACTION


In [6]:
# Cell 5 — Verify precursor signal integrity

print("=" * 70)
print("PRECURSOR SIGNAL INTEGRITY CHECK")
print("=" * 70)

for record in precursor_windows:

    infant = record["infant"]
    event_number = record["event_number"]

    ecg_signal = record["ecg_signal"]
    resp_signal = record["resp_signal"]

    ecg_fs = record["ecg_fs"]
    resp_fs = record["resp_fs"]

    print()
    print(
        f"Infant {infant} | Event {event_number}"
    )

    print(
        f"  ECG  : {len(ecg_signal)} samples | "
        f"{ecg_fs:.1f} Hz | "
        f"{len(ecg_signal) / ecg_fs:.3f} s"
    )

    print(
        f"  RESP : {len(resp_signal)} samples | "
        f"{resp_fs:.1f} Hz | "
        f"{len(resp_signal) / resp_fs:.3f} s"
    )

    print(
        f"  ECG finite  : {np.all(np.isfinite(ecg_signal))}"
    )

    print(
        f"  RESP finite : {np.all(np.isfinite(resp_signal))}"
    )

    assert np.all(np.isfinite(ecg_signal))
    assert np.all(np.isfinite(resp_signal))

    assert np.isclose(
        len(ecg_signal) / ecg_fs,
        PRECURSOR_DURATION_S,
        atol=1 / ecg_fs
    )

    assert np.isclose(
        len(resp_signal) / resp_fs,
        PRECURSOR_DURATION_S,
        atol=1 / resp_fs
    )

print()
print("=" * 70)
print("PASS: ALL PRECURSOR SIGNALS ARE FINITE AND EXACTLY 15 s")
print("STATUS: DEVELOPMENT / SIGNAL INTEGRITY")
print("=" * 70)

PRECURSOR SIGNAL INTEGRITY CHECK

Infant 1 | Event 2
  ECG  : 3750 samples | 250.0 Hz | 15.000 s
  RESP : 7500 samples | 500.0 Hz | 15.000 s
  ECG finite  : True
  RESP finite : True

Infant 3 | Event 2
  ECG  : 7500 samples | 500.0 Hz | 15.000 s
  RESP : 750 samples | 50.0 Hz | 15.000 s
  ECG finite  : True
  RESP finite : True

Infant 3 | Event 3
  ECG  : 7500 samples | 500.0 Hz | 15.000 s
  RESP : 750 samples | 50.0 Hz | 15.000 s
  ECG finite  : True
  RESP finite : True

Infant 9 | Event 1
  ECG  : 7500 samples | 500.0 Hz | 15.000 s
  RESP : 750 samples | 50.0 Hz | 15.000 s
  ECG finite  : True
  RESP finite : True

Infant 9 | Event 2
  ECG  : 7500 samples | 500.0 Hz | 15.000 s
  RESP : 750 samples | 50.0 Hz | 15.000 s
  ECG finite  : True
  RESP finite : True

PASS: ALL PRECURSOR SIGNALS ARE FINITE AND EXACTLY 15 s
STATUS: DEVELOPMENT / SIGNAL INTEGRITY


In [7]:
# Cell 5 — Verify precursor signal integrity

print("=" * 70)
print("PRECURSOR SIGNAL INTEGRITY CHECK")
print("=" * 70)

for record in precursor_windows:

    infant = record["infant"]
    event_number = record["event_number"]

    ecg_signal = record["ecg_signal"]
    resp_signal = record["resp_signal"]

    ecg_fs = record["ecg_fs"]
    resp_fs = record["resp_fs"]

    print()
    print(
        f"Infant {infant} | Event {event_number}"
    )

    print(
        f"  ECG  : {len(ecg_signal)} samples | "
        f"{ecg_fs:.1f} Hz | "
        f"{len(ecg_signal) / ecg_fs:.3f} s"
    )

    print(
        f"  RESP : {len(resp_signal)} samples | "
        f"{resp_fs:.1f} Hz | "
        f"{len(resp_signal) / resp_fs:.3f} s"
    )

    print(
        f"  ECG finite  : {np.all(np.isfinite(ecg_signal))}"
    )

    print(
        f"  RESP finite : {np.all(np.isfinite(resp_signal))}"
    )

    assert np.all(np.isfinite(ecg_signal))
    assert np.all(np.isfinite(resp_signal))

    assert np.isclose(
        len(ecg_signal) / ecg_fs,
        PRECURSOR_DURATION_S,
        atol=1 / ecg_fs
    )

    assert np.isclose(
        len(resp_signal) / resp_fs,
        PRECURSOR_DURATION_S,
        atol=1 / resp_fs
    )

print()
print("=" * 70)
print("PASS: ALL PRECURSOR SIGNALS ARE FINITE AND EXACTLY 15 s")
print("STATUS: DEVELOPMENT / SIGNAL INTEGRITY")
print("=" * 70)

PRECURSOR SIGNAL INTEGRITY CHECK

Infant 1 | Event 2
  ECG  : 3750 samples | 250.0 Hz | 15.000 s
  RESP : 7500 samples | 500.0 Hz | 15.000 s
  ECG finite  : True
  RESP finite : True

Infant 3 | Event 2
  ECG  : 7500 samples | 500.0 Hz | 15.000 s
  RESP : 750 samples | 50.0 Hz | 15.000 s
  ECG finite  : True
  RESP finite : True

Infant 3 | Event 3
  ECG  : 7500 samples | 500.0 Hz | 15.000 s
  RESP : 750 samples | 50.0 Hz | 15.000 s
  ECG finite  : True
  RESP finite : True

Infant 9 | Event 1
  ECG  : 7500 samples | 500.0 Hz | 15.000 s
  RESP : 750 samples | 50.0 Hz | 15.000 s
  ECG finite  : True
  RESP finite : True

Infant 9 | Event 2
  ECG  : 7500 samples | 500.0 Hz | 15.000 s
  RESP : 750 samples | 50.0 Hz | 15.000 s
  ECG finite  : True
  RESP finite : True

PASS: ALL PRECURSOR SIGNALS ARE FINITE AND EXACTLY 15 s
STATUS: DEVELOPMENT / SIGNAL INTEGRITY


In [8]:
# Cell 6 — Self-contained preprocessing of the five precursor windows
#
# This cell intentionally does NOT depend on variables from the old notebook.
# The five development precursor events were frozen in Cell 3.

import numpy as np
import pandas as pd
import wfdb

from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt


# ==============================================================
# CONFIGURATION
# ==============================================================

PICS_DB = "picsdb"
PICS_VERSION = "1.0.0"

PRECURSOR_DURATION = 15.0
FILTER_CONTEXT = 60.0

# ECG
ECG_LOW = 0.5
ECG_HIGH = 40.0
ECG_FILTER_ORDER = 4

# ECG notch
NOTCH_FREQ = 50.0
NOTCH_Q = 30.0

# Respiration
RESP_LOW = 0.03
RESP_HIGH = 2.0
RESP_FILTER_ORDER = 4


# ==============================================================
# FIVE FROZEN DEVELOPMENT PRECURSOR EVENTS
# ==============================================================

precursor_events_clean = pd.DataFrame([
    {
        "infant": 1,
        "event_number": 2,
        "event_time_s": 3456.512
    },
    {
        "infant": 3,
        "event_number": 2,
        "event_time_s": 8143.846
    },
    {
        "infant": 3,
        "event_number": 3,
        "event_time_s": 9427.300
    },
    {
        "infant": 9,
        "event_number": 1,
        "event_time_s": 6900.050
    },
    {
        "infant": 9,
        "event_number": 2,
        "event_time_s": 10072.600
    }
])

precursor_events_clean["precursor_start_s"] = (
    precursor_events_clean["event_time_s"] - PRECURSOR_DURATION
)

precursor_events_clean["precursor_end_s"] = (
    precursor_events_clean["event_time_s"]
)


# ==============================================================
# FILTER FUNCTIONS
# ==============================================================

def filter_ecg(signal, fs):
    
    signal = np.asarray(signal, dtype=float)

    sos = butter(
        ECG_FILTER_ORDER,
        [ECG_LOW, ECG_HIGH],
        btype="bandpass",
        fs=fs,
        output="sos"
    )

    filtered = sosfiltfilt(sos, signal)

    b_notch, a_notch = iirnotch(
        NOTCH_FREQ,
        NOTCH_Q,
        fs=fs
    )

    filtered = filtfilt(
        b_notch,
        a_notch,
        filtered
    )

    return filtered


def filter_respiration(signal, fs):
    
    signal = np.asarray(signal, dtype=float)

    sos = butter(
        RESP_FILTER_ORDER,
        [RESP_LOW, RESP_HIGH],
        btype="bandpass",
        fs=fs,
        output="sos"
    )

    return sosfiltfilt(sos, signal)


# ==============================================================
# PROCESS ALL FIVE WINDOWS
# ==============================================================

processed_precursor_windows = []
preprocessing_failures = []


print("=" * 70)
print("PRECURSOR WINDOW PREPROCESSING")
print("=" * 70)

print()
print("Precursor duration :", PRECURSOR_DURATION, "s")
print("Filtering context  :", FILTER_CONTEXT, "s each side")
print("Total filter span  :", PRECURSOR_DURATION + 2 * FILTER_CONTEXT, "s")

print()

for _, event in precursor_events_clean.iterrows():

    infant = int(event["infant"])
    event_number = int(event["event_number"])

    precursor_start = float(event["precursor_start_s"])
    precursor_end = float(event["precursor_end_s"])

    context_start = precursor_start - FILTER_CONTEXT
    context_end = precursor_end + FILTER_CONTEXT

    try:

        # ----------------------------------------------------------
        # Read headers to obtain authoritative sampling rates
        # ----------------------------------------------------------

        ecg_header = wfdb.rdheader(
            f"infant{infant}_ecg",
            pn_dir=f"{PICS_DB}/{PICS_VERSION}"
        )

        resp_header = wfdb.rdheader(
            f"infant{infant}_resp",
            pn_dir=f"{PICS_DB}/{PICS_VERSION}"
        )

        ecg_fs = float(ecg_header.fs)
        resp_fs = float(resp_header.fs)

        # ----------------------------------------------------------
        # Convert context times to sample indices
        # ----------------------------------------------------------

        ecg_sampfrom = int(round(context_start * ecg_fs))
        ecg_sampto = int(round(context_end * ecg_fs))

        resp_sampfrom = int(round(context_start * resp_fs))
        resp_sampto = int(round(context_end * resp_fs))

        # ----------------------------------------------------------
        # Read ECG context
        # ----------------------------------------------------------

        ecg_record = wfdb.rdrecord(
            f"infant{infant}_ecg",
            pn_dir=f"{PICS_DB}/{PICS_VERSION}",
            sampfrom=ecg_sampfrom,
            sampto=ecg_sampto
        )

        ecg_context = np.asarray(
            ecg_record.p_signal[:, 0],
            dtype=float
        )

        # ----------------------------------------------------------
        # Read respiration context
        # ----------------------------------------------------------

        resp_record = wfdb.rdrecord(
            f"infant{infant}_resp",
            pn_dir=f"{PICS_DB}/{PICS_VERSION}",
            sampfrom=resp_sampfrom,
            sampto=resp_sampto
        )

        resp_context = np.asarray(
            resp_record.p_signal[:, 0],
            dtype=float
        )

        # ----------------------------------------------------------
        # Raw signal integrity
        # ----------------------------------------------------------

        if not np.all(np.isfinite(ecg_context)):
            raise ValueError("ECG context contains NaN/Inf")

        if not np.all(np.isfinite(resp_context)):
            raise ValueError("Respiration context contains NaN/Inf")

        # ----------------------------------------------------------
        # Filter full context
        # ----------------------------------------------------------

        ecg_filtered = filter_ecg(
            ecg_context,
            ecg_fs
        )

        resp_filtered = filter_respiration(
            resp_context,
            resp_fs
        )

        # ----------------------------------------------------------
        # Extract central 15-second window
        # ----------------------------------------------------------

        ecg_keep_start = int(round(FILTER_CONTEXT * ecg_fs))
        ecg_keep_length = int(round(PRECURSOR_DURATION * ecg_fs))
        ecg_keep_end = ecg_keep_start + ecg_keep_length

        resp_keep_start = int(round(FILTER_CONTEXT * resp_fs))
        resp_keep_length = int(round(PRECURSOR_DURATION * resp_fs))
        resp_keep_end = resp_keep_start + resp_keep_length

        ecg_final = ecg_filtered[
            ecg_keep_start:ecg_keep_end
        ]

        resp_final = resp_filtered[
            resp_keep_start:resp_keep_end
        ]

        # ----------------------------------------------------------
        # Expected sample counts
        # ----------------------------------------------------------

        expected_ecg = int(
            round(PRECURSOR_DURATION * ecg_fs)
        )

        expected_resp = int(
            round(PRECURSOR_DURATION * resp_fs)
        )

        # ----------------------------------------------------------
        # Final integrity checks
        # ----------------------------------------------------------

        assert len(ecg_final) == expected_ecg
        assert len(resp_final) == expected_resp

        assert np.all(np.isfinite(ecg_final))
        assert np.all(np.isfinite(resp_final))

        # ----------------------------------------------------------
        # Store processed window
        # ----------------------------------------------------------

        processed_precursor_windows.append({
            "infant": infant,
            "event_number": event_number,
            "event_time_s": event["event_time_s"],
            "precursor_start_s": precursor_start,
            "precursor_end_s": precursor_end,
            "window_type": "precursor",
            "risk_label": 1,

            "ecg_fs": ecg_fs,
            "resp_fs": resp_fs,

            "ecg_signal": ecg_final,
            "resp_signal": resp_final,

            "ecg_samples": len(ecg_final),
            "resp_samples": len(resp_final),

            "filter_context_s": FILTER_CONTEXT
        })

        print(
            f"Infant {infant} | "
            f"Event {event_number} | "
            f"{precursor_start:.3f}–{precursor_end:.3f} s | "
            f"ECG {len(ecg_final)} | "
            f"RESP {len(resp_final)} | PASS"
        )

    except Exception as e:

        preprocessing_failures.append({
            "infant": infant,
            "event_number": event_number,
            "error": str(e)
        })

        print(
            f"Infant {infant} | "
            f"Event {event_number} | FAIL | {e}"
        )


# ==============================================================
# FINAL CHECK
# ==============================================================

print()
print("=" * 70)
print("PREPROCESSING SUMMARY")
print("=" * 70)

print("Expected windows :", len(precursor_events_clean))
print("Processed        :", len(processed_precursor_windows))
print("Failed           :", len(preprocessing_failures))

if preprocessing_failures:
    print()
    print("FAILURES:")
    for failure in preprocessing_failures:
        print(
            f"Infant {failure['infant']} | "
            f"Event {failure['event_number']} | "
            f"{failure['error']}"
        )

    raise RuntimeError(
        "One or more precursor windows failed preprocessing."
    )

assert len(processed_precursor_windows) == 5

print()
print("PASS: ALL FIVE PRECURSOR WINDOWS PREPROCESSED")
print("PASS: EXACT 15 s WINDOWS RETAINED")
print("PASS: ALL ECG AND RESPIRATION SAMPLES ARE FINITE")
print("STATUS: DEVELOPMENT / PREPROCESSING")
print("=" * 70)

PRECURSOR WINDOW PREPROCESSING

Precursor duration : 15.0 s
Filtering context  : 60.0 s each side
Total filter span  : 135.0 s

Infant 1 | Event 2 | 3441.512–3456.512 s | ECG 3750 | RESP 7500 | PASS
Infant 3 | Event 2 | 8128.846–8143.846 s | ECG 7500 | RESP 750 | PASS
Infant 3 | Event 3 | 9412.300–9427.300 s | ECG 7500 | RESP 750 | PASS
Infant 9 | Event 1 | 6885.050–6900.050 s | ECG 7500 | RESP 750 | PASS
Infant 9 | Event 2 | 10057.600–10072.600 s | ECG 7500 | RESP 750 | PASS

PREPROCESSING SUMMARY
Expected windows : 5
Processed        : 5
Failed           : 0

PASS: ALL FIVE PRECURSOR WINDOWS PREPROCESSED
PASS: EXACT 15 s WINDOWS RETAINED
PASS: ALL ECG AND RESPIRATION SAMPLES ARE FINITE
STATUS: DEVELOPMENT / PREPROCESSING


In [9]:
# Cell 7 — Load and verify the 50 frozen control windows
#
# The control windows were already selected and repaired in the previous
# development workflow. This clean notebook reuses that verified artifact
# rather than regenerating controls.

from pathlib import Path
import pandas as pd
import numpy as np


CONTROL_WINDOWS_PATH = (
    PROJECT_ROOT
    / "reports"
    / "pics_final_control_windows.csv"
)

print("=" * 70)
print("FROZEN CONTROL-WINDOW VERIFICATION")
print("=" * 70)

print()
print("Control-window file:")
print(CONTROL_WINDOWS_PATH)

assert CONTROL_WINDOWS_PATH.exists(), (
    f"Control-window file not found: {CONTROL_WINDOWS_PATH}"
)

control_windows_df = pd.read_csv(CONTROL_WINDOWS_PATH)

print()
print("Loaded shape:", control_windows_df.shape)

print()
print("Columns:")
for column in control_windows_df.columns:
    print(" -", column)


# --------------------------------------------------------------
# Required structural columns
# --------------------------------------------------------------

required_columns = [
    "infant",
    "control_start_s",
    "control_end_s",
    "control_duration_s",
    "risk_label",
    "window_type"
]

missing_columns = [
    column
    for column in required_columns
    if column not in control_windows_df.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)


# --------------------------------------------------------------
# Basic dataset checks
# --------------------------------------------------------------

assert len(control_windows_df) == 50

assert set(
    control_windows_df["infant"].astype(int).unique()
) == set(range(1, 11))

assert set(
    control_windows_df["risk_label"].astype(int).unique()
) == {0}

assert set(
    control_windows_df["window_type"].astype(str).unique()
) == {"control"}


# --------------------------------------------------------------
# Exactly five controls per infant
# --------------------------------------------------------------

controls_per_infant = (
    control_windows_df
    .assign(infant=control_windows_df["infant"].astype(int))
    .groupby("infant")
    .size()
    .sort_index()
)

print()
print("Controls per infant:")
print(controls_per_infant.to_string())

assert len(controls_per_infant) == 10
assert (controls_per_infant == 5).all()


# --------------------------------------------------------------
# Duration checks
# --------------------------------------------------------------

control_durations = (
    control_windows_df["control_duration_s"]
    .astype(float)
)

assert np.all(np.isfinite(control_durations))
assert np.allclose(control_durations, 15.0)

print()
print(
    "Control duration:",
    control_durations.min(),
    "to",
    control_durations.max(),
    "s"
)


# --------------------------------------------------------------
# Timing checks
# --------------------------------------------------------------

start_times = control_windows_df["control_start_s"].astype(float)
end_times = control_windows_df["control_end_s"].astype(float)

assert np.all(np.isfinite(start_times))
assert np.all(np.isfinite(end_times))

assert np.all(end_times > start_times)

assert np.allclose(
    end_times - start_times,
    15.0
)


# --------------------------------------------------------------
# Duplicate-window check
# --------------------------------------------------------------

duplicate_count = control_windows_df[
    ["infant", "control_start_s", "control_end_s"]
].duplicated().sum()

print()
print("Duplicate control windows:", duplicate_count)

assert duplicate_count == 0


# --------------------------------------------------------------
# Display the complete control inventory
# --------------------------------------------------------------

print()
print("=" * 70)
print("CONTROL WINDOW INVENTORY")
print("=" * 70)

display(
    control_windows_df[
        [
            "infant",
            "control_start_s",
            "control_end_s",
            "control_duration_s",
            "risk_label",
            "window_type"
        ]
    ].sort_values(
        ["infant", "control_start_s"]
    ).reset_index(drop=True)
)


# --------------------------------------------------------------
# Final status
# --------------------------------------------------------------

print()
print("=" * 70)
print("PASS: 50 CONTROL WINDOWS VERIFIED")
print("PASS: EXACTLY 5 CONTROLS PER INFANT")
print("PASS: ALL CONTROL WINDOWS ARE 15 s")
print("PASS: ALL CONTROL LABELS ARE 0")
print("PASS: NO DUPLICATE CONTROL WINDOWS")
print("STATUS: DEVELOPMENT / FROZEN CONTROL INVENTORY")
print("=" * 70)

FROZEN CONTROL-WINDOW VERIFICATION

Control-window file:
c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_final_control_windows.csv

Loaded shape: (50, 10)

Columns:
 - infant
 - event_number
 - event_time_s
 - precursor_start_s
 - precursor_end_s
 - control_start_s
 - control_end_s
 - control_duration_s
 - risk_label
 - window_type

Controls per infant:
infant
1     5
2     5
3     5
4     5
5     5
6     5
7     5
8     5
9     5
10    5

Control duration: 15.0 to 15.0 s

Duplicate control windows: 0

CONTROL WINDOW INVENTORY


,infant,control_start_s,control_end_s,control_duration_s,risk_label,window_type
0,1,14789.788,14804.788,15.0,0,control
1,1,70419.780,70434.780,15.0,0,control
2,1,71319.780,71334.780,15.0,0,control
3,1,107559.096,107574.096,15.0,0,control
4,1,127371.824,127386.824,15.0,0,control
5,2,14753.164,14768.164,15.0,0,control
6,2,83082.830,83097.830,15.0,0,control
7,2,116128.184,116143.184,15.0,0,control
8,2,120167.988,120182.988,15.0,0,control
9,2,153792.438,153807.438,15.0,0,control



PASS: 50 CONTROL WINDOWS VERIFIED
PASS: EXACTLY 5 CONTROLS PER INFANT
PASS: ALL CONTROL WINDOWS ARE 15 s
PASS: ALL CONTROL LABELS ARE 0
PASS: NO DUPLICATE CONTROL WINDOWS
STATUS: DEVELOPMENT / FROZEN CONTROL INVENTORY


In [10]:
# Cell 8 — Extract and preprocess the 50 control windows
#
# Uses the verified frozen control inventory from Cell 7.
# Applies the SAME preprocessing used for precursor windows:
#
# ECG:
#   0.5–40 Hz Butterworth order 4
#   50 Hz notch, Q=30
#
# Respiration:
#   0.03–2.0 Hz Butterworth order 4
#
# Filtering:
#   60 s context on each side
#   retain central 15 s

print("=" * 70)
print("CONTROL WINDOW EXTRACTION AND PREPROCESSING")
print("=" * 70)

processed_control_windows = []
control_processing_failures = []

print()
print("Total control windows :", len(control_windows_df))
print("Control duration      :", PRECURSOR_DURATION, "s")
print("Filtering context     :", FILTER_CONTEXT, "s each side")
print()


for row_number, row in control_windows_df.iterrows():

    infant = int(row["infant"])

    control_start = float(row["control_start_s"])
    control_end = float(row["control_end_s"])

    context_start = control_start - FILTER_CONTEXT
    context_end = control_end + FILTER_CONTEXT

    try:

        # ----------------------------------------------------------
        # Read authoritative recording headers
        # ----------------------------------------------------------

        ecg_header = wfdb.rdheader(
            f"infant{infant}_ecg",
            pn_dir=f"{PICS_DB}/{PICS_VERSION}"
        )

        resp_header = wfdb.rdheader(
            f"infant{infant}_resp",
            pn_dir=f"{PICS_DB}/{PICS_VERSION}"
        )

        ecg_fs = float(ecg_header.fs)
        resp_fs = float(resp_header.fs)

        # ----------------------------------------------------------
        # Check that context is inside recording
        # ----------------------------------------------------------

        ecg_duration = (
            float(ecg_header.sig_len) / ecg_fs
        )

        resp_duration = (
            float(resp_header.sig_len) / resp_fs
        )

        if context_end > ecg_duration:
            raise ValueError(
                f"ECG context exceeds recording duration "
                f"({context_end:.3f} > {ecg_duration:.3f})"
            )

        if context_end > resp_duration:
            raise ValueError(
                f"RESP context exceeds recording duration "
                f"({context_end:.3f} > {resp_duration:.3f})"
            )

        if context_start < 0:
            raise ValueError(
                f"Negative context start: {context_start:.3f}"
            )

        # ----------------------------------------------------------
        # Convert time to samples
        # ----------------------------------------------------------

        ecg_sampfrom = int(
            round(context_start * ecg_fs)
        )

        ecg_sampto = int(
            round(context_end * ecg_fs)
        )

        resp_sampfrom = int(
            round(context_start * resp_fs)
        )

        resp_sampto = int(
            round(context_end * resp_fs)
        )

        # ----------------------------------------------------------
        # Read ECG context
        # ----------------------------------------------------------

        ecg_record = wfdb.rdrecord(
            f"infant{infant}_ecg",
            pn_dir=f"{PICS_DB}/{PICS_VERSION}",
            sampfrom=ecg_sampfrom,
            sampto=ecg_sampto
        )

        ecg_context = np.asarray(
            ecg_record.p_signal[:, 0],
            dtype=float
        )

        # ----------------------------------------------------------
        # Read respiration context
        # ----------------------------------------------------------

        resp_record = wfdb.rdrecord(
            f"infant{infant}_resp",
            pn_dir=f"{PICS_DB}/{PICS_VERSION}",
            sampfrom=resp_sampfrom,
            sampto=resp_sampto
        )

        resp_context = np.asarray(
            resp_record.p_signal[:, 0],
            dtype=float
        )

        # ----------------------------------------------------------
        # Raw signal checks
        # ----------------------------------------------------------

        if not np.all(np.isfinite(ecg_context)):
            raise ValueError("Non-finite ECG samples")

        if not np.all(np.isfinite(resp_context)):
            raise ValueError("Non-finite respiration samples")

        # ----------------------------------------------------------
        # Filter full context
        # ----------------------------------------------------------

        ecg_filtered = filter_ecg(
            ecg_context,
            ecg_fs
        )

        resp_filtered = filter_respiration(
            resp_context,
            resp_fs
        )

        # ----------------------------------------------------------
        # Extract central 15-second region
        # ----------------------------------------------------------

        ecg_keep_start = int(
            round(FILTER_CONTEXT * ecg_fs)
        )

        ecg_keep_length = int(
            round(PRECURSOR_DURATION * ecg_fs)
        )

        ecg_keep_end = (
            ecg_keep_start + ecg_keep_length
        )

        resp_keep_start = int(
            round(FILTER_CONTEXT * resp_fs)
        )

        resp_keep_length = int(
            round(PRECURSOR_DURATION * resp_fs)
        )

        resp_keep_end = (
            resp_keep_start + resp_keep_length
        )

        ecg_final = ecg_filtered[
            ecg_keep_start:ecg_keep_end
        ]

        resp_final = resp_filtered[
            resp_keep_start:resp_keep_end
        ]

        # ----------------------------------------------------------
        # Expected sample counts
        # ----------------------------------------------------------

        expected_ecg_samples = int(
            round(PRECURSOR_DURATION * ecg_fs)
        )

        expected_resp_samples = int(
            round(PRECURSOR_DURATION * resp_fs)
        )

        # ----------------------------------------------------------
        # Final checks
        # ----------------------------------------------------------

        assert len(ecg_final) == expected_ecg_samples
        assert len(resp_final) == expected_resp_samples

        assert np.all(np.isfinite(ecg_final))
        assert np.all(np.isfinite(resp_final))

        # ----------------------------------------------------------
        # Store processed control
        # ----------------------------------------------------------

        processed_control_windows.append({
            "infant": infant,
            "event_number": np.nan,
            "event_time_s": np.nan,
            "precursor_start_s": np.nan,
            "precursor_end_s": np.nan,

            "control_start_s": control_start,
            "control_end_s": control_end,
            "control_duration_s": control_end - control_start,

            "risk_label": 0,
            "window_type": "control",

            "ecg_fs": ecg_fs,
            "resp_fs": resp_fs,

            "ecg_signal": ecg_final,
            "resp_signal": resp_final,

            "ecg_samples": len(ecg_final),
            "resp_samples": len(resp_final),

            "filter_context_s": FILTER_CONTEXT
        })

        print(
            f"Window {row_number + 1:02d}/50 | "
            f"Infant {infant} | "
            f"{control_start:.3f}–{control_end:.3f} s | "
            f"ECG {len(ecg_final)} | "
            f"RESP {len(resp_final)} | PASS"
        )

    except Exception as e:

        control_processing_failures.append({
            "row": int(row_number),
            "infant": infant,
            "control_start_s": control_start,
            "control_end_s": control_end,
            "error": str(e)
        })

        print(
            f"Window {row_number + 1:02d}/50 | "
            f"Infant {infant} | FAIL | {e}"
        )


# ==============================================================
# FINAL SUMMARY
# ==============================================================

print()
print("=" * 70)
print("CONTROL PREPROCESSING SUMMARY")
print("=" * 70)

print("Expected controls :", len(control_windows_df))
print("Processed         :", len(processed_control_windows))
print("Failed            :", len(control_processing_failures))

if control_processing_failures:

    print()
    print("FAILED CONTROL WINDOWS:")

    for failure in control_processing_failures:
        print(
            f"Row {failure['row']} | "
            f"Infant {failure['infant']} | "
            f"{failure['control_start_s']:.3f}–"
            f"{failure['control_end_s']:.3f} s | "
            f"{failure['error']}"
        )

    raise RuntimeError(
        "Control preprocessing failed. "
        "Do not continue until the failed windows are resolved."
    )

assert len(processed_control_windows) == 50

print()
print("PASS: ALL 50 CONTROL WINDOWS PREPROCESSED")
print("PASS: ALL CONTROL WINDOWS ARE EXACTLY 15 s")
print("PASS: ALL ECG SIGNALS ARE FINITE")
print("PASS: ALL RESPIRATION SIGNALS ARE FINITE")
print("STATUS: DEVELOPMENT / CONTROL PREPROCESSING")
print("=" * 70)

CONTROL WINDOW EXTRACTION AND PREPROCESSING

Total control windows : 50
Control duration      : 15.0 s
Filtering context     : 60.0 s each side

Window 01/50 | Infant 1 | 14789.788–14804.788 s | ECG 3750 | RESP 7500 | PASS


KeyboardInterrupt: 

In [28]:
# ============================================================
# CELL 8F — Inspect Actual Cell 8 Failures
# ============================================================

print("=" * 70)
print("ACTUAL CONTROL PROCESSING FAILURES")
print("=" * 70)

print(
    "Number of processed controls:",
    len(processed_control_windows)
)

print(
    "Number of failed controls:",
    len(control_processing_failures)
)

print("\nFAILED WINDOWS:")

for failure in control_processing_failures:
    print(
        f"Row {failure['row']} | "
        f"Infant {failure['infant']} | "
        f"{failure['control_start_s']:.3f}–"
        f"{failure['control_end_s']:.3f} s | "
        f"ERROR: {failure['error']}"
    )

ACTUAL CONTROL PROCESSING FAILURES
Number of processed controls: 1
Number of failed controls: 0

FAILED WINDOWS:


In [11]:
# ============================================================
# CELL 8-DIAGNOSTIC — Inspect Control Preprocessing Failures
# ============================================================

print("Variables related to control preprocessing failures:")

failure_vars = [
    name for name in globals()
    if "fail" in name.lower()
    or "error" in name.lower()
    or "control" in name.lower()
]

print(failure_vars)

print("\nDetailed failure objects:")

for name in failure_vars:
    obj = globals()[name]

    if isinstance(obj, (list, tuple)) and len(obj) > 0:
        print(f"\n--- {name} ---")
        print("Type:", type(obj))
        print("Length:", len(obj))

        for item in obj[:10]:
            print(item)

Variables related to control preprocessing failures:
['preprocessing_failures', 'CONTROL_WINDOWS_PATH', 'control_windows_df', 'controls_per_infant', 'control_durations', 'processed_control_windows', 'control_processing_failures', 'control_start', 'control_end']

Detailed failure objects:

--- processed_control_windows ---
Type: <class 'list'>
Length: 1
{'infant': 1, 'event_number': nan, 'event_time_s': nan, 'precursor_start_s': nan, 'precursor_end_s': nan, 'control_start_s': 14789.788, 'control_end_s': 14804.788, 'control_duration_s': 15.0, 'risk_label': 0, 'window_type': 'control', 'ecg_fs': 250.0, 'resp_fs': 500.0, 'ecg_signal': array([-0.03407784, -0.00662724, -0.01185938, ...,  0.0316916 ,
        0.02089184,  0.00469727], shape=(3750,)), 'resp_signal': array([-0.55250737, -0.54711859, -0.54162906, ..., -0.01726173,
       -0.02356549, -0.02984511], shape=(7500,)), 'ecg_samples': 3750, 'resp_samples': 7500, 'filter_context_s': 60.0}


In [25]:
# ============================================================
# CELL 8-REBUILD — Clean Control Preprocessing
#
# Purpose:
#   Rebuild all 50 control windows using the exact same
#   preprocessing methodology as the authoritative Cell 8.
#
# IMPORTANT:
#   Fresh variable names are used so stale notebook state
#   cannot affect the result.
# ============================================================

print("=" * 70)
print("CLEAN CONTROL WINDOW EXTRACTION AND PREPROCESSING")
print("=" * 70)

processed_control_windows_v2 = []
control_processing_failures_v2 = []

total_controls = len(control_windows_df)

print(f"Total control windows : {total_controls}")
print(f"Control duration      : {PRECURSOR_DURATION} s")
print(f"Filtering context     : {FILTER_CONTEXT} s each side")
print()

for row_number, row in control_windows_df.iterrows():

    infant = int(row["infant"])

    control_start = float(row["control_start_s"])
    control_end = float(row["control_end_s"])

    context_start = control_start - FILTER_CONTEXT
    context_end = control_end + FILTER_CONTEXT

    try:

        # ----------------------------------------------------
        # Read headers
        # ----------------------------------------------------
        ecg_header = wfdb.rdheader(
            f"infant{infant}_ecg",
            pn_dir=f"{PICS_DB}/{PICS_VERSION}"
        )

        resp_header = wfdb.rdheader(
            f"infant{infant}_resp",
            pn_dir=f"{PICS_DB}/{PICS_VERSION}"
        )

        ecg_fs = float(ecg_header.fs)
        resp_fs = float(resp_header.fs)

        # ----------------------------------------------------
        # Recording-duration checks
        # ----------------------------------------------------
        ecg_duration = (
            float(ecg_header.sig_len) / ecg_fs
        )

        resp_duration = (
            float(resp_header.sig_len) / resp_fs
        )

        if context_start < 0:
            raise ValueError(
                f"Negative context start: {context_start:.3f}"
            )

        if context_end > ecg_duration:
            raise ValueError(
                f"ECG context exceeds recording duration "
                f"({context_end:.3f} > {ecg_duration:.3f})"
            )

        if context_end > resp_duration:
            raise ValueError(
                f"RESP context exceeds recording duration "
                f"({context_end:.3f} > {resp_duration:.3f})"
            )

        # ----------------------------------------------------
        # Sample indices
        # ----------------------------------------------------
        ecg_sampfrom = int(
            round(context_start * ecg_fs)
        )

        ecg_sampto = int(
            round(context_end * ecg_fs)
        )

        resp_sampfrom = int(
            round(context_start * resp_fs)
        )

        resp_sampto = int(
            round(context_end * resp_fs)
        )

        # ----------------------------------------------------
        # ECG
        # ----------------------------------------------------
        ecg_record = wfdb.rdrecord(
            f"infant{infant}_ecg",
            pn_dir=f"{PICS_DB}/{PICS_VERSION}",
            sampfrom=ecg_sampfrom,
            sampto=ecg_sampto
        )

        ecg_context = np.asarray(
            ecg_record.p_signal[:, 0],
            dtype=float
        )

        # ----------------------------------------------------
        # Respiration
        # ----------------------------------------------------
        resp_record = wfdb.rdrecord(
            f"infant{infant}_resp",
            pn_dir=f"{PICS_DB}/{PICS_VERSION}",
            sampfrom=resp_sampfrom,
            sampto=resp_sampto
        )

        resp_context = np.asarray(
            resp_record.p_signal[:, 0],
            dtype=float
        )

        # ----------------------------------------------------
        # Raw integrity
        # ----------------------------------------------------
        if not np.all(np.isfinite(ecg_context)):
            raise ValueError(
                "Non-finite ECG samples"
            )

        if not np.all(np.isfinite(resp_context)):
            raise ValueError(
                "Non-finite respiration samples"
            )

        # ----------------------------------------------------
        # EXACT SAME project preprocessing
        # ----------------------------------------------------
        ecg_filtered = filter_ecg(
            ecg_context,
            ecg_fs
        )

        resp_filtered = filter_respiration(
            resp_context,
            resp_fs
        )

        # ----------------------------------------------------
        # Central 15-second region
        # ----------------------------------------------------
        ecg_keep_start = int(
            round(FILTER_CONTEXT * ecg_fs)
        )

        ecg_keep_length = int(
            round(PRECURSOR_DURATION * ecg_fs)
        )

        ecg_keep_end = (
            ecg_keep_start + ecg_keep_length
        )

        resp_keep_start = int(
            round(FILTER_CONTEXT * resp_fs)
        )

        resp_keep_length = int(
            round(PRECURSOR_DURATION * resp_fs)
        )

        resp_keep_end = (
            resp_keep_start + resp_keep_length
        )

        ecg_final = ecg_filtered[
            ecg_keep_start:ecg_keep_end
        ]

        resp_final = resp_filtered[
            resp_keep_start:resp_keep_end
        ]

        # ----------------------------------------------------
        # Expected lengths
        # ----------------------------------------------------
        expected_ecg_samples = int(
            round(PRECURSOR_DURATION * ecg_fs)
        )

        expected_resp_samples = int(
            round(PRECURSOR_DURATION * resp_fs)
        )

        if len(ecg_final) != expected_ecg_samples:
            raise ValueError(
                f"ECG sample count mismatch: "
                f"{len(ecg_final)} != "
                f"{expected_ecg_samples}"
            )

        if len(resp_final) != expected_resp_samples:
            raise ValueError(
                f"RESP sample count mismatch: "
                f"{len(resp_final)} != "
                f"{expected_resp_samples}"
            )

        if not np.all(np.isfinite(ecg_final)):
            raise ValueError(
                "Non-finite filtered ECG"
            )

        if not np.all(np.isfinite(resp_final)):
            raise ValueError(
                "Non-finite filtered respiration"
            )

        # ----------------------------------------------------
        # Store
        # ----------------------------------------------------
        processed_control_windows_v2.append({
            "infant": infant,
            "event_number": np.nan,
            "event_time_s": np.nan,
            "precursor_start_s": np.nan,
            "precursor_end_s": np.nan,

            "control_start_s": control_start,
            "control_end_s": control_end,
            "control_duration_s":
                control_end - control_start,

            "risk_label": 0,
            "window_type": "control",

            "ecg_fs": ecg_fs,
            "resp_fs": resp_fs,

            "ecg_signal": ecg_final,
            "resp_signal": resp_final,

            "ecg_samples": len(ecg_final),
            "resp_samples": len(resp_final),

            "filter_context_s": FILTER_CONTEXT
        })

        print(
            f"Window {row_number + 1:02d}/{total_controls} | "
            f"Infant {infant} | "
            f"{control_start:.3f}–{control_end:.3f} s | "
            f"PASS"
        )

    except Exception as e:

        control_processing_failures_v2.append({
            "row": int(row_number),
            "infant": infant,
            "control_start_s": control_start,
            "control_end_s": control_end,
            "error": str(e)
        })

        print(
            f"Window {row_number + 1:02d}/{total_controls} | "
            f"Infant {infant} | "
            f"FAIL | {e}"
        )

# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 70)
print("CLEAN CONTROL PREPROCESSING SUMMARY")
print("=" * 70)

print(
    "Expected controls :",
    total_controls
)

print(
    "Processed          :",
    len(processed_control_windows_v2)
)

print(
    "Failed             :",
    len(control_processing_failures_v2)
)

if control_processing_failures_v2:

    print("\nFAILED WINDOWS:")

    for failure in control_processing_failures_v2:
        print(
            f"Row {failure['row']} | "
            f"Infant {failure['infant']} | "
            f"{failure['control_start_s']:.3f}–"
            f"{failure['control_end_s']:.3f} s | "
            f"{failure['error']}"
        )

else:
    print(
        "\nPASS: No control preprocessing failures."
    )

print("=" * 70)

CLEAN CONTROL WINDOW EXTRACTION AND PREPROCESSING
Total control windows : 50
Control duration      : 15.0 s
Filtering context     : 60.0 s each side

Window 01/50 | Infant 1 | 14789.788–14804.788 s | PASS
Window 02/50 | Infant 1 | 70419.780–70434.780 s | PASS
Window 03/50 | Infant 1 | 71319.780–71334.780 s | PASS
Window 04/50 | Infant 1 | 107559.096–107574.096 s | PASS
Window 05/50 | Infant 1 | 127371.824–127386.824 s | PASS
Window 06/50 | Infant 2 | 14753.164–14768.164 s | PASS
Window 07/50 | Infant 2 | 83082.830–83097.830 s | PASS
Window 08/50 | Infant 2 | 116128.184–116143.184 s | PASS
Window 09/50 | Infant 2 | 120167.988–120182.988 s | PASS
Window 10/50 | Infant 2 | 153792.438–153807.438 s | PASS
Window 11/50 | Infant 3 | 27729.666–27744.666 s | PASS
Window 12/50 | Infant 3 | 57212.180–57227.180 s | PASS
Window 13/50 | Infant 3 | 70430.948–70445.948 s | PASS
Window 14/50 | Infant 3 | 78290.984–78305.984 s | PASS
Window 15/50 | Infant 3 | FAIL | 502 Error: Bad Gateway for url: https

In [26]:
# ============================================================
# CELL 8F-v2 — Inspect Clean Rebuild Failures
# ============================================================

print("=" * 70)
print("CLEAN REBUILD FAILURE SUMMARY")
print("=" * 70)

print(
    "Processed:",
    len(processed_control_windows_v2)
)

print(
    "Failed:",
    len(control_processing_failures_v2)
)

print("\nFAILED WINDOWS:")

for failure in control_processing_failures_v2:
    print(
        f"Row {failure['row']} | "
        f"Infant {failure['infant']} | "
        f"{failure['control_start_s']:.3f}–"
        f"{failure['control_end_s']:.3f} s | "
        f"{failure['error']}"
    )

CLEAN REBUILD FAILURE SUMMARY
Processed: 49
Failed: 1

FAILED WINDOWS:
Row 14 | Infant 3 | 131788.630–131803.630 s | 502 Error: Bad Gateway for url: https://physionet.org/files/picsdb/1.0.0/infant3_resp.hea


In [27]:
# ============================================================
# CELL 8G — Retry Single Failed Control Window
# ============================================================

failure = control_processing_failures_v2[0]

infant = int(failure["infant"])
control_start = float(failure["control_start_s"])
control_end = float(failure["control_end_s"])

print("=" * 70)
print("RETRYING FAILED CONTROL WINDOW")
print("=" * 70)

print(
    f"Infant {infant} | "
    f"{control_start:.3f}–{control_end:.3f} s"
)

context_start = control_start - FILTER_CONTEXT
context_end = control_end + FILTER_CONTEXT

# ------------------------------------------------------------
# Read headers
# ------------------------------------------------------------

ecg_header = wfdb.rdheader(
    f"infant{infant}_ecg",
    pn_dir=f"{PICS_DB}/{PICS_VERSION}"
)

resp_header = wfdb.rdheader(
    f"infant{infant}_resp",
    pn_dir=f"{PICS_DB}/{PICS_VERSION}"
)

ecg_fs = float(ecg_header.fs)
resp_fs = float(resp_header.fs)

print(
    f"ECG fs: {ecg_fs} Hz | "
    f"RESP fs: {resp_fs} Hz"
)

# ------------------------------------------------------------
# Sample indices
# ------------------------------------------------------------

ecg_sampfrom = int(
    round(context_start * ecg_fs)
)

ecg_sampto = int(
    round(context_end * ecg_fs)
)

resp_sampfrom = int(
    round(context_start * resp_fs)
)

resp_sampto = int(
    round(context_end * resp_fs)
)

# ------------------------------------------------------------
# Read ECG
# ------------------------------------------------------------

ecg_record = wfdb.rdrecord(
    f"infant{infant}_ecg",
    pn_dir=f"{PICS_DB}/{PICS_VERSION}",
    sampfrom=ecg_sampfrom,
    sampto=ecg_sampto
)

ecg_context = np.asarray(
    ecg_record.p_signal[:, 0],
    dtype=float
)

# ------------------------------------------------------------
# Read respiration
# ------------------------------------------------------------

resp_record = wfdb.rdrecord(
    f"infant{infant}_resp",
    pn_dir=f"{PICS_DB}/{PICS_VERSION}",
    sampfrom=resp_sampfrom,
    sampto=resp_sampto
)

resp_context = np.asarray(
    resp_record.p_signal[:, 0],
    dtype=float
)

print(
    f"Retrieved ECG: {len(ecg_context)} samples"
)

print(
    f"Retrieved RESP: {len(resp_context)} samples"
)

# ------------------------------------------------------------
# Raw checks
# ------------------------------------------------------------

assert np.all(np.isfinite(ecg_context))
assert np.all(np.isfinite(resp_context))

# ------------------------------------------------------------
# Same project preprocessing
# ------------------------------------------------------------

ecg_filtered = filter_ecg(
    ecg_context,
    ecg_fs
)

resp_filtered = filter_respiration(
    resp_context,
    resp_fs
)

# ------------------------------------------------------------
# Central 15-second region
# ------------------------------------------------------------

ecg_keep_start = int(
    round(FILTER_CONTEXT * ecg_fs)
)

ecg_keep_length = int(
    round(PRECURSOR_DURATION * ecg_fs)
)

resp_keep_start = int(
    round(FILTER_CONTEXT * resp_fs)
)

resp_keep_length = int(
    round(PRECURSOR_DURATION * resp_fs)
)

ecg_final = ecg_filtered[
    ecg_keep_start:
    ecg_keep_start + ecg_keep_length
]

resp_final = resp_filtered[
    resp_keep_start:
    resp_keep_start + resp_keep_length
]

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

expected_ecg_samples = int(
    round(PRECURSOR_DURATION * ecg_fs)
)

expected_resp_samples = int(
    round(PRECURSOR_DURATION * resp_fs)
)

assert len(ecg_final) == expected_ecg_samples
assert len(resp_final) == expected_resp_samples

assert np.all(np.isfinite(ecg_final))
assert np.all(np.isfinite(resp_final))

# ------------------------------------------------------------
# Store recovered window
# ------------------------------------------------------------

recovered_control_window = {
    "infant": infant,
    "event_number": np.nan,
    "event_time_s": np.nan,
    "precursor_start_s": np.nan,
    "precursor_end_s": np.nan,

    "control_start_s": control_start,
    "control_end_s": control_end,
    "control_duration_s":
        control_end - control_start,

    "risk_label": 0,
    "window_type": "control",

    "ecg_fs": ecg_fs,
    "resp_fs": resp_fs,

    "ecg_signal": ecg_final,
    "resp_signal": resp_final,

    "ecg_samples": len(ecg_final),
    "resp_samples": len(resp_final),

    "filter_context_s": FILTER_CONTEXT
}

print()
print("PASS: Failed control window recovered.")
print(
    f"ECG samples: {len(ecg_final)}"
)
print(
    f"RESP samples: {len(resp_final)}"
)
print("PASS: Exact 15-second duration.")
print("PASS: No NaN/Inf values.")

RETRYING FAILED CONTROL WINDOW
Infant 3 | 131788.630–131803.630 s
ECG fs: 500.0 Hz | RESP fs: 50.0 Hz
Retrieved ECG: 67500 samples
Retrieved RESP: 6750 samples

PASS: Failed control window recovered.
ECG samples: 7500
RESP samples: 750
PASS: Exact 15-second duration.
PASS: No NaN/Inf values.


In [30]:
# ============================================================
# CELL 8H-DIAGNOSTIC — INSPECT PROCESSED CONTROL STRUCTURE
# ============================================================

print("=" * 70)
print("INSPECTING PROCESSED CONTROL WINDOW STRUCTURE")
print("=" * 70)

print("\nKeys in processed_control_windows_v2[0]:")
print(processed_control_windows_v2[0].keys())

print("\nKeys in recovered_control_window:")
print(recovered_control_window.keys())

print("\nExample processed control:")
for key, value in processed_control_windows_v2[0].items():
    if isinstance(value, np.ndarray):
        print(f"{key}: numpy array, shape={value.shape}")
    else:
        print(f"{key}: {value}")

INSPECTING PROCESSED CONTROL WINDOW STRUCTURE

Keys in processed_control_windows_v2[0]:
dict_keys(['infant', 'event_number', 'event_time_s', 'precursor_start_s', 'precursor_end_s', 'control_start_s', 'control_end_s', 'control_duration_s', 'risk_label', 'window_type', 'ecg_fs', 'resp_fs', 'ecg_signal', 'resp_signal', 'ecg_samples', 'resp_samples', 'filter_context_s'])

Keys in recovered_control_window:
dict_keys(['infant', 'event_number', 'event_time_s', 'precursor_start_s', 'precursor_end_s', 'control_start_s', 'control_end_s', 'control_duration_s', 'risk_label', 'window_type', 'ecg_fs', 'resp_fs', 'ecg_signal', 'resp_signal', 'ecg_samples', 'resp_samples', 'filter_context_s'])

Example processed control:
infant: 1
event_number: nan
event_time_s: nan
precursor_start_s: nan
precursor_end_s: nan
control_start_s: 14789.788
control_end_s: 14804.788
control_duration_s: 15.0
risk_label: 0
window_type: control
ecg_fs: 250.0
resp_fs: 500.0
ecg_signal: numpy array, shape=(3750,)
resp_signal: nu

In [31]:
# ============================================================
# CELL 8H — FINALIZE 50 CONTROL WINDOWS
# ============================================================

print("=" * 70)
print("FINALIZING CONTROL WINDOWS")
print("=" * 70)

# Start with the 49 successfully processed controls
final_processed_control_windows = processed_control_windows_v2.copy()

# Add the one successfully recovered control
final_processed_control_windows.append(
    recovered_control_window
)

print(f"Processed controls: {len(final_processed_control_windows)}")

# ------------------------------------------------------------
# Basic count check
# ------------------------------------------------------------

assert len(final_processed_control_windows) == 50, (
    f"Expected 50 controls, got {len(final_processed_control_windows)}"
)

# ------------------------------------------------------------
# Verify every control window
# ------------------------------------------------------------

for i, window in enumerate(final_processed_control_windows, start=1):

    assert window["risk_label"] == 0, (
        f"Control {i} has incorrect risk label"
    )

    assert window["window_type"] == "control", (
        f"Control {i} has incorrect window type"
    )

    assert np.isfinite(window["ecg_signal"]).all(), (
        f"Control {i} ECG contains NaN/Inf"
    )

    assert np.isfinite(window["resp_signal"]).all(), (
        f"Control {i} respiration contains NaN/Inf"
    )

    duration = (
        window["control_end_s"] -
        window["control_start_s"]
    )

    assert np.isclose(duration, 15.0), (
        f"Control {i} duration is {duration:.6f}s"
    )

    assert np.isclose(
        window["control_duration_s"], 15.0
    ), (
        f"Control {i} stored duration is "
        f"{window['control_duration_s']}"
    )

# ------------------------------------------------------------
# Check infant distribution
# ------------------------------------------------------------

control_infant_counts = {}

for window in final_processed_control_windows:

    infant = int(window["infant"])

    control_infant_counts[infant] = (
        control_infant_counts.get(infant, 0) + 1
    )

print("\nControls per infant:")

for infant in sorted(control_infant_counts):

    print(
        f"Infant {infant}: "
        f"{control_infant_counts[infant]} controls"
    )

assert set(control_infant_counts.keys()) == set(range(1, 11)), (
    "Controls are not present for all 10 infants."
)

assert all(
    count == 5
    for count in control_infant_counts.values()
), (
    "Each infant must have exactly 5 controls."
)

# ------------------------------------------------------------
# Check duplicate windows
# ------------------------------------------------------------

control_keys = [
    (
        int(window["infant"]),
        round(float(window["control_start_s"]), 6),
        round(float(window["control_end_s"]), 6)
    )
    for window in final_processed_control_windows
]

assert len(control_keys) == len(set(control_keys)), (
    "Duplicate control windows detected."
)

# ------------------------------------------------------------
# Final integrity summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL CONTROL WINDOW CHECK")
print("=" * 70)

print(f"Total controls       : {len(final_processed_control_windows)}")
print(f"Expected              : 50")
print(f"Infants represented   : {len(control_infant_counts)}")
print(f"Duration per window   : 15 seconds")
print(f"Label                 : 0")
print(f"Window type           : control")
print("NaN/Inf               : None")
print("Duplicate windows     : None")

print("\nPASS: Final 50-control set verified.")

FINALIZING CONTROL WINDOWS
Processed controls: 50

Controls per infant:
Infant 1: 5 controls
Infant 2: 5 controls
Infant 3: 5 controls
Infant 4: 5 controls
Infant 5: 5 controls
Infant 6: 5 controls
Infant 7: 5 controls
Infant 8: 5 controls
Infant 9: 5 controls
Infant 10: 5 controls

FINAL CONTROL WINDOW CHECK
Total controls       : 50
Expected              : 50
Infants represented   : 10
Duration per window   : 15 seconds
Label                 : 0
Window type           : control
NaN/Inf               : None
Duplicate windows     : None

PASS: Final 50-control set verified.


In [33]:
# ============================================================
# CELL 8I — BUILD FINAL 55-WINDOW SIGNAL DATASET
# ============================================================

print("=" * 70)
print("BUILDING FINAL 55-WINDOW SIGNAL DATASET")
print("=" * 70)

# Combine the already processed precursor windows
# with the fully verified control windows.
all_window_signals_final = (
    processed_precursor_windows.copy()
    + final_processed_control_windows.copy()
)

print(f"Total windows: {len(all_window_signals_final)}")

# ------------------------------------------------------------
# Basic count checks
# ------------------------------------------------------------

assert len(all_window_signals_final) == 55, (
    f"Expected 55 windows, got {len(all_window_signals_final)}"
)

# ------------------------------------------------------------
# Label distribution
# ------------------------------------------------------------

label_counts = {}

for window in all_window_signals_final:
    label = int(window["risk_label"])
    label_counts[label] = label_counts.get(label, 0) + 1

print("\nRisk-label distribution:")
for label in sorted(label_counts):
    print(f"Label {label}: {label_counts[label]} windows")

assert label_counts.get(0, 0) == 50, (
    "Expected exactly 50 control windows."
)

assert label_counts.get(1, 0) == 5, (
    "Expected exactly 5 precursor windows."
)

# ------------------------------------------------------------
# Check window types
# ------------------------------------------------------------

window_type_counts = {}

for window in all_window_signals_final:
    window_type = window["window_type"]
    window_type_counts[window_type] = (
        window_type_counts.get(window_type, 0) + 1
    )

print("\nWindow-type distribution:")
for window_type, count in window_type_counts.items():
    print(f"{window_type}: {count}")

# ------------------------------------------------------------
# Verify signal integrity
# ------------------------------------------------------------

for i, window in enumerate(all_window_signals_final, start=1):

    assert np.isfinite(window["ecg_signal"]).all(), (
        f"Window {i} ECG contains NaN/Inf"
    )

    assert np.isfinite(window["resp_signal"]).all(), (
        f"Window {i} respiration contains NaN/Inf"
    )

    assert np.isclose(
        len(window["ecg_signal"]) / window["ecg_fs"],
        15.0
    ), (
        f"Window {i} ECG is not exactly 15 seconds."
    )

    assert np.isclose(
        len(window["resp_signal"]) / window["resp_fs"],
        15.0
    ), (
        f"Window {i} respiration is not exactly 15 seconds."
    )

# ------------------------------------------------------------
# Infant distribution
# ------------------------------------------------------------

all_infant_counts = {}

for window in all_window_signals_final:

    infant = int(window["infant"])

    all_infant_counts[infant] = (
        all_infant_counts.get(infant, 0) + 1
    )

print("\nTotal windows per infant:")

for infant in sorted(all_infant_counts):
    print(
        f"Infant {infant}: "
        f"{all_infant_counts[infant]} windows"
    )

assert set(all_infant_counts.keys()) == set(range(1, 11)), (
    "Not all 10 infants are represented."
)

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL 55-WINDOW SIGNAL CHECK")
print("=" * 70)

print(f"Total windows       : {len(all_window_signals_final)}")
print(f"Controls            : {label_counts[0]}")
print(f"Precursor windows   : {label_counts[1]}")
print(f"Infants             : {len(all_infant_counts)}")
print("Window duration     : 15 seconds")
print("ECG finite          : PASS")
print("Respiration finite  : PASS")

print("\nPASS: Final 55-window signal dataset verified.")

BUILDING FINAL 55-WINDOW SIGNAL DATASET
Total windows: 55

Risk-label distribution:
Label 0: 50 windows
Label 1: 5 windows

Window-type distribution:
precursor: 5
control: 50

Total windows per infant:
Infant 1: 6 windows
Infant 2: 5 windows
Infant 3: 7 windows
Infant 4: 5 windows
Infant 5: 5 windows
Infant 6: 5 windows
Infant 7: 5 windows
Infant 8: 5 windows
Infant 9: 7 windows
Infant 10: 5 windows

FINAL 55-WINDOW SIGNAL CHECK
Total windows       : 55
Controls            : 50
Precursor windows   : 5
Infants             : 10
Window duration     : 15 seconds
ECG finite          : PASS
Respiration finite  : PASS

PASS: Final 55-window signal dataset verified.


In [16]:
# ============================================================
# CELL 8A — Inspect Exact Cell 8 Failure
# ============================================================

print("=" * 70)
print("CELL 8 PREPROCESSING FAILURE DIAGNOSTIC")
print("=" * 70)

print("\npreprocessing_failures type:")
print(type(preprocessing_failures))

print("\npreprocessing_failures length:")
try:
    print(len(preprocessing_failures))
except Exception:
    print("Length not available")

print("\npreprocessing_failures contents:")
print(preprocessing_failures)

print("\n" + "=" * 70)
print("CONTROL_WINDOWS_PATH")
print("=" * 70)
print(CONTROL_WINDOWS_PATH)

print("\n" + "=" * 70)
print("Number of successfully processed controls")
print("=" * 70)
print(len(processed_control_windows))

CELL 8 PREPROCESSING FAILURE DIAGNOSTIC

preprocessing_failures type:
<class 'list'>

preprocessing_failures length:
0

preprocessing_failures contents:
[]

CONTROL_WINDOWS_PATH
c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_final_control_windows.csv

Number of successfully processed controls
1


In [17]:
# ============================================================
# CELL 8B-DIAGNOSTIC — Compare Expected vs Processed Controls
# ============================================================

print("=" * 70)
print("CONTROL PREPROCESSING STATE")
print("=" * 70)

print("\nExpected control windows:")
print("Shape:", control_windows_df.shape)

print("\nControl dataframe columns:")
print(control_windows_df.columns.tolist())

print("\nExpected controls per infant:")
print(control_windows_df.groupby("infant").size())

print("\nSuccessfully processed controls:")
print(len(processed_control_windows))

print("\nProcessed control keys:")

for item in processed_control_windows:
    print(
        "Infant:", item.get("infant"),
        "| Start:",
        item.get("control_start_s"),
        "| End:",
        item.get("control_end_s")
    )

print("\n" + "=" * 70)
print("FIRST EXPECTED CONTROL WINDOWS")
print("=" * 70)

display(
    control_windows_df.head(10)
)

CONTROL PREPROCESSING STATE

Expected control windows:
Shape: (50, 10)

Control dataframe columns:
['infant', 'event_number', 'event_time_s', 'precursor_start_s', 'precursor_end_s', 'control_start_s', 'control_end_s', 'control_duration_s', 'risk_label', 'window_type']

Expected controls per infant:
infant
1     5
2     5
3     5
4     5
5     5
6     5
7     5
8     5
9     5
10    5
dtype: int64

Successfully processed controls:
1

Processed control keys:
Infant: 1 | Start: 14789.788 | End: 14804.788

FIRST EXPECTED CONTROL WINDOWS


,infant,event_number,event_time_s,precursor_start_s,precursor_end_s,control_start_s,control_end_s,control_duration_s,risk_label,window_type
0,1,NaN,NaN,NaN,NaN,14789.788,14804.788,15.0,0,control
1,1,NaN,NaN,NaN,NaN,70419.780,70434.780,15.0,0,control
2,1,NaN,NaN,NaN,NaN,71319.780,71334.780,15.0,0,control
3,1,NaN,NaN,NaN,NaN,107559.096,107574.096,15.0,0,control
4,1,NaN,NaN,NaN,NaN,127371.824,127386.824,15.0,0,control
5,2,NaN,NaN,NaN,NaN,14753.164,14768.164,15.0,0,control
6,2,NaN,NaN,NaN,NaN,83082.830,83097.830,15.0,0,control
7,2,NaN,NaN,NaN,NaN,116128.184,116143.184,15.0,0,control
8,2,NaN,NaN,NaN,NaN,120167.988,120182.988,15.0,0,control
9,2,NaN,NaN,NaN,NaN,153792.438,153807.438,15.0,0,control


In [18]:
# ============================================================
# CELL 8B-DIAGNOSTIC — Identify Processing Progress
# ============================================================

print("=" * 70)
print("CONTROL PROCESSING PROGRESS")
print("=" * 70)

print("\nExpected controls:", len(control_windows_df))
print("Processed controls:", len(processed_control_windows))
print("Recorded failures:", len(preprocessing_failures))

print("\nProcessed control windows:")

for i, item in enumerate(processed_control_windows):
    print(
        f"{i}: Infant {item['infant']} | "
        f"{item['control_start_s']:.3f} - "
        f"{item['control_end_s']:.3f} s"
    )

print("\nNext expected control after processed windows:")

if len(processed_control_windows) > 0:

    processed_keys = {
        (
            int(item["infant"]),
            round(float(item["control_start_s"]), 6),
            round(float(item["control_end_s"]), 6)
        )
        for item in processed_control_windows
    }

    remaining_controls = control_windows_df[
        ~control_windows_df.apply(
            lambda row: (
                int(row["infant"]),
                round(float(row["control_start_s"]), 6),
                round(float(row["control_end_s"]), 6)
            ) in processed_keys,
            axis=1
        )
    ]

    print(
        remaining_controls[
            [
                "infant",
                "control_start_s",
                "control_end_s"
            ]
        ].head(10).to_string(index=False)
    )

else:
    print("No successfully processed controls found.")

print("\nPASS: Processing state inspected without modifying data.")

CONTROL PROCESSING PROGRESS

Expected controls: 50
Processed controls: 1
Recorded failures: 0

Processed control windows:
0: Infant 1 | 14789.788 - 14804.788 s

Next expected control after processed windows:
 infant  control_start_s  control_end_s
      1        70419.780      70434.780
      1        71319.780      71334.780
      1       107559.096     107574.096
      1       127371.824     127386.824
      2        14753.164      14768.164
      2        83082.830      83097.830
      2       116128.184     116143.184
      2       120167.988     120182.988
      2       153792.438     153807.438
      3        27729.666      27744.666

PASS: Processing state inspected without modifying data.


In [19]:
# ============================================================
# CELL 8C — Test the Next Unprocessed Control Window
# ============================================================

TEST_INFANT = 1
TEST_START = 70419.780
TEST_END = 70434.780

print("=" * 70)
print("TESTING NEXT UNPROCESSED CONTROL WINDOW")
print("=" * 70)

print(f"Infant: {TEST_INFANT}")
print(f"Start:  {TEST_START:.3f} s")
print(f"End:    {TEST_END:.3f} s")
print(f"Duration: {TEST_END - TEST_START:.3f} s")

# ------------------------------------------------------------
# Find the exact row in the control dataframe
# ------------------------------------------------------------
test_rows = control_windows_df[
    (control_windows_df["infant"] == TEST_INFANT) &
    (np.isclose(
        control_windows_df["control_start_s"],
        TEST_START
    )) &
    (np.isclose(
        control_windows_df["control_end_s"],
        TEST_END
    ))
]

print("\nMatching control rows:", len(test_rows))

display(test_rows)

TESTING NEXT UNPROCESSED CONTROL WINDOW
Infant: 1
Start:  70419.780 s
End:    70434.780 s
Duration: 15.000 s

Matching control rows: 1


,infant,event_number,event_time_s,precursor_start_s,precursor_end_s,control_start_s,control_end_s,control_duration_s,risk_label,window_type
1,1,NaN,NaN,NaN,NaN,70419.78,70434.78,15.0,0,control


In [21]:
# ============================================================
# CELL 8D — Direct Test of One Control Window
# ============================================================

TEST_INFANT = 1
TEST_START = 70419.780
TEST_END = 70434.780

print("=" * 70)
print("DIRECT CONTROL WINDOW RETRIEVAL + PREPROCESSING TEST")
print("=" * 70)

# ------------------------------------------------------------
# Find the required record sampling rates
# ------------------------------------------------------------
ecg_record = wfdb.rdrecord(
    f"infant{TEST_INFANT}_ecg",
    pn_dir=f"{PICS_DB}/{PICS_VERSION}",
    sampto=10
)

resp_record = wfdb.rdrecord(
    f"infant{TEST_INFANT}_resp",
    pn_dir=f"{PICS_DB}/{PICS_VERSION}",
    sampto=10
)

ecg_fs = float(ecg_record.fs)
resp_fs = float(resp_record.fs)

print(f"ECG sampling rate:  {ecg_fs} Hz")
print(f"RESP sampling rate: {resp_fs} Hz")

# ------------------------------------------------------------
# Add 60-second context for zero-phase filtering
# ------------------------------------------------------------
FILTER_CONTEXT = 60.0

context_start = max(
    0.0,
    TEST_START - FILTER_CONTEXT
)

context_end = TEST_END + FILTER_CONTEXT

print(
    f"\nContext window: "
    f"{context_start:.3f} - {context_end:.3f} s"
)

# ------------------------------------------------------------
# Sample indices
# ------------------------------------------------------------
ecg_start = int(
    np.floor(context_start * ecg_fs)
)

ecg_end = int(
    np.ceil(context_end * ecg_fs)
)

resp_start = int(
    np.floor(context_start * resp_fs)
)

resp_end = int(
    np.ceil(context_end * resp_fs)
)

print(
    f"ECG samples requested:  "
    f"{ecg_start} - {ecg_end}"
)

print(
    f"RESP samples requested: "
    f"{resp_start} - {resp_end}"
)

# ------------------------------------------------------------
# Retrieve ECG
# ------------------------------------------------------------
print("\nRetrieving ECG...")

ecg_segment = wfdb.rdsamp(
    f"infant{TEST_INFANT}_ecg",
    pn_dir=f"{PICS_DB}/{PICS_VERSION}",
    sampfrom=ecg_start,
    sampto=ecg_end
)

ecg_signal = np.asarray(
    ecg_segment[0][:, 0],
    dtype=float
)

print(
    "ECG retrieved:",
    len(ecg_signal),
    "samples"
)

# ------------------------------------------------------------
# Retrieve respiration
# ------------------------------------------------------------
print("Retrieving respiration...")

resp_segment = wfdb.rdsamp(
    f"infant{TEST_INFANT}_resp",
    pn_dir=f"{PICS_DB}/{PICS_VERSION}",
    sampfrom=resp_start,
    sampto=resp_end
)

resp_signal = np.asarray(
    resp_segment[0][:, 0],
    dtype=float
)

print(
    "RESP retrieved:",
    len(resp_signal),
    "samples"
)

# ------------------------------------------------------------
# Basic signal checks
# ------------------------------------------------------------
assert len(ecg_signal) > 0
assert len(resp_signal) > 0

assert np.all(np.isfinite(ecg_signal))
assert np.all(np.isfinite(resp_signal))

print("\nPASS: Raw ECG and respiration retrieved.")
print("PASS: No NaN/Inf values detected.")

DIRECT CONTROL WINDOW RETRIEVAL + PREPROCESSING TEST
ECG sampling rate:  250.0 Hz
RESP sampling rate: 500.0 Hz

Context window: 70359.780 - 70494.780 s
ECG samples requested:  17589945 - 17623695
RESP samples requested: 35179890 - 35247390

Retrieving ECG...
ECG retrieved: 33750 samples
Retrieving respiration...
RESP retrieved: 67500 samples

PASS: Raw ECG and respiration retrieved.
PASS: No NaN/Inf values detected.


In [22]:
# ============================================================
# CELL 8E — Direct Preprocessing Test
# ============================================================

from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt

print("=" * 70)
print("DIRECT PREPROCESSING TEST")
print("=" * 70)

# ------------------------------------------------------------
# ECG preprocessing
# Same project specification:
#   Bandpass: 0.5–40 Hz
#   Butterworth order: 4
#   Zero-phase filtering
#   50 Hz notch, Q=30
# ------------------------------------------------------------

ecg_sos = butter(
    4,
    [0.5, 40.0],
    btype="bandpass",
    fs=ecg_fs,
    output="sos"
)

ecg_filtered = sosfiltfilt(
    ecg_sos,
    ecg_signal
)

print(
    "ECG bandpass filtering: PASS"
)

# 50 Hz notch
b_notch, a_notch = iirnotch(
    50.0,
    30.0,
    fs=ecg_fs
)

ecg_filtered = filtfilt(
    b_notch,
    a_notch,
    ecg_filtered
)

print(
    "ECG 50 Hz notch filtering: PASS"
)

# ------------------------------------------------------------
# Respiration preprocessing
# Same project specification:
#   Bandpass: 0.03–2.0 Hz
#   Butterworth order: 4
#   Zero-phase filtering
# ------------------------------------------------------------

resp_sos = butter(
    4,
    [0.03, 2.0],
    btype="bandpass",
    fs=resp_fs,
    output="sos"
)

resp_filtered = sosfiltfilt(
    resp_sos,
    resp_signal
)

print(
    "Respiration filtering: PASS"
)

# ------------------------------------------------------------
# Numerical integrity
# ------------------------------------------------------------

assert np.all(np.isfinite(ecg_filtered))
assert np.all(np.isfinite(resp_filtered))

print(
    "\nFiltered ECG samples:",
    len(ecg_filtered)
)

print(
    "Filtered RESP samples:",
    len(resp_filtered)
)

print(
    "\nECG filtered range:",
    float(np.min(ecg_filtered)),
    "to",
    float(np.max(ecg_filtered))
)

print(
    "RESP filtered range:",
    float(np.min(resp_filtered)),
    "to",
    float(np.max(resp_filtered))
)

# ------------------------------------------------------------
# Extract exact 15-second target window from the context
# ------------------------------------------------------------

ecg_target_start = int(
    round(
        (TEST_START - context_start)
        * ecg_fs
    )
)

ecg_target_end = int(
    round(
        (TEST_END - context_start)
        * ecg_fs
    )
)

resp_target_start = int(
    round(
        (TEST_START - context_start)
        * resp_fs
    )
)

resp_target_end = int(
    round(
        (TEST_END - context_start)
        * resp_fs
    )
)

ecg_target = ecg_filtered[
    ecg_target_start:ecg_target_end
]

resp_target = resp_filtered[
    resp_target_start:resp_target_end
]

print(
    "\nTarget ECG samples:",
    len(ecg_target)
)

print(
    "Target RESP samples:",
    len(resp_target)
)

# ------------------------------------------------------------
# Exact 15-second checks
# ------------------------------------------------------------

expected_ecg_samples = int(
    round(15.0 * ecg_fs)
)

expected_resp_samples = int(
    round(15.0 * resp_fs)
)

assert len(ecg_target) == expected_ecg_samples
assert len(resp_target) == expected_resp_samples

assert np.all(np.isfinite(ecg_target))
assert np.all(np.isfinite(resp_target))

print(
    "\nPASS: ECG preprocessing completed."
)

print(
    "PASS: Respiration preprocessing completed."
)

print(
    "PASS: Exact 15-second ECG/RESP windows extracted."
)

print(
    "PASS: No NaN/Inf values after preprocessing."
)

DIRECT PREPROCESSING TEST
ECG bandpass filtering: PASS
ECG 50 Hz notch filtering: PASS
Respiration filtering: PASS

Filtered ECG samples: 33750
Filtered RESP samples: 67500

ECG filtered range: -1.8306097202558649 to 2.460701321783981
RESP filtered range: -2.044503452604231 to 2.581558869872349

Target ECG samples: 3750
Target RESP samples: 7500

PASS: ECG preprocessing completed.
PASS: Respiration preprocessing completed.
PASS: Exact 15-second ECG/RESP windows extracted.
PASS: No NaN/Inf values after preprocessing.


In [34]:
# Cell 12 — Locate the surviving five precursor signal windows

print("=" * 70)
print("LOCATING FIVE PRECURSOR SIGNAL WINDOWS")
print("=" * 70)

# Snapshot globals() so the namespace cannot change during inspection
global_items = list(globals().items())

candidates = []

for name, value in global_items:

    if name.startswith("_"):
        continue

    if not isinstance(value, list):
        continue

    # We specifically want the 5-window precursor collection
    if len(value) != 5:
        continue

    if len(value) == 0:
        continue

    if not all(isinstance(x, dict) for x in value):
        continue

    keys = set(value[0].keys())

    # Signal-bearing window dictionaries
    if (
        "infant" in keys
        and "ecg_signal" in keys
        and "resp_signal" in keys
        and "risk_label" in keys
    ):
        candidates.append(name)


print()
print("Candidate five-window variables:")

for name in candidates:
    print(f"  - {name}")

print()

if len(candidates) == 1:

    recovered_precursor_name = candidates[0]

    precursor_window_signals = list(
        globals()[recovered_precursor_name]
    )

    print(f"Recovered variable : {recovered_precursor_name}")
    print(f"Recovered windows  : {len(precursor_window_signals)}")

    print()
    print("Precursor windows:")

    for i, record in enumerate(
        precursor_window_signals,
        start=1
    ):
        print(
            f"  {i}/5 | "
            f"Infant {int(record['infant'])} | "
            f"{float(record['precursor_start_s']):.3f}–"
            f"{float(record['precursor_end_s']):.3f} s | "
            f"ECG {len(record['ecg_signal'])} | "
            f"RESP {len(record['resp_signal'])}"
        )

    assert len(precursor_window_signals) == 5

    print()
    print("=" * 70)
    print("PASS: FIVE PRECURSOR WINDOWS RECOVERED")
    print("=" * 70)

elif len(candidates) == 0:

    print("=" * 70)
    print("NO FIVE-WINDOW PRECURSOR VARIABLE FOUND")
    print("DO NOT RERUN THE PREPROCESSING YET")
    print("=" * 70)

else:

    print("=" * 70)
    print("MULTIPLE FIVE-WINDOW CANDIDATES FOUND")
    print("DO NOT CONTINUE UNTIL THE CORRECT ONE IS IDENTIFIED")
    print("=" * 70)

LOCATING FIVE PRECURSOR SIGNAL WINDOWS

Candidate five-window variables:
  - precursor_windows
  - processed_precursor_windows

MULTIPLE FIVE-WINDOW CANDIDATES FOUND
DO NOT CONTINUE UNTIL THE CORRECT ONE IS IDENTIFIED


In [35]:
# Cell 13 — Compare the two five-window precursor collections

print("=" * 70)
print("COMPARING PRECURSOR WINDOW COLLECTIONS")
print("=" * 70)

for variable_name in [
    "precursor_windows",
    "processed_precursor_windows"
]:

    value = globals().get(variable_name, None)

    print()
    print(f"VARIABLE: {variable_name}")

    if value is None:
        print("  NOT AVAILABLE")
        continue

    print(f"  Type       : {type(value).__name__}")
    print(f"  Count      : {len(value)}")

    if len(value) > 0 and isinstance(value[0], dict):

        record = value[0]

        print(f"  Keys       : {list(record.keys())}")

        if "ecg_signal" in record:
            print(f"  ECG samples: {len(record['ecg_signal'])}")

        if "resp_signal" in record:
            print(f"  RESP samples: {len(record['resp_signal'])}")

        if "ecg_fs" in record:
            print(f"  ECG fs     : {record['ecg_fs']}")

        if "resp_fs" in record:
            print(f"  RESP fs    : {record['resp_fs']}")

        if "window_type" in record:
            print(f"  Window type: {record['window_type']}")

        if "risk_label" in record:
            print(f"  Risk label : {record['risk_label']}")

print()
print("=" * 70)
print("COMPARISON COMPLETE")
print("=" * 70)

COMPARING PRECURSOR WINDOW COLLECTIONS

VARIABLE: precursor_windows
  Type       : list
  Count      : 5
  Keys       : ['infant', 'event_number', 'event_time_s', 'precursor_start_s', 'precursor_end_s', 'ecg_fs', 'resp_fs', 'ecg_samples', 'resp_samples', 'ecg_duration_s', 'resp_duration_s', 'ecg_signal', 'resp_signal', 'risk_label', 'window_type']
  ECG samples: 3750
  RESP samples: 7500
  ECG fs     : 250.0
  RESP fs    : 500.0
  Window type: precursor
  Risk label : 1

VARIABLE: processed_precursor_windows
  Type       : list
  Count      : 5
  Keys       : ['infant', 'event_number', 'event_time_s', 'precursor_start_s', 'precursor_end_s', 'window_type', 'risk_label', 'ecg_fs', 'resp_fs', 'ecg_signal', 'resp_signal', 'ecg_samples', 'resp_samples', 'filter_context_s']
  ECG samples: 3750
  RESP samples: 7500
  ECG fs     : 250.0
  RESP fs    : 500.0
  Window type: precursor
  Risk label : 1

COMPARISON COMPLETE


In [37]:
# Cell 12 — Combine the verified processed precursor and control windows

print("=" * 70)
print("COMBINING VERIFIED PROCESSED WINDOWS")
print("=" * 70)

# ---------------------------------------------------------------
# Use the variables that were actually verified in the notebook
# ---------------------------------------------------------------

assert "processed_precursor_windows" in globals(), (
    "processed_precursor_windows is not available."
)

assert "final_processed_control_windows" in globals(), (
    "final_processed_control_windows is not available."
)

# ---------------------------------------------------------------
# Verify component counts
# ---------------------------------------------------------------

assert len(processed_precursor_windows) == 5, (
    f"Expected 5 processed precursor windows, "
    f"found {len(processed_precursor_windows)}"
)

assert len(final_processed_control_windows) == 50, (
    f"Expected 50 verified processed control windows, "
    f"found {len(final_processed_control_windows)}"
)

# ---------------------------------------------------------------
# Combine
# ---------------------------------------------------------------

all_window_signals = (
    list(processed_precursor_windows)
    + list(final_processed_control_windows)
)

# ---------------------------------------------------------------
# Basic integrity checks
# ---------------------------------------------------------------

assert len(all_window_signals) == 55, (
    f"Expected 55 total signal windows, "
    f"found {len(all_window_signals)}"
)

# Verify labels
risk_labels = [
    int(record["risk_label"])
    for record in all_window_signals
]

assert risk_labels.count(0) == 50
assert risk_labels.count(1) == 5

# Verify window types
window_types = [
    record["window_type"]
    for record in all_window_signals
]

assert window_types.count("control") == 50
assert window_types.count("precursor") == 5

# ---------------------------------------------------------------
# Verify all signals are finite
# ---------------------------------------------------------------

for i, record in enumerate(all_window_signals, start=1):

    assert np.all(np.isfinite(record["ecg_signal"])), (
        f"Non-finite ECG signal in combined window {i}"
    )

    assert np.all(np.isfinite(record["resp_signal"])), (
        f"Non-finite respiration signal in combined window {i}"
    )

# ---------------------------------------------------------------
# Summary
# ---------------------------------------------------------------

print()
print("Processed precursor windows :", len(processed_precursor_windows))
print("Verified control windows    :", len(final_processed_control_windows))
print("Total signal windows        :", len(all_window_signals))

print()
print("Risk-label distribution:")
print("  Risk 0 / Control    :", risk_labels.count(0))
print("  Risk 1 / Precursor  :", risk_labels.count(1))

print()
print("Window-type distribution:")
print("  Control             :", window_types.count("control"))
print("  Precursor           :", window_types.count("precursor"))

print()
print("=" * 70)
print("PASS: 55 SIGNAL WINDOWS COMBINED")
print("PASS: 5 PRECURSOR + 50 CONTROL")
print("PASS: ALL ECG AND RESPIRATION SIGNALS ARE FINITE")
print("STATUS: DEVELOPMENT DATASET")
print("=" * 70)

COMBINING VERIFIED PROCESSED WINDOWS

Processed precursor windows : 5
Verified control windows    : 50
Total signal windows        : 55

Risk-label distribution:
  Risk 0 / Control    : 50
  Risk 1 / Precursor  : 5

Window-type distribution:
  Control             : 50
  Precursor           : 5

PASS: 55 SIGNAL WINDOWS COMBINED
PASS: 5 PRECURSOR + 50 CONTROL
PASS: ALL ECG AND RESPIRATION SIGNALS ARE FINITE
STATUS: DEVELOPMENT DATASET


In [38]:
# Cell 14A — Check current feature-related variables

print("=" * 70)
print("CURRENT FEATURE DATASET STATE")
print("=" * 70)

for name in [
    "all_window_signals",
    "feature_rows",
    "final_feature_dataset",
    "PHYSIOLOGICAL_FEATURES"
]:
    print(
        f"{name:30s} : "
        f"{'AVAILABLE' if name in globals() else 'NOT AVAILABLE'}"
    )

if "all_window_signals" in globals():
    print()
    print("all_window_signals count:", len(all_window_signals))

if "feature_rows" in globals():
    print(
        "feature_rows count:",
        len(feature_rows)
    )

if "final_feature_dataset" in globals():
    print(
        "final_feature_dataset shape:",
        final_feature_dataset.shape
    )

print()
print("=" * 70)

CURRENT FEATURE DATASET STATE
all_window_signals             : AVAILABLE
feature_rows                   : NOT AVAILABLE
final_feature_dataset          : NOT AVAILABLE
PHYSIOLOGICAL_FEATURES         : NOT AVAILABLE

all_window_signals count: 55



In [39]:
# Cell 14B — Restore the frozen 20 physiological feature definitions

PHYSIOLOGICAL_FEATURES = [
    # ECG features
    "ecg_mean",
    "ecg_std",
    "ecg_rms",
    "r_peak_count",
    "mean_rr",
    "std_rr",
    "mean_hr",
    "min_hr",
    "max_hr",
    "std_hr",
    "sdnn",
    "rmssd",

    # Respiration features
    "resp_mean",
    "resp_std",
    "resp_rms",
    "resp_peak_count",
    "mean_resp_interval",
    "std_resp_interval",
    "mean_resp_rate",
    "std_resp_rate"
]

print("=" * 70)
print("PHYSIOLOGICAL FEATURE DEFINITIONS")
print("=" * 70)
print(f"Total features: {len(PHYSIOLOGICAL_FEATURES)}")
print()
print("ECG features :", 12)
print("Resp features:", 8)
print()
print(PHYSIOLOGICAL_FEATURES)
print("=" * 70)

PHYSIOLOGICAL FEATURE DEFINITIONS
Total features: 20

ECG features : 12
Resp features: 8

['ecg_mean', 'ecg_std', 'ecg_rms', 'r_peak_count', 'mean_rr', 'std_rr', 'mean_hr', 'min_hr', 'max_hr', 'std_hr', 'sdnn', 'rmssd', 'resp_mean', 'resp_std', 'resp_rms', 'resp_peak_count', 'mean_resp_interval', 'std_resp_interval', 'mean_resp_rate', 'std_resp_rate']


In [40]:
# Cell 14C — Inspect the actual structure of all_window_signals

print("=" * 70)
print("ALL_WINDOW_SIGNALS STRUCTURE")
print("=" * 70)

print("Number of windows:", len(all_window_signals))
print()

first_record = all_window_signals[0]

print("First record type:", type(first_record))

if isinstance(first_record, dict):
    print("\nActual dictionary keys:")
    for key in first_record.keys():
        value = first_record[key]

        if isinstance(value, np.ndarray):
            print(f"  {key}: numpy array, shape={value.shape}, dtype={value.dtype}")
        else:
            print(f"  {key}: {type(value).__name__} -> {value}")

else:
    print("\nFirst record contents:")
    print(first_record)

print("=" * 70)

ALL_WINDOW_SIGNALS STRUCTURE
Number of windows: 55

First record type: <class 'dict'>

Actual dictionary keys:
  infant: int -> 1
  event_number: int -> 2
  event_time_s: float64 -> 3456.512
  precursor_start_s: float -> 3441.512
  precursor_end_s: float -> 3456.512
  window_type: str -> precursor
  risk_label: int -> 1
  ecg_fs: float -> 250.0
  resp_fs: float -> 500.0
  ecg_signal: numpy array, shape=(3750,), dtype=float64
  resp_signal: numpy array, shape=(7500,), dtype=float64
  ecg_samples: int -> 3750
  resp_samples: int -> 7500
  filter_context_s: float -> 60.0


In [41]:
# Cell 14D-Diagnostic — Check available signal-processing functions

print("=" * 70)
print("AVAILABLE FEATURE-EXTRACTION DEPENDENCIES")
print("=" * 70)

names_to_check = [
    "find_peaks",
    "filter_ecg",
    "filter_respiration",
    "ecg_peaks",
    "resp_peaks",
    "RESP_PEAK_DISTANCE",
    "RESP_PEAK_PROMINENCE",
    "ECG_PEAK_DISTANCE",
    "ECG_PEAK_PROMINENCE",
]

for name in names_to_check:
    print(
        f"{name:30s}: "
        f"{'AVAILABLE' if name in globals() else 'NOT AVAILABLE'}"
    )

print("=" * 70)

AVAILABLE FEATURE-EXTRACTION DEPENDENCIES
find_peaks                    : AVAILABLE
filter_ecg                    : AVAILABLE
filter_respiration            : AVAILABLE
ecg_peaks                     : NOT AVAILABLE
resp_peaks                    : NOT AVAILABLE
RESP_PEAK_DISTANCE            : NOT AVAILABLE
RESP_PEAK_PROMINENCE          : NOT AVAILABLE
ECG_PEAK_DISTANCE             : NOT AVAILABLE
ECG_PEAK_PROMINENCE           : NOT AVAILABLE


In [42]:
# Cell 14D-Setup — Restore validated Stage 4 feature-extraction parameters

ECG_MIN_RR_SECONDS = 0.3
RESP_MIN_PEAK_DISTANCE_SECONDS = 0.8
RESP_PROMINENCE_FACTOR = 0.1

print("=" * 70)
print("STAGE 4 FEATURE-EXTRACTION PARAMETERS")
print("=" * 70)

print(
    "ECG minimum R-R separation      :",
    ECG_MIN_RR_SECONDS,
    "s"
)

print(
    "Respiration minimum peak gap   :",
    RESP_MIN_PEAK_DISTANCE_SECONDS,
    "s"
)

print(
    "Respiration prominence factor  :",
    RESP_PROMINENCE_FACTOR
)

print()
print("ECG detector  : find_peaks")
print("Resp detector : find_peaks")
print()
print("PASS — Validated Stage 4 parameters restored.")
print("=" * 70)

STAGE 4 FEATURE-EXTRACTION PARAMETERS
ECG minimum R-R separation      : 0.3 s
Respiration minimum peak gap   : 0.8 s
Respiration prominence factor  : 0.1

ECG detector  : find_peaks
Resp detector : find_peaks

PASS — Validated Stage 4 parameters restored.


In [44]:
# Cell 14D — Rebuild the validated 20-feature dataset

from scipy.signal import find_peaks

# ---------------------------------------------------------------
# Reset feature rows because the previous extraction stopped
# partway through.
# ---------------------------------------------------------------

feature_rows = []

print("=" * 70)
print("REBUILDING 20-FEATURE DATASET")
print("=" * 70)

print("Input signal windows:", len(all_window_signals))
print("Expected windows    : 55")
print("Physiological features:", len(PHYSIOLOGICAL_FEATURES))
print()

# ---------------------------------------------------------------
# Extract features from all 55 windows
# ---------------------------------------------------------------

for window_index, window in enumerate(all_window_signals):

    ecg_signal = np.asarray(
        window["ecg_signal"],
        dtype=float
    )

    resp_signal = np.asarray(
        window["resp_signal"],
        dtype=float
    )

    ecg_fs = float(window["ecg_fs"])
    resp_fs = float(window["resp_fs"])

    # -----------------------------------------------------------
    # Signal validity
    # -----------------------------------------------------------

    assert np.all(np.isfinite(ecg_signal)), (
        f"Window {window_index}: ECG contains NaN/Inf."
    )

    assert np.all(np.isfinite(resp_signal)), (
        f"Window {window_index}: respiration contains NaN/Inf."
    )

    # ===========================================================
    # ECG FEATURES
    # ===========================================================

    ecg_mean = np.mean(ecg_signal)
    ecg_std = np.std(ecg_signal)
    ecg_rms = np.sqrt(np.mean(ecg_signal ** 2))

    # Validated ECG minimum R-R separation = 0.3 s
    ecg_peaks, _ = find_peaks(
        ecg_signal,
        distance=int(
            ECG_MIN_RR_SECONDS * ecg_fs
        ),
        prominence=0.3
    )

    r_peak_count = len(ecg_peaks)

    # -----------------------------------------------------------
    # RR intervals
    # -----------------------------------------------------------

    if r_peak_count >= 2:

        ecg_peak_times = (
            ecg_peaks / ecg_fs
        )

        rr_intervals = np.diff(
            ecg_peak_times
        )

        rr_intervals = rr_intervals[
            np.isfinite(rr_intervals)
            & (rr_intervals > 0)
        ]

    else:

        rr_intervals = np.array(
            [],
            dtype=float
        )

    if len(rr_intervals) > 0:

        mean_rr = np.mean(rr_intervals)
        std_rr = np.std(rr_intervals)

        hr_values = (
            60.0 / rr_intervals
        )

        hr_values = hr_values[
            np.isfinite(hr_values)
            & (hr_values > 0)
        ]

        if len(hr_values) > 0:

            mean_hr = np.mean(hr_values)
            min_hr = np.min(hr_values)
            max_hr = np.max(hr_values)
            std_hr = np.std(hr_values)

        else:

            mean_hr = np.nan
            min_hr = np.nan
            max_hr = np.nan
            std_hr = np.nan

    else:

        mean_rr = np.nan
        std_rr = np.nan
        mean_hr = np.nan
        min_hr = np.nan
        max_hr = np.nan
        std_hr = np.nan

    # -----------------------------------------------------------
    # HRV
    # -----------------------------------------------------------

    if len(rr_intervals) >= 2:

        sdnn = (
            np.std(
                rr_intervals,
                ddof=1
            ) * 1000.0
        )

        successive_differences = np.diff(
            rr_intervals
        )

        rmssd = (
            np.sqrt(
                np.mean(
                    successive_differences ** 2
                )
            ) * 1000.0
        )

    else:

        sdnn = np.nan
        rmssd = np.nan

    # ===========================================================
    # RESPIRATION FEATURES
    # ===========================================================

    resp_mean = np.mean(resp_signal)
    resp_std = np.std(resp_signal)
    resp_rms = np.sqrt(
        np.mean(resp_signal ** 2)
    )

    # Validated Stage 4 respiration parameters
    resp_peak_distance = int(
        RESP_MIN_PEAK_DISTANCE_SECONDS
        * resp_fs
    )

    resp_prominence = (
        RESP_PROMINENCE_FACTOR
        * np.std(resp_signal)
    )

    resp_peaks, _ = find_peaks(
        resp_signal,
        distance=resp_peak_distance,
        prominence=resp_prominence
    )

    resp_peak_count = len(resp_peaks)

    # -----------------------------------------------------------
    # Respiration intervals/rates
    #
    # If fewer than 2 peaks are available:
    # retain the window and use NaN for interval/rate features.
    # -----------------------------------------------------------

    if resp_peak_count >= 2:

        resp_peak_times = (
            resp_peaks / resp_fs
        )

        resp_intervals = np.diff(
            resp_peak_times
        )

        resp_intervals = resp_intervals[
            np.isfinite(resp_intervals)
            & (resp_intervals > 0)
        ]

    else:

        resp_intervals = np.array(
            [],
            dtype=float
        )

    if len(resp_intervals) > 0:

        mean_resp_interval = np.mean(
            resp_intervals
        )

        std_resp_interval = np.std(
            resp_intervals
        )

        resp_rates = (
            60.0 / resp_intervals
        )

        resp_rates = resp_rates[
            np.isfinite(resp_rates)
            & (resp_rates > 0)
        ]

        if len(resp_rates) > 0:

            mean_resp_rate = np.mean(
                resp_rates
            )

            std_resp_rate = np.std(
                resp_rates
            )

        else:

            mean_resp_rate = np.nan
            std_resp_rate = np.nan

    else:

        mean_resp_interval = np.nan
        std_resp_interval = np.nan
        mean_resp_rate = np.nan
        std_resp_rate = np.nan

    # ===========================================================
    # METADATA
    # ===========================================================

    # Precursor windows do not have control timing fields.
    control_start_s = window.get(
        "control_start_s",
        np.nan
    )

    control_end_s = window.get(
        "control_end_s",
        np.nan
    )

    # ===========================================================
    # COMPLETE FEATURE ROW
    # ===========================================================

    row = {

        # Metadata — 9
        "infant": window["infant"],
        "event_number": window.get(
            "event_number",
            np.nan
        ),
        "event_time_s": window.get(
            "event_time_s",
            np.nan
        ),
        "precursor_start_s": window.get(
            "precursor_start_s",
            np.nan
        ),
        "precursor_end_s": window.get(
            "precursor_end_s",
            np.nan
        ),
        "control_start_s": control_start_s,
        "control_end_s": control_end_s,
        "risk_label": window["risk_label"],
        "window_type": window["window_type"],

        # ECG — 12
        "ecg_mean": ecg_mean,
        "ecg_std": ecg_std,
        "ecg_rms": ecg_rms,
        "r_peak_count": r_peak_count,
        "mean_rr": mean_rr,
        "std_rr": std_rr,
        "mean_hr": mean_hr,
        "min_hr": min_hr,
        "max_hr": max_hr,
        "std_hr": std_hr,
        "sdnn": sdnn,
        "rmssd": rmssd,

        # Respiration — 8
        "resp_mean": resp_mean,
        "resp_std": resp_std,
        "resp_rms": resp_rms,
        "resp_peak_count": resp_peak_count,
        "mean_resp_interval": mean_resp_interval,
        "std_resp_interval": std_resp_interval,
        "mean_resp_rate": mean_resp_rate,
        "std_resp_rate": std_resp_rate,
    }

    feature_rows.append(row)

# ---------------------------------------------------------------
# Create dataframe
# ---------------------------------------------------------------

final_feature_dataset = pd.DataFrame(
    feature_rows
)

# Exact authoritative column order
metadata_columns = [
    "infant",
    "event_number",
    "event_time_s",
    "precursor_start_s",
    "precursor_end_s",
    "control_start_s",
    "control_end_s",
    "risk_label",
    "window_type"
]

final_feature_dataset = final_feature_dataset[
    metadata_columns
    + PHYSIOLOGICAL_FEATURES
]

# ---------------------------------------------------------------
# Validation
# ---------------------------------------------------------------

expected_shape = (55, 29)

assert final_feature_dataset.shape == expected_shape, (
    f"Expected {expected_shape}, "
    f"got {final_feature_dataset.shape}"
)

assert (
    final_feature_dataset["risk_label"]
    .value_counts()
    .get(0, 0)
    == 50
)

assert (
    final_feature_dataset["risk_label"]
    .value_counts()
    .get(1, 0)
    == 5
)

feature_matrix = final_feature_dataset[
    PHYSIOLOGICAL_FEATURES
]

inf_count = int(
    np.isinf(
        feature_matrix
        .select_dtypes(include=[np.number])
        .to_numpy()
    ).sum()
)

assert inf_count == 0, (
    f"Found {inf_count} infinite feature values."
)

nan_count = int(
    feature_matrix.isna()
    .sum()
    .sum()
)

# ---------------------------------------------------------------
# Summary
# ---------------------------------------------------------------

print("=" * 70)
print("FEATURE EXTRACTION COMPLETE")
print("=" * 70)

print(
    "Dataset shape:",
    final_feature_dataset.shape
)

print()
print("Labels:")
print(
    final_feature_dataset[
        "risk_label"
    ].value_counts().sort_index()
)

print()
print(
    "NaN feature values:",
    nan_count
)

print(
    "Infinite feature values:",
    inf_count
)

print()
print(
    "Expected shape:",
    expected_shape
)

print()
print(
    "PASS — 55 × 29 feature dataset rebuilt."
)

print("=" * 70)

REBUILDING 20-FEATURE DATASET
Input signal windows: 55
Expected windows    : 55
Physiological features: 20

FEATURE EXTRACTION COMPLETE
Dataset shape: (55, 29)

Labels:
risk_label
0    50
1     5
Name: count, dtype: int64

NaN feature values: 0
Infinite feature values: 0

Expected shape: (55, 29)

PASS — 55 × 29 feature dataset rebuilt.


In [45]:
# Cell 14E — Save and independently verify Stage 4 feature dataset

FINAL_DATASET_PATH = (
    REPORTS_DIR / "pics_final_precursor_risk_dataset.csv"
)

final_feature_dataset.to_csv(
    FINAL_DATASET_PATH,
    index=False
)

# ---------------------------------------------------------------
# Independent verification from saved CSV
# ---------------------------------------------------------------

check_dataset = pd.read_csv(
    FINAL_DATASET_PATH
)

print("=" * 70)
print("STAGE 4 FEATURE DATASET CHECKPOINT")
print("=" * 70)

print("Saved file :", FINAL_DATASET_PATH)
print("Shape      :", check_dataset.shape)

print()
print("Labels:")
print(
    check_dataset["risk_label"]
    .value_counts()
    .sort_index()
)

print()

feature_matrix_check = check_dataset[
    PHYSIOLOGICAL_FEATURES
].to_numpy(dtype=float)

print(
    "NaN count  :",
    int(
        np.isnan(feature_matrix_check).sum()
    )
)

print(
    "Inf count  :",
    int(
        np.isinf(feature_matrix_check).sum()
    )
)

print(
    "All finite :",
    np.isfinite(feature_matrix_check).all()
)

print()

assert check_dataset.shape == (55, 29)

assert (
    check_dataset["risk_label"]
    .value_counts()
    .get(0, 0)
    == 50
)

assert (
    check_dataset["risk_label"]
    .value_counts()
    .get(1, 0)
    == 5
)

print(
    "PASS — Stage 4 feature dataset checkpoint saved."
)

print("=" * 70)

STAGE 4 FEATURE DATASET CHECKPOINT
Saved file : c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_final_precursor_risk_dataset.csv
Shape      : (55, 29)

Labels:
risk_label
0    50
1     5
Name: count, dtype: int64

NaN count  : 0
Inf count  : 0
All finite : True

PASS — Stage 4 feature dataset checkpoint saved.


In [47]:
# Cell 15 — Build Leave-One-Subject-Out (LOSO) folds

print("=" * 70)
print("LOSO FOLD CONSTRUCTION")
print("=" * 70)

# Use the frozen dataset
loso_dataset = final_feature_dataset.copy()

infants = sorted(loso_dataset["infant"].unique())

loso_folds = []

for test_infant in infants:

    train_data = loso_dataset[
        loso_dataset["infant"] != test_infant
    ].copy()

    test_data = loso_dataset[
        loso_dataset["infant"] == test_infant
    ].copy()

    loso_folds.append({
        "test_infant": int(test_infant),
        "train_data": train_data,
        "test_data": test_data
    })

# -------------------------------------------------------------
# Verify fold structure
# -------------------------------------------------------------
print(f"Number of infants : {len(infants)}")
print(f"Number of folds   : {len(loso_folds)}")
print()

print("Fold distribution:")
print("-" * 70)

for fold in loso_folds:

    test_infant = fold["test_infant"]
    train_data = fold["train_data"]
    test_data = fold["test_data"]

    train_positive = int((train_data["risk_label"] == 1).sum())
    train_negative = int((train_data["risk_label"] == 0).sum())

    test_positive = int((test_data["risk_label"] == 1).sum())
    test_negative = int((test_data["risk_label"] == 0).sum())

    print(
        f"Test Infant {test_infant:2d} | "
        f"Train: {len(train_data):2d} "
        f"(pos={train_positive}, neg={train_negative}) | "
        f"Test: {len(test_data):1d} "
        f"(pos={test_positive}, neg={test_negative})"
    )

# -------------------------------------------------------------
# Leakage check
# -------------------------------------------------------------
print()
print("=" * 70)
print("LEAKAGE CHECK")
print("=" * 70)

leakage_found = False

for fold in loso_folds:

    train_infants = set(fold["train_data"]["infant"].unique())
    test_infants = set(fold["test_data"]["infant"].unique())

    overlap = train_infants.intersection(test_infants)

    if overlap:
        leakage_found = True
        print(
            f"FAIL — Fold {fold['test_infant']} "
            f"has overlapping infants: {overlap}"
        )

if not leakage_found:
    print("PASS — No infant appears in both train and test.")

print("=" * 70)

LOSO FOLD CONSTRUCTION
Number of infants : 10
Number of folds   : 10

Fold distribution:
----------------------------------------------------------------------
Test Infant  1 | Train: 49 (pos=4, neg=45) | Test: 6 (pos=1, neg=5)
Test Infant  2 | Train: 50 (pos=5, neg=45) | Test: 5 (pos=0, neg=5)
Test Infant  3 | Train: 48 (pos=3, neg=45) | Test: 7 (pos=2, neg=5)
Test Infant  4 | Train: 50 (pos=5, neg=45) | Test: 5 (pos=0, neg=5)
Test Infant  5 | Train: 50 (pos=5, neg=45) | Test: 5 (pos=0, neg=5)
Test Infant  6 | Train: 50 (pos=5, neg=45) | Test: 5 (pos=0, neg=5)
Test Infant  7 | Train: 50 (pos=5, neg=45) | Test: 5 (pos=0, neg=5)
Test Infant  8 | Train: 50 (pos=5, neg=45) | Test: 5 (pos=0, neg=5)
Test Infant  9 | Train: 48 (pos=3, neg=45) | Test: 7 (pos=2, neg=5)
Test Infant 10 | Train: 50 (pos=5, neg=45) | Test: 5 (pos=0, neg=5)

LEAKAGE CHECK
PASS — No infant appears in both train and test.


In [48]:
# Cell 16 — LOSO baseline Random Forest
#
# Status: DEVELOPMENT / VALIDATION
# Purpose: Obtain strictly subject-independent out-of-fold predictions.
#
# Important:
# - Each fold trains only on the other infants.
# - No test infant is used during training.
# - No artificial sensitivity/recall is assigned to folds
#   containing zero positive test samples.

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=" * 70)
print("LOSO RANDOM FOREST BASELINE")
print("=" * 70)

X_all = final_feature_dataset[PHYSIOLOGICAL_FEATURES].to_numpy(dtype=float)
y_all = final_feature_dataset["risk_label"].to_numpy(dtype=int)

oof_predictions = []
fold_results = []

for fold in loso_folds:

    test_infant = fold["test_infant"]
    train_data = fold["train_data"]
    test_data = fold["test_data"]

    # ---------------------------------------------------------
    # Training data
    # ---------------------------------------------------------
    X_train = train_data[PHYSIOLOGICAL_FEATURES].to_numpy(dtype=float)
    y_train = train_data["risk_label"].to_numpy(dtype=int)

    # ---------------------------------------------------------
    # Test data
    # ---------------------------------------------------------
    X_test = test_data[PHYSIOLOGICAL_FEATURES].to_numpy(dtype=float)
    y_test = test_data["risk_label"].to_numpy(dtype=int)

    # ---------------------------------------------------------
    # Random Forest
    # ---------------------------------------------------------
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_SEED,
        class_weight="balanced",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # Probability and class prediction
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.50).astype(int)

    # ---------------------------------------------------------
    # Store OOF predictions
    # ---------------------------------------------------------
    for i in range(len(test_data)):

        oof_predictions.append({
            "infant": int(test_data.iloc[i]["infant"]),
            "risk_label": int(y_test[i]),
            "predicted_label": int(y_pred[i]),
            "predicted_probability": float(y_prob[i])
        })

    # ---------------------------------------------------------
    # Fold metrics
    # ---------------------------------------------------------
    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_test, y_pred)

    # Precision is defined even if there are no positive
    # ground-truth samples, but may be zero when no positive
    # predictions are produced.
    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    # Recall/sensitivity is undefined if the test fold contains
    # no positive samples.
    if np.sum(y_test == 1) > 0:
        sensitivity = recall_score(
            y_test,
            y_pred,
            zero_division=0
        )
    else:
        sensitivity = np.nan

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    # AUROC requires both classes in the test set.
    if len(np.unique(y_test)) == 2:
        auroc = roc_auc_score(y_test, y_prob)
    else:
        auroc = np.nan

    # AUPRC requires positive samples.
    if np.sum(y_test == 1) > 0:
        auprc = average_precision_score(y_test, y_prob)
    else:
        auprc = np.nan

    fold_results.append({
        "test_infant": test_infant,
        "n_train": len(train_data),
        "n_test": len(test_data),
        "train_positive": int(np.sum(y_train == 1)),
        "train_negative": int(np.sum(y_train == 0)),
        "test_positive": int(np.sum(y_test == 1)),
        "test_negative": int(np.sum(y_test == 0)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy,
        "precision": precision,
        "sensitivity": sensitivity,
        "f1": f1,
        "auroc": auroc,
        "auprc": auprc
    })


# -------------------------------------------------------------
# Convert results to DataFrames
# -------------------------------------------------------------
loso_fold_summary = pd.DataFrame(fold_results)
loso_oof_predictions = pd.DataFrame(oof_predictions)

print()
print("Fold results:")
display(loso_fold_summary)

print()
print("=" * 70)
print("POOLED OOF CONFUSION MATRIX")
print("=" * 70)

cm_pooled = confusion_matrix(
    loso_oof_predictions["risk_label"],
    loso_oof_predictions["predicted_label"],
    labels=[0, 1]
)

print(cm_pooled)

tn, fp, fn, tp = cm_pooled.ravel()

print()
print(f"TN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")
print(f"TP = {tp}")

print()
print("=" * 70)
print("POOLED OOF METRICS")
print("=" * 70)

pooled_accuracy = accuracy_score(
    loso_oof_predictions["risk_label"],
    loso_oof_predictions["predicted_label"]
)

pooled_precision = precision_score(
    loso_oof_predictions["risk_label"],
    loso_oof_predictions["predicted_label"],
    zero_division=0
)

pooled_sensitivity = recall_score(
    loso_oof_predictions["risk_label"],
    loso_oof_predictions["predicted_label"],
    zero_division=0
)

pooled_f1 = f1_score(
    loso_oof_predictions["risk_label"],
    loso_oof_predictions["predicted_label"],
    zero_division=0
)

pooled_auroc = roc_auc_score(
    loso_oof_predictions["risk_label"],
    loso_oof_predictions["predicted_probability"]
)

pooled_auprc = average_precision_score(
    loso_oof_predictions["risk_label"],
    loso_oof_predictions["predicted_probability"]
)

print(f"Accuracy    : {pooled_accuracy:.4f}")
print(f"Sensitivity : {pooled_sensitivity:.4f}")
print(f"Precision   : {pooled_precision:.4f}")
print(f"F1          : {pooled_f1:.4f}")
print(f"AUROC       : {pooled_auroc:.4f}")
print(f"AUPRC       : {pooled_auprc:.4f}")

print()
print("=" * 70)
print("STATUS: DEVELOPMENT / VALIDATION RESULT")
print("=" * 70)

LOSO RANDOM FOREST BASELINE

Fold results:


,test_infant,n_train,n_test,train_positive,train_negative,test_positive,test_negative,tn,fp,fn,tp,accuracy,precision,sensitivity,f1,auroc,auprc
0,1,49,6,4,45,1,5,5,0,1,0,0.833333,0.0,0.0,0.000000,1.0,1.000000
1,2,50,5,5,45,0,5,4,1,0,0,0.800000,0.0,NaN,0.000000,NaN,NaN
2,3,48,7,3,45,2,5,5,0,1,1,0.857143,1.0,0.5,0.666667,0.5,0.642857
3,4,50,5,5,45,0,5,5,0,0,0,1.000000,0.0,NaN,0.000000,NaN,NaN
4,5,50,5,5,45,0,5,4,1,0,0,0.800000,0.0,NaN,0.000000,NaN,NaN
5,6,50,5,5,45,0,5,4,1,0,0,0.800000,0.0,NaN,0.000000,NaN,NaN
6,7,50,5,5,45,0,5,5,0,0,0,1.000000,0.0,NaN,0.000000,NaN,NaN
7,8,50,5,5,45,0,5,5,0,0,0,1.000000,0.0,NaN,0.000000,NaN,NaN
8,9,48,7,3,45,2,5,5,0,2,0,0.714286,0.0,0.0,0.000000,1.0,1.000000
9,10,50,5,5,45,0,5,5,0,0,0,1.000000,0.0,NaN,0.000000,NaN,NaN



POOLED OOF CONFUSION MATRIX
[[47  3]
 [ 4  1]]

TN = 47
FP = 3
FN = 4
TP = 1

POOLED OOF METRICS
Accuracy    : 0.8727
Sensitivity : 0.2000
Precision   : 0.2500
F1          : 0.2222
AUROC       : 0.5880
AUPRC       : 0.2048

STATUS: DEVELOPMENT / VALIDATION RESULT


In [49]:
# Cell 17 — Save LOSO baseline results

# -------------------------------------------------------------
# Save fold-level results
# -------------------------------------------------------------
LOSO_FOLD_PATH = REPORTS_DIR / "pics_final_loso_fold_summary.csv"
loso_fold_summary.to_csv(
    LOSO_FOLD_PATH,
    index=False
)

# -------------------------------------------------------------
# Save pooled out-of-fold predictions
# -------------------------------------------------------------
LOSO_PRED_PATH = REPORTS_DIR / "pics_final_loso_predictions.csv"
loso_oof_predictions.to_csv(
    LOSO_PRED_PATH,
    index=False
)

# -------------------------------------------------------------
# Calculate specificity from pooled OOF predictions
# -------------------------------------------------------------
specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

# -------------------------------------------------------------
# Create one-row metrics table
# -------------------------------------------------------------
loso_metrics = pd.DataFrame([{
    "model": "Random Forest",
    "validation": "LOSO",
    "n_infants": len(infants),
    "n_windows": len(loso_oof_predictions),
    "positive_windows": int(
        (loso_oof_predictions["risk_label"] == 1).sum()
    ),
    "negative_windows": int(
        (loso_oof_predictions["risk_label"] == 0).sum()
    ),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp),
    "accuracy": pooled_accuracy,
    "sensitivity": pooled_sensitivity,
    "specificity": specificity,
    "precision": pooled_precision,
    "f1": pooled_f1,
    "auroc": pooled_auroc,
    "auprc": pooled_auprc,
    "status": "DEVELOPMENT / VALIDATION RESULT"
}])

LOSO_METRICS_PATH = REPORTS_DIR / "pics_final_loso_metrics.csv"

loso_metrics.to_csv(
    LOSO_METRICS_PATH,
    index=False
)

# -------------------------------------------------------------
# Verification
# -------------------------------------------------------------
print("=" * 70)
print("LOSO BASELINE ARTIFACTS SAVED")
print("=" * 70)

print(f"Fold summary      : {LOSO_FOLD_PATH}")
print(f"OOF predictions    : {LOSO_PRED_PATH}")
print(f"Metrics            : {LOSO_METRICS_PATH}")
print()

print("Specificity:", f"{specificity:.4f}")
print()

display(loso_metrics)

print()
print("PASS — LOSO baseline artifacts saved.")
print("=" * 70)

LOSO BASELINE ARTIFACTS SAVED
Fold summary      : c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_final_loso_fold_summary.csv
OOF predictions    : c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_final_loso_predictions.csv
Metrics            : c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_final_loso_metrics.csv

Specificity: 0.9400



,model,validation,n_infants,n_windows,positive_windows,negative_windows,tn,fp,fn,tp,accuracy,sensitivity,specificity,precision,f1,auroc,auprc,status
0,Random Forest,LOSO,10,55,5,50,47,3,4,1,0.872727,0.2,0.94,0.25,0.222222,0.588,0.204753,DEVELOPMENT / VALIDATION RESULT



PASS — LOSO baseline artifacts saved.


In [50]:
# Cell 17 — Save LOSO baseline results

# -------------------------------------------------------------
# Save fold-level results
# -------------------------------------------------------------
LOSO_FOLD_PATH = REPORTS_DIR / "pics_final_loso_fold_summary.csv"
loso_fold_summary.to_csv(
    LOSO_FOLD_PATH,
    index=False
)

# -------------------------------------------------------------
# Save pooled out-of-fold predictions
# -------------------------------------------------------------
LOSO_PRED_PATH = REPORTS_DIR / "pics_final_loso_predictions.csv"
loso_oof_predictions.to_csv(
    LOSO_PRED_PATH,
    index=False
)

# -------------------------------------------------------------
# Calculate specificity from pooled OOF predictions
# -------------------------------------------------------------
specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

# -------------------------------------------------------------
# Create one-row metrics table
# -------------------------------------------------------------
loso_metrics = pd.DataFrame([{
    "model": "Random Forest",
    "validation": "LOSO",
    "n_infants": len(infants),
    "n_windows": len(loso_oof_predictions),
    "positive_windows": int(
        (loso_oof_predictions["risk_label"] == 1).sum()
    ),
    "negative_windows": int(
        (loso_oof_predictions["risk_label"] == 0).sum()
    ),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp),
    "accuracy": pooled_accuracy,
    "sensitivity": pooled_sensitivity,
    "specificity": specificity,
    "precision": pooled_precision,
    "f1": pooled_f1,
    "auroc": pooled_auroc,
    "auprc": pooled_auprc,
    "status": "DEVELOPMENT / VALIDATION RESULT"
}])

LOSO_METRICS_PATH = REPORTS_DIR / "pics_final_loso_metrics.csv"

loso_metrics.to_csv(
    LOSO_METRICS_PATH,
    index=False
)

# -------------------------------------------------------------
# Verification
# -------------------------------------------------------------
print("=" * 70)
print("LOSO BASELINE ARTIFACTS SAVED")
print("=" * 70)

print(f"Fold summary      : {LOSO_FOLD_PATH}")
print(f"OOF predictions    : {LOSO_PRED_PATH}")
print(f"Metrics            : {LOSO_METRICS_PATH}")
print()

print("Specificity:", f"{specificity:.4f}")
print()

display(loso_metrics)

print()
print("PASS — LOSO baseline artifacts saved.")
print("=" * 70)

LOSO BASELINE ARTIFACTS SAVED
Fold summary      : c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_final_loso_fold_summary.csv
OOF predictions    : c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_final_loso_predictions.csv
Metrics            : c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_final_loso_metrics.csv

Specificity: 0.9400



,model,validation,n_infants,n_windows,positive_windows,negative_windows,tn,fp,fn,tp,accuracy,sensitivity,specificity,precision,f1,auroc,auprc,status
0,Random Forest,LOSO,10,55,5,50,47,3,4,1,0.872727,0.2,0.94,0.25,0.222222,0.588,0.204753,DEVELOPMENT / VALIDATION RESULT



PASS — LOSO baseline artifacts saved.


In [51]:
# Cell 18 — LOSO Random Forest feature importance
#
# Status: DEVELOPMENT
# Purpose: Measure feature importance across independent LOSO folds.

from sklearn.ensemble import RandomForestClassifier

feature_importance_rows = []

for fold in loso_folds:

    test_infant = fold["test_infant"]
    train_data = fold["train_data"]

    X_train = train_data[PHYSIOLOGICAL_FEATURES].to_numpy(dtype=float)
    y_train = train_data["risk_label"].to_numpy(dtype=int)

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_SEED,
        class_weight="balanced",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    for feature, importance in zip(
        PHYSIOLOGICAL_FEATURES,
        model.feature_importances_
    ):
        feature_importance_rows.append({
            "test_infant": int(test_infant),
            "feature": feature,
            "importance": float(importance)
        })


# -------------------------------------------------------------
# Create fold-level importance table
# -------------------------------------------------------------
loso_feature_importance_folds = pd.DataFrame(
    feature_importance_rows
)

# -------------------------------------------------------------
# Aggregate across the 10 LOSO folds
# -------------------------------------------------------------
loso_feature_importance = (
    loso_feature_importance_folds
    .groupby("feature")["importance"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("mean", ascending=False)
    .reset_index(drop=True)
)

loso_feature_importance.columns = [
    "feature",
    "mean_importance",
    "std_importance"
]

# -------------------------------------------------------------
# Save results
# -------------------------------------------------------------
FEATURE_IMPORTANCE_PATH = (
    REPORTS_DIR / "pics_final_loso_rf_feature_importance.csv"
)

loso_feature_importance.to_csv(
    FEATURE_IMPORTANCE_PATH,
    index=False
)

# -------------------------------------------------------------
# Display
# -------------------------------------------------------------
print("=" * 70)
print("LOSO RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 70)

print(f"Folds analysed : {len(infants)}")
print(f"Features       : {len(PHYSIOLOGICAL_FEATURES)}")
print()

display(loso_feature_importance)

print()
print(f"Saved to: {FEATURE_IMPORTANCE_PATH}")

print()
print("=" * 70)
print("TOP 5 FEATURES")
print("=" * 70)

display(loso_feature_importance.head(5))

print()
print("PASS — Feature importance calculated across all LOSO folds.")
print("=" * 70)

LOSO RANDOM FOREST FEATURE IMPORTANCE
Folds analysed : 10
Features       : 20



,feature,mean_importance,std_importance
0,mean_resp_rate,0.121180,0.023547
1,ecg_rms,0.084093,0.028864
2,ecg_std,0.075294,0.025920
3,resp_std,0.062633,0.014278
4,resp_rms,0.062186,0.014314
5,mean_resp_interval,0.059874,0.014679
6,mean_hr,0.057164,0.027134
7,std_resp_rate,0.053313,0.009988
8,min_hr,0.052658,0.015157
9,max_hr,0.050453,0.020601



Saved to: c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_final_loso_rf_feature_importance.csv

TOP 5 FEATURES


,feature,mean_importance,std_importance
0,mean_resp_rate,0.121180,0.023547
1,ecg_rms,0.084093,0.028864
2,ecg_std,0.075294,0.025920
3,resp_std,0.062633,0.014278
4,resp_rms,0.062186,0.014314



PASS — Feature importance calculated across all LOSO folds.


In [52]:
# Cell 19 — Stage 4 final checkpoint

STAGE4_CHECKPOINT = {
    "stage": "Stage 4 — Precursor Risk Dataset and Baseline",
    "status": "DEVELOPMENT / VALIDATION",
    "dataset_version": "1.0-development",
    "n_infants": int(final_feature_dataset["infant"].nunique()),
    "n_windows": int(len(final_feature_dataset)),
    "n_controls": int((final_feature_dataset["risk_label"] == 0).sum()),
    "n_precursors": int((final_feature_dataset["risk_label"] == 1).sum()),
    "window_duration_s": 15,
    "n_features": len(PHYSIOLOGICAL_FEATURES),
    "loso_folds": len(loso_folds),
    "loso_accuracy": float(pooled_accuracy),
    "loso_sensitivity": float(pooled_sensitivity),
    "loso_specificity": float(specificity),
    "loso_precision": float(pooled_precision),
    "loso_f1": float(pooled_f1),
    "loso_auroc": float(pooled_auroc),
    "loso_auprc": float(pooled_auprc),
    "dataset_file": str(FINAL_DATASET_PATH),
    "loso_metrics_file": str(LOSO_METRICS_PATH),
    "loso_predictions_file": str(LOSO_PRED_PATH),
    "feature_importance_file": str(FEATURE_IMPORTANCE_PATH)
}

STAGE4_CHECKPOINT_PATH = REPORTS_DIR / "pics_stage4_checkpoint.json"

with open(STAGE4_CHECKPOINT_PATH, "w") as f:
    json.dump(STAGE4_CHECKPOINT, f, indent=4)

print("=" * 70)
print("STAGE 4 CHECKPOINT")
print("=" * 70)

print(f"Dataset windows : {STAGE4_CHECKPOINT['n_windows']}")
print(f"Controls        : {STAGE4_CHECKPOINT['n_controls']}")
print(f"Precursors      : {STAGE4_CHECKPOINT['n_precursors']}")
print(f"Features        : {STAGE4_CHECKPOINT['n_features']}")
print(f"LOSO folds      : {STAGE4_CHECKPOINT['loso_folds']}")
print()
print(f"Accuracy        : {STAGE4_CHECKPOINT['loso_accuracy']:.4f}")
print(f"Sensitivity     : {STAGE4_CHECKPOINT['loso_sensitivity']:.4f}")
print(f"Specificity     : {STAGE4_CHECKPOINT['loso_specificity']:.4f}")
print(f"Precision       : {STAGE4_CHECKPOINT['loso_precision']:.4f}")
print(f"F1              : {STAGE4_CHECKPOINT['loso_f1']:.4f}")
print(f"AUROC           : {STAGE4_CHECKPOINT['loso_auroc']:.4f}")
print(f"AUPRC           : {STAGE4_CHECKPOINT['loso_auprc']:.4f}")
print()
print(f"Checkpoint saved: {STAGE4_CHECKPOINT_PATH}")
print()
print("PASS — Stage 4 checkpoint saved.")
print("=" * 70)

STAGE 4 CHECKPOINT
Dataset windows : 55
Controls        : 50
Precursors      : 5
Features        : 20
LOSO folds      : 10

Accuracy        : 0.8727
Sensitivity     : 0.2000
Specificity     : 0.9400
Precision       : 0.2500
F1              : 0.2222
AUROC           : 0.5880
AUPRC           : 0.2048

Checkpoint saved: c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_stage4_checkpoint.json

PASS — Stage 4 checkpoint saved.


In [54]:
# Cell 20 — Construct 15-second windows as temporal feature sequences
#
# Each 15-second window:
#   -> 5-second feature windows
#   -> 1-second step
#   -> 11 temporal steps
#   -> 20 physiological features per step
#
# Missing-value policy:
#   If a 5-second respiration segment has fewer than
#   2 valid respiration peaks, interval/rate features
#   are set to NaN rather than discarding the sequence.
#
# Status: DEVELOPMENT
# Purpose: Prepare sequence-shaped data for GRU / 1D-CNN.

from scipy.signal import find_peaks

SEQUENCE_WINDOW_S = 15.0
FEATURE_WINDOW_S = 5.0
FEATURE_STEP_S = 1.0

N_SEQUENCE_STEPS = int(
    (SEQUENCE_WINDOW_S - FEATURE_WINDOW_S) /
    FEATURE_STEP_S
) + 1

print("=" * 70)
print("TEMPORAL SEQUENCE CONSTRUCTION")
print("=" * 70)

print(f"Sequence duration : {SEQUENCE_WINDOW_S:.1f} s")
print(f"Feature duration  : {FEATURE_WINDOW_S:.1f} s")
print(f"Feature step      : {FEATURE_STEP_S:.1f} s")
print(f"Time steps        : {N_SEQUENCE_STEPS}")
print(f"Features / step   : {len(PHYSIOLOGICAL_FEATURES)}")
print()


def extract_temporal_features(ecg, resp, ecg_fs, resp_fs):
    """
    Extract the same 20 physiological features used
    in the frozen Stage 4 dataset.

    If fewer than two valid respiration peaks are present,
    respiration interval/rate features are set to NaN.
    The temporal window itself is retained.
    """

    ecg = np.asarray(ecg, dtype=float)
    resp = np.asarray(resp, dtype=float)

    # ---------------------------------------------------------
    # ECG features
    # ---------------------------------------------------------
    ecg_peaks, _ = find_peaks(
        ecg,
        distance=int(0.30 * ecg_fs),
        prominence=0.3
    )

    ecg_peak_times = ecg_peaks / ecg_fs

    if len(ecg_peak_times) > 1:

        rr = np.diff(ecg_peak_times)

        valid_rr = rr[
            (rr >= 0.30) &
            (rr <= 2.00)
        ]

        if len(valid_rr) == 0:
            return None

        hr = 60.0 / valid_rr

        mean_rr = np.mean(valid_rr)
        std_rr = np.std(valid_rr)

        mean_hr = np.mean(hr)
        min_hr = np.min(hr)
        max_hr = np.max(hr)
        std_hr = np.std(hr)

        sdnn = np.std(valid_rr * 1000.0)

        if len(valid_rr) > 1:
            successive_diff = np.diff(valid_rr * 1000.0)
            rmssd = np.sqrt(
                np.mean(successive_diff ** 2)
            )
        else:
            rmssd = np.nan

    else:
        return None

    # ---------------------------------------------------------
    # Respiration features
    # ---------------------------------------------------------
    resp_prominence = max(
        1e-6,
        0.10 * np.std(resp)
    )

    resp_peaks, _ = find_peaks(
        resp,
        distance=int(0.80 * resp_fs),
        prominence=resp_prominence
    )

    resp_peak_times = resp_peaks / resp_fs

    # Basic respiration waveform features are always retained
    resp_mean = np.mean(resp)
    resp_std = np.std(resp)
    resp_rms = np.sqrt(np.mean(resp ** 2))
    resp_peak_count = len(resp_peaks)

    # ---------------------------------------------------------
    # Respiration interval/rate features
    # ---------------------------------------------------------
    if len(resp_peak_times) > 1:

        resp_intervals = np.diff(resp_peak_times)

        valid_resp_intervals = resp_intervals[
            (resp_intervals >= 0.80) &
            (resp_intervals <= 10.0)
        ]

        if len(valid_resp_intervals) > 0:

            mean_resp_interval = np.mean(
                valid_resp_intervals
            )

            std_resp_interval = np.std(
                valid_resp_intervals
            )

            resp_rates = 60.0 / valid_resp_intervals

            mean_resp_rate = np.mean(resp_rates)
            std_resp_rate = np.std(resp_rates)

        else:

            mean_resp_interval = np.nan
            std_resp_interval = np.nan
            mean_resp_rate = np.nan
            std_resp_rate = np.nan

    else:

        mean_resp_interval = np.nan
        std_resp_interval = np.nan
        mean_resp_rate = np.nan
        std_resp_rate = np.nan

    # ---------------------------------------------------------
    # Return exact 20-feature vector
    # ---------------------------------------------------------
    features = [
        # ECG
        np.mean(ecg),
        np.std(ecg),
        np.sqrt(np.mean(ecg ** 2)),
        len(ecg_peaks),
        mean_rr,
        std_rr,
        mean_hr,
        min_hr,
        max_hr,
        std_hr,
        sdnn,
        rmssd,

        # Respiration
        resp_mean,
        resp_std,
        resp_rms,
        resp_peak_count,
        mean_resp_interval,
        std_resp_interval,
        mean_resp_rate,
        std_resp_rate
    ]

    features = np.asarray(
        features,
        dtype=float
    )

    # Only reject infinite values.
    # NaN is intentionally allowed for unavailable
    # respiration interval/rate features.
    if np.isinf(features).any():
        return None

    return features


# -------------------------------------------------------------
# Build sequences
# -------------------------------------------------------------
sequence_rows = []
sequence_data = []
sequence_labels = []

failed_sequences = 0

for window_index, record in enumerate(all_window_signals):

    ecg = np.asarray(
        record["ecg_signal"],
        dtype=float
    )

    resp = np.asarray(
        record["resp_signal"],
        dtype=float
    )

    ecg_fs = float(record["ecg_fs"])
    resp_fs = float(record["resp_fs"])

    # Number of samples corresponding to each feature window
    ecg_feature_samples = int(
        FEATURE_WINDOW_S * ecg_fs
    )

    resp_feature_samples = int(
        FEATURE_WINDOW_S * resp_fs
    )

    ecg_step_samples = int(
        FEATURE_STEP_S * ecg_fs
    )

    resp_step_samples = int(
        FEATURE_STEP_S * resp_fs
    )

    temporal_features = []

    for step in range(N_SEQUENCE_STEPS):

        ecg_start = step * ecg_step_samples
        ecg_end = (
            ecg_start +
            ecg_feature_samples
        )

        resp_start = step * resp_step_samples
        resp_end = (
            resp_start +
            resp_feature_samples
        )

        ecg_segment = ecg[
            ecg_start:ecg_end
        ]

        resp_segment = resp[
            resp_start:resp_end
        ]

        # Exact 5-second segment check
        if (
            len(ecg_segment) != ecg_feature_samples
            or
            len(resp_segment) != resp_feature_samples
        ):
            temporal_features = []
            break

        feature_vector = extract_temporal_features(
            ecg_segment,
            resp_segment,
            ecg_fs,
            resp_fs
        )

        if feature_vector is None:
            temporal_features = []
            break

        temporal_features.append(
            feature_vector
        )

    # ---------------------------------------------------------
    # Store complete sequence
    # ---------------------------------------------------------
    if len(temporal_features) == N_SEQUENCE_STEPS:

        sequence_array = np.asarray(
            temporal_features,
            dtype=float
        )

        sequence_data.append(
            sequence_array
        )

        sequence_labels.append(
            int(record["risk_label"])
        )

        sequence_rows.append({
            "sequence_index": len(sequence_rows),
            "source_window_index": window_index,
            "infant": int(record["infant"]),
            "event_number": record.get(
                "event_number",
                np.nan
            ),
            "event_time_s": record.get(
                "event_time_s",
                np.nan
            ),
            "window_type": record["window_type"],
            "risk_label": int(
                record["risk_label"]
            )
        })

    else:
        failed_sequences += 1


# -------------------------------------------------------------
# Convert to arrays
# -------------------------------------------------------------
X_sequences = np.asarray(
    sequence_data,
    dtype=float
)

y_sequences = np.asarray(
    sequence_labels,
    dtype=int
)

sequence_metadata = pd.DataFrame(
    sequence_rows
)


# -------------------------------------------------------------
# Validation
# -------------------------------------------------------------
print("=" * 70)
print("TEMPORAL SEQUENCE RESULT")
print("=" * 70)

print(f"Sequences created : {len(X_sequences)}")
print(f"Expected           : 55")
print(f"Failed             : {failed_sequences}")
print()

print(f"X shape            : {X_sequences.shape}")
print(f"Expected shape     : (55, 11, 20)")
print()

print("Label distribution:")
print(
    pd.Series(y_sequences)
    .value_counts()
    .sort_index()
)
print()

missing_count = int(
    np.isnan(X_sequences).sum()
)

total_values = int(
    X_sequences.size
)

missing_percent = (
    100.0 * missing_count / total_values
    if total_values > 0
    else np.nan
)

print(f"Missing values     : {missing_count}")
print(f"Missing percentage : {missing_percent:.2f}%")

print()

print(
    "All non-missing values finite:",
    bool(
        np.isfinite(
            X_sequences[
                ~np.isnan(X_sequences)
            ]
        ).all()
    )
)

print()

if (
    X_sequences.shape == (55, 11, 20)
    and failed_sequences == 0
    and np.isfinite(
        X_sequences[
            ~np.isnan(X_sequences)
        ]
    ).all()
):
    print(
        "PASS — Temporal sequence dataset "
        "constructed successfully."
    )
else:
    print(
        "CHECK REQUIRED — Sequence dimensions "
        "or feature extraction need inspection."
    )

print("=" * 70)

TEMPORAL SEQUENCE CONSTRUCTION
Sequence duration : 15.0 s
Feature duration  : 5.0 s
Feature step      : 1.0 s
Time steps        : 11
Features / step   : 20

TEMPORAL SEQUENCE RESULT
Sequences created : 55
Expected           : 55
Failed             : 0

X shape            : (55, 11, 20)
Expected shape     : (55, 11, 20)

Label distribution:
0    50
1     5
Name: count, dtype: int64

Missing values     : 60
Missing percentage : 0.50%

All non-missing values finite: True

PASS — Temporal sequence dataset constructed successfully.


In [55]:
# Cell 20C — Construct temporal sequences with explicit missing-feature handling
#
# Status: DEVELOPMENT
#
# Important:
# - The frozen 55 x 29 Stage 4 dataset is NOT modified.
# - A 5-second window with only one respiration peak is retained.
# - Respiration interval/rate features requiring >=2 peaks are NaN.
# - NaN handling will be performed inside each LOSO training fold.
#   This prevents information leakage from the held-out infant.

from scipy.signal import find_peaks

SEQUENCE_WINDOW_S = 15.0
FEATURE_WINDOW_S = 5.0
FEATURE_STEP_S = 1.0

N_SEQUENCE_STEPS = int(
    (SEQUENCE_WINDOW_S - FEATURE_WINDOW_S) /
    FEATURE_STEP_S
) + 1


def extract_temporal_features_with_missing(
    ecg,
    resp,
    ecg_fs,
    resp_fs
):
    """
    Extract the same 20 physiological features used in Stage 4.

    Features that cannot be calculated from a short temporal
    subwindow are represented as NaN rather than fabricating
    a value.
    """

    ecg = np.asarray(ecg, dtype=float)
    resp = np.asarray(resp, dtype=float)

    # ---------------------------------------------------------
    # ECG
    # ---------------------------------------------------------
    ecg_peaks, _ = find_peaks(
        ecg,
        distance=int(0.30 * ecg_fs),
        prominence=0.3
    )

    ecg_peak_times = ecg_peaks / ecg_fs

    if len(ecg_peak_times) > 1:

        rr = np.diff(ecg_peak_times)

        valid_rr = rr[
            (rr >= 0.30) &
            (rr <= 2.00)
        ]

    else:
        valid_rr = np.array([])

    if len(valid_rr) > 0:

        hr = 60.0 / valid_rr

        mean_rr = np.mean(valid_rr)
        std_rr = np.std(valid_rr)

        mean_hr = np.mean(hr)
        min_hr = np.min(hr)
        max_hr = np.max(hr)
        std_hr = np.std(hr)

        sdnn = np.std(valid_rr * 1000.0)

        if len(valid_rr) > 1:
            successive_diff = np.diff(valid_rr * 1000.0)
            rmssd = np.sqrt(
                np.mean(successive_diff ** 2)
            )
        else:
            rmssd = np.nan

    else:

        mean_rr = np.nan
        std_rr = np.nan
        mean_hr = np.nan
        min_hr = np.nan
        max_hr = np.nan
        std_hr = np.nan
        sdnn = np.nan
        rmssd = np.nan

    # ---------------------------------------------------------
    # Respiration
    # ---------------------------------------------------------
    resp_prominence = max(
        1e-6,
        0.10 * np.std(resp)
    )

    resp_peaks, _ = find_peaks(
        resp,
        distance=int(0.80 * resp_fs),
        prominence=resp_prominence
    )

    resp_peak_times = resp_peaks / resp_fs

    if len(resp_peak_times) > 1:

        resp_intervals = np.diff(resp_peak_times)

        valid_resp_intervals = resp_intervals[
            (resp_intervals >= 0.80) &
            (resp_intervals <= 10.0)
        ]

    else:
        valid_resp_intervals = np.array([])

    if len(valid_resp_intervals) > 0:

        mean_resp_interval = np.mean(
            valid_resp_intervals
        )

        std_resp_interval = np.std(
            valid_resp_intervals
        )

        resp_rates = 60.0 / valid_resp_intervals

        mean_resp_rate = np.mean(resp_rates)
        std_resp_rate = np.std(resp_rates)

    else:

        # Not enough respiratory peaks to calculate
        # interval-derived features.
        mean_resp_interval = np.nan
        std_resp_interval = np.nan
        mean_resp_rate = np.nan
        std_resp_rate = np.nan

    # ---------------------------------------------------------
    # Exact 20-feature vector
    # ---------------------------------------------------------
    features = np.asarray([
        # ECG — 12
        np.mean(ecg),
        np.std(ecg),
        np.sqrt(np.mean(ecg ** 2)),
        len(ecg_peaks),
        mean_rr,
        std_rr,
        mean_hr,
        min_hr,
        max_hr,
        std_hr,
        sdnn,
        rmssd,

        # Respiration — 8
        np.mean(resp),
        np.std(resp),
        np.sqrt(np.mean(resp ** 2)),
        len(resp_peaks),
        mean_resp_interval,
        std_resp_interval,
        mean_resp_rate,
        std_resp_rate
    ], dtype=float)

    return features


# -------------------------------------------------------------
# Build all 55 sequences
# -------------------------------------------------------------
sequence_data = []
sequence_labels = []
sequence_rows = []

for window_index, record in enumerate(all_window_signals):

    ecg = np.asarray(record["ecg_signal"], dtype=float)
    resp = np.asarray(record["resp_signal"], dtype=float)

    ecg_fs = float(record["ecg_fs"])
    resp_fs = float(record["resp_fs"])

    ecg_feature_samples = int(
        FEATURE_WINDOW_S * ecg_fs
    )

    resp_feature_samples = int(
        FEATURE_WINDOW_S * resp_fs
    )

    ecg_step_samples = int(
        FEATURE_STEP_S * ecg_fs
    )

    resp_step_samples = int(
        FEATURE_STEP_S * resp_fs
    )

    temporal_features = []

    for step in range(N_SEQUENCE_STEPS):

        ecg_start = step * ecg_step_samples
        ecg_end = ecg_start + ecg_feature_samples

        resp_start = step * resp_step_samples
        resp_end = resp_start + resp_feature_samples

        ecg_segment = ecg[
            ecg_start:ecg_end
        ]

        resp_segment = resp[
            resp_start:resp_end
        ]

        # Exact 5-second check
        if (
            len(ecg_segment) != ecg_feature_samples
            or
            len(resp_segment) != resp_feature_samples
        ):
            raise ValueError(
                f"Window {window_index}, step {step}: "
                "incorrect segment length."
            )

        feature_vector = extract_temporal_features_with_missing(
            ecg_segment,
            resp_segment,
            ecg_fs,
            resp_fs
        )

        temporal_features.append(feature_vector)

    sequence_array = np.asarray(
        temporal_features,
        dtype=float
    )

    sequence_data.append(sequence_array)

    sequence_labels.append(
        int(record["risk_label"])
    )

    sequence_rows.append({
        "sequence_index": window_index,
        "source_window_index": window_index,
        "infant": int(record["infant"]),
        "event_number": record.get(
            "event_number",
            np.nan
        ),
        "event_time_s": record.get(
            "event_time_s",
            np.nan
        ),
        "window_type": record["window_type"],
        "risk_label": int(record["risk_label"])
    })


# -------------------------------------------------------------
# Convert to arrays
# -------------------------------------------------------------
X_sequences = np.asarray(
    sequence_data,
    dtype=float
)

y_sequences = np.asarray(
    sequence_labels,
    dtype=int
)

sequence_metadata = pd.DataFrame(
    sequence_rows
)


# -------------------------------------------------------------
# Missing-value analysis
# -------------------------------------------------------------
missing_mask = np.isnan(X_sequences)

total_missing = int(
    missing_mask.sum()
)

total_values = int(
    X_sequences.size
)

missing_percentage = (
    100.0 * total_missing / total_values
)


# Missing values by feature
missing_by_feature = pd.DataFrame({
    "feature": PHYSIOLOGICAL_FEATURES,
    "missing_values": missing_mask.sum(axis=(0, 1))
})

missing_by_feature["missing_percentage"] = (
    100.0 *
    missing_by_feature["missing_values"] /
    (X_sequences.shape[0] * X_sequences.shape[1])
)


# -------------------------------------------------------------
# Validation
# -------------------------------------------------------------
print("=" * 70)
print("TEMPORAL SEQUENCE DATASET")
print("=" * 70)

print(f"Sequences created : {len(X_sequences)}")
print(f"Expected           : 55")

print()
print(f"X shape            : {X_sequences.shape}")
print(f"Expected shape     : (55, 11, 20)")

print()
print("Label distribution:")
print(
    pd.Series(y_sequences)
    .value_counts()
    .sort_index()
)

print()
print(f"Total feature values : {total_values}")
print(f"Missing values       : {total_missing}")
print(f"Missing percentage   : {missing_percentage:.2f}%")

print()
print("Missing values by feature:")
display(
    missing_by_feature[
        missing_by_feature["missing_values"] > 0
    ]
)

print()
print(
    "Finite non-missing values:",
    bool(
        np.isfinite(
            X_sequences[
                ~np.isnan(X_sequences)
            ]
        ).all()
    )
)

print()
print("=" * 70)

if (
    X_sequences.shape == (55, 11, 20)
    and len(y_sequences) == 55
):
    print(
        "PASS — All 55 temporal sequences retained."
    )
    print(
        "Missing interval-derived features will be "
        "handled within LOSO training folds."
    )
else:
    print(
        "CHECK REQUIRED — Unexpected sequence dimensions."
    )

print("=" * 70)

TEMPORAL SEQUENCE DATASET
Sequences created : 55
Expected           : 55

X shape            : (55, 11, 20)
Expected shape     : (55, 11, 20)

Label distribution:
0    50
1     5
Name: count, dtype: int64

Total feature values : 12100
Missing values       : 60
Missing percentage   : 0.50%

Missing values by feature:


,feature,missing_values,missing_percentage
16,mean_resp_interval,15,2.479339
17,std_resp_interval,15,2.479339
18,mean_resp_rate,15,2.479339
19,std_resp_rate,15,2.479339



Finite non-missing values: True

PASS — All 55 temporal sequences retained.
Missing interval-derived features will be handled within LOSO training folds.


In [56]:
# Cell 21 — Save and verify temporal sequence checkpoint

TEMPORAL_X_PATH = REPORTS_DIR / "pics_final_temporal_X_sequences.npy"
TEMPORAL_Y_PATH = REPORTS_DIR / "pics_final_temporal_y_sequences.npy"
TEMPORAL_META_PATH = REPORTS_DIR / "pics_final_temporal_sequence_metadata.csv"

# ---------------------------------------------------------
# 1. Basic existence checks
# ---------------------------------------------------------
assert "X_sequences" in globals(), "X_sequences is not available."
assert "y_sequences" in globals(), "y_sequences is not available."
assert "final_feature_dataset" in globals(), "final_feature_dataset is not available."

# ---------------------------------------------------------
# 2. Verify expected sequence structure
# ---------------------------------------------------------
assert X_sequences.shape == (55, 11, 20), \
    f"Unexpected X_sequences shape: {X_sequences.shape}"

assert y_sequences.shape == (55,), \
    f"Unexpected y_sequences shape: {y_sequences.shape}"

assert len(final_feature_dataset) == 55, \
    f"Unexpected metadata rows: {len(final_feature_dataset)}"

# ---------------------------------------------------------
# 3. Verify labels
# ---------------------------------------------------------
label_counts = pd.Series(y_sequences).value_counts().sort_index().to_dict()

assert label_counts.get(0, 0) == 50, \
    f"Expected 50 controls, found {label_counts.get(0, 0)}"

assert label_counts.get(1, 0) == 5, \
    f"Expected 5 precursor windows, found {label_counts.get(1, 0)}"

# ---------------------------------------------------------
# 4. Verify finite/non-finite values
# ---------------------------------------------------------
total_values = X_sequences.size
missing_values = int(np.isnan(X_sequences).sum())
infinite_values = int(np.isinf(X_sequences).sum())

# Non-missing values must be finite
finite_non_missing = np.isfinite(X_sequences[~np.isnan(X_sequences)]).all()

assert infinite_values == 0, \
    f"Found {infinite_values} infinite values"

assert finite_non_missing, \
    "Non-missing temporal values are not all finite"

# ---------------------------------------------------------
# 5. Save temporal arrays
# ---------------------------------------------------------
np.save(TEMPORAL_X_PATH, X_sequences)
np.save(TEMPORAL_Y_PATH, y_sequences)

# ---------------------------------------------------------
# 6. Save metadata in the same sequence order
# ---------------------------------------------------------
sequence_metadata = final_feature_dataset.copy()

sequence_metadata.insert(
    0,
    "sequence_index",
    np.arange(len(sequence_metadata))
)

sequence_metadata.to_csv(
    TEMPORAL_META_PATH,
    index=False
)

# ---------------------------------------------------------
# 7. Reload and verify saved artifacts
# ---------------------------------------------------------
X_check = np.load(TEMPORAL_X_PATH)
y_check = np.load(TEMPORAL_Y_PATH)
meta_check = pd.read_csv(TEMPORAL_META_PATH)

assert X_check.shape == (55, 11, 20)
assert y_check.shape == (55,)
assert len(meta_check) == 55

assert np.array_equal(
    np.nan_to_num(X_check, nan=999999.0),
    np.nan_to_num(X_sequences, nan=999999.0)
)

assert np.array_equal(y_check, y_sequences)

# ---------------------------------------------------------
# 8. Report checkpoint
# ---------------------------------------------------------
print("=" * 60)
print("TEMPORAL SEQUENCE CHECKPOINT")
print("=" * 60)

print(f"X_sequences shape      : {X_sequences.shape}")
print(f"y_sequences shape      : {y_sequences.shape}")
print(f"Total feature values   : {total_values}")
print(f"Missing values         : {missing_values}")
print(f"Missing percentage     : {100 * missing_values / total_values:.2f}%")
print(f"Infinite values        : {infinite_values}")
print(f"Control sequences      : {label_counts.get(0, 0)}")
print(f"Precursor sequences    : {label_counts.get(1, 0)}")
print(f"Metadata rows          : {len(meta_check)}")

print("\nSaved files:")
print(f"  {TEMPORAL_X_PATH}")
print(f"  {TEMPORAL_Y_PATH}")
print(f"  {TEMPORAL_META_PATH}")

print("\nPASS — Temporal sequence checkpoint saved and verified.")

TEMPORAL SEQUENCE CHECKPOINT
X_sequences shape      : (55, 11, 20)
y_sequences shape      : (55,)
Total feature values   : 12100
Missing values         : 60
Missing percentage     : 0.50%
Infinite values        : 0
Control sequences      : 50
Precursor sequences    : 5
Metadata rows          : 55

Saved files:
  c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_final_temporal_X_sequences.npy
  c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_final_temporal_y_sequences.npy
  c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_final_temporal_sequence_metadata.csv

PASS — Temporal sequence checkpoint saved and verified.


In [57]:
# Cell 22 — Prepare LOSO temporal folds

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# ---------------------------------------------------------
# 1. Load saved temporal checkpoint
# ---------------------------------------------------------
X_sequences = np.load(TEMPORAL_X_PATH)
y_sequences = np.load(TEMPORAL_Y_PATH)
sequence_metadata = pd.read_csv(TEMPORAL_META_PATH)

# ---------------------------------------------------------
# 2. Verify checkpoint
# ---------------------------------------------------------
assert X_sequences.shape == (55, 11, 20)
assert y_sequences.shape == (55,)
assert len(sequence_metadata) == 55

# ---------------------------------------------------------
# 3. Identify infant for each sequence
# ---------------------------------------------------------
infant_ids = sequence_metadata["infant"].astype(int).to_numpy()

unique_infants = np.sort(np.unique(infant_ids))

assert len(unique_infants) == 10
assert np.array_equal(unique_infants, np.arange(1, 11))

# ---------------------------------------------------------
# 4. Verify sequence-level alignment
# ---------------------------------------------------------
assert np.array_equal(
    sequence_metadata["sequence_index"].to_numpy(),
    np.arange(55)
)

assert np.array_equal(
    sequence_metadata["risk_label"].to_numpy(),
    y_sequences
)

# ---------------------------------------------------------
# 5. Create LOSO folds
# ---------------------------------------------------------
loso_folds = []

for test_infant in unique_infants:

    train_mask = infant_ids != test_infant
    test_mask = infant_ids == test_infant

    X_train = X_sequences[train_mask]
    X_test = X_sequences[test_mask]

    y_train = y_sequences[train_mask]
    y_test = y_sequences[test_mask]

    # Flatten temporal dimension ONLY for fold preparation.
    # 11 time steps × 20 features = 220 values per sequence.
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_test_flat = X_test.reshape(X_test.shape[0], -1)

    # -----------------------------------------------------
    # Imputation fitted ONLY on training infant set
    # -----------------------------------------------------
    imputer = SimpleImputer(strategy="median")

    X_train_imp = imputer.fit_transform(X_train_flat)
    X_test_imp = imputer.transform(X_test_flat)

    # -----------------------------------------------------
    # Scaling fitted ONLY on training infant set
    # -----------------------------------------------------
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)

    loso_folds.append({
        "test_infant": int(test_infant),
        "train_infants": np.sort(np.unique(infant_ids[train_mask])),
        "X_train": X_train_scaled,
        "X_test": X_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "imputer": imputer,
        "scaler": scaler
    })

# ---------------------------------------------------------
# 6. Verify LOSO integrity
# ---------------------------------------------------------
assert len(loso_folds) == 10

for fold in loso_folds:

    test_infant = fold["test_infant"]
    train_infants = fold["train_infants"]

    assert test_infant not in train_infants
    assert len(train_infants) == 9

    assert fold["X_train"].shape[0] == len(fold["y_train"])
    assert fold["X_test"].shape[0] == len(fold["y_test"])

    assert np.isfinite(fold["X_train"]).all()
    assert np.isfinite(fold["X_test"]).all()

# ---------------------------------------------------------
# 7. Print fold summary
# ---------------------------------------------------------
print("=" * 60)
print("TEMPORAL LOSO FOLD PREPARATION")
print("=" * 60)

for fold in loso_folds:

    print(
        f"Test Infant {fold['test_infant']:2d} | "
        f"Train: {len(fold['y_train']):2d} | "
        f"Test: {len(fold['y_test']):2d} | "
        f"Train positives: {int(fold['y_train'].sum()):2d} | "
        f"Test positives: {int(fold['y_test'].sum()):2d}"
    )

print("\nExpected:")
print("  10 LOSO folds")
print("  9 training infants per fold")
print("  1 test infant per fold")
print("  Imputation: training fold only")
print("  Scaling: training fold only")
print("  Temporal input: 11 × 20")

print("\nPASS — LOSO temporal folds prepared without subject leakage.")

TEMPORAL LOSO FOLD PREPARATION
Test Infant  1 | Train: 49 | Test:  6 | Train positives:  4 | Test positives:  1
Test Infant  2 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0
Test Infant  3 | Train: 48 | Test:  7 | Train positives:  3 | Test positives:  2
Test Infant  4 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0
Test Infant  5 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0
Test Infant  6 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0
Test Infant  7 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0
Test Infant  8 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0
Test Infant  9 | Train: 48 | Test:  7 | Train positives:  3 | Test positives:  2
Test Infant 10 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0

Expected:
  10 LOSO folds
  9 training infants per fold
  1 test infant per fold
  Imputation: training fold only
  Scaling: training fold only
  Temporal inp

In [58]:
# Cell 23 — Temporal LOSO Logistic Regression baseline

from sklearn.linear_model import LogisticRegression

temporal_loso_results = []

for fold in loso_folds:

    test_infant = fold["test_infant"]

    X_train = fold["X_train"]
    X_test = fold["X_test"]
    y_train = fold["y_train"]
    y_test = fold["y_test"]

    # -----------------------------------------------------
    # Train baseline model
    # -----------------------------------------------------
    model = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_SEED
    )

    model.fit(X_train, y_train)

    # -----------------------------------------------------
    # Predictions
    # -----------------------------------------------------
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.50).astype(int)

    # -----------------------------------------------------
    # Store fold predictions
    # -----------------------------------------------------
    for i in range(len(y_test)):

        temporal_loso_results.append({
            "test_infant": test_infant,
            "y_true": int(y_test[i]),
            "y_pred": int(y_pred[i]),
            "y_prob": float(y_prob[i])
        })

# ---------------------------------------------------------
# Convert to DataFrame
# ---------------------------------------------------------
temporal_loso_predictions = pd.DataFrame(
    temporal_loso_results
)

# ---------------------------------------------------------
# Basic verification
# ---------------------------------------------------------
assert len(temporal_loso_predictions) == 55

assert set(
    temporal_loso_predictions["test_infant"].unique()
) == set(range(1, 11))

assert temporal_loso_predictions["y_true"].isin([0, 1]).all()
assert temporal_loso_predictions["y_pred"].isin([0, 1]).all()

assert np.isfinite(
    temporal_loso_predictions["y_prob"]
).all()

print("=" * 60)
print("TEMPORAL LOSO LOGISTIC REGRESSION")
print("=" * 60)

print(f"Total OOF predictions : {len(temporal_loso_predictions)}")
print(
    f"True controls         : "
    f"{(temporal_loso_predictions['y_true'] == 0).sum()}"
)
print(
    f"True precursor        : "
    f"{(temporal_loso_predictions['y_true'] == 1).sum()}"
)
print(
    f"Predicted controls    : "
    f"{(temporal_loso_predictions['y_pred'] == 0).sum()}"
)
print(
    f"Predicted precursor   : "
    f"{(temporal_loso_predictions['y_pred'] == 1).sum()}"
)

print("\nPASS — All 55 LOSO out-of-fold predictions generated.")

TEMPORAL LOSO LOGISTIC REGRESSION
Total OOF predictions : 55
True controls         : 50
True precursor        : 5
Predicted controls    : 50
Predicted precursor   : 5

PASS — All 55 LOSO out-of-fold predictions generated.


In [59]:
# Cell 24 — Pooled temporal LOSO metrics

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# ---------------------------------------------------------
# Extract pooled out-of-fold results
# ---------------------------------------------------------
y_true = temporal_loso_predictions["y_true"].to_numpy()
y_pred = temporal_loso_predictions["y_pred"].to_numpy()
y_prob = temporal_loso_predictions["y_prob"].to_numpy()

# ---------------------------------------------------------
# Confusion matrix
# ---------------------------------------------------------
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

TN, FP, FN, TP = cm.ravel()

# ---------------------------------------------------------
# Classification metrics
# ---------------------------------------------------------
accuracy = accuracy_score(y_true, y_pred)

sensitivity = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

specificity = TN / (TN + FP) if (TN + FP) > 0 else np.nan

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

# ---------------------------------------------------------
# Probability-based metrics
# ---------------------------------------------------------
auroc = roc_auc_score(y_true, y_prob)

auprc = average_precision_score(y_true, y_prob)

# ---------------------------------------------------------
# Print results
# ---------------------------------------------------------
print("=" * 60)
print("TEMPORAL LOSO LOGISTIC REGRESSION — POOLED RESULTS")
print("=" * 60)

print("\nConfusion Matrix:")
print(cm)

print("\nCounts:")
print(f"TN = {TN}")
print(f"FP = {FP}")
print(f"FN = {FN}")
print(f"TP = {TP}")

print("\nMetrics:")
print(f"Accuracy     : {accuracy:.4f}")
print(f"Sensitivity  : {sensitivity:.4f}")
print(f"Specificity  : {specificity:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"AUROC        : {auroc:.4f}")
print(f"AUPRC        : {auprc:.4f}")

print("\nEvaluation:")
print("  Method      : Leave-One-Subject-Out")
print("  Predictions : Out-of-fold")
print("  Test leakage: None")
print("  Status      : DEVELOPMENT / VALIDATION RESULT")

TEMPORAL LOSO LOGISTIC REGRESSION — POOLED RESULTS

Confusion Matrix:
[[47  3]
 [ 3  2]]

Counts:
TN = 47
FP = 3
FN = 3
TP = 2

Metrics:
Accuracy     : 0.8909
Sensitivity  : 0.4000
Specificity  : 0.9400
Precision    : 0.4000
F1 Score     : 0.4000
AUROC        : 0.8240
AUPRC        : 0.3049

Evaluation:
  Method      : Leave-One-Subject-Out
  Predictions : Out-of-fold
  Test leakage: None
  Status      : DEVELOPMENT / VALIDATION RESULT


In [60]:
# Cell 25 — Save temporal LOSO baseline checkpoint

TEMPORAL_PRED_PATH = REPORTS_DIR / "pics_temporal_loso_logistic_predictions.csv"
TEMPORAL_METRICS_PATH = REPORTS_DIR / "pics_temporal_loso_logistic_metrics.csv"

# ---------------------------------------------------------
# 1. Save out-of-fold predictions
# ---------------------------------------------------------
temporal_loso_predictions.to_csv(
    TEMPORAL_PRED_PATH,
    index=False
)

# ---------------------------------------------------------
# 2. Create metrics table
# ---------------------------------------------------------
temporal_loso_metrics = pd.DataFrame([{
    "model": "Temporal Logistic Regression",
    "evaluation": "LOSO pooled out-of-fold",
    "TN": int(TN),
    "FP": int(FP),
    "FN": int(FN),
    "TP": int(TP),
    "accuracy": float(accuracy),
    "sensitivity": float(sensitivity),
    "specificity": float(specificity),
    "precision": float(precision),
    "f1": float(f1),
    "auroc": float(auroc),
    "auprc": float(auprc),
    "positive_windows": int(y_true.sum()),
    "total_windows": int(len(y_true)),
    "status": "DEVELOPMENT / VALIDATION RESULT"
}])

temporal_loso_metrics.to_csv(
    TEMPORAL_METRICS_PATH,
    index=False
)

# ---------------------------------------------------------
# 3. Reload and verify
# ---------------------------------------------------------
pred_check = pd.read_csv(TEMPORAL_PRED_PATH)
metrics_check = pd.read_csv(TEMPORAL_METRICS_PATH)

assert len(pred_check) == 55
assert len(metrics_check) == 1

assert int(metrics_check.loc[0, "TP"]) == 2
assert int(metrics_check.loc[0, "FN"]) == 3
assert int(metrics_check.loc[0, "TN"]) == 47
assert int(metrics_check.loc[0, "FP"]) == 3

print("=" * 60)
print("TEMPORAL BASELINE CHECKPOINT")
print("=" * 60)

print(f"Predictions saved : {TEMPORAL_PRED_PATH}")
print(f"Metrics saved     : {TEMPORAL_METRICS_PATH}")

print("\nVerified:")
print("  55 LOSO predictions")
print("  TN = 47")
print("  FP = 3")
print("  FN = 3")
print("  TP = 2")
print("  AUROC = 0.8240")
print("  AUPRC = 0.3049")

print("\nPASS — Temporal Logistic Regression checkpoint saved.")

TEMPORAL BASELINE CHECKPOINT
Predictions saved : c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_temporal_loso_logistic_predictions.csv
Metrics saved     : c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_temporal_loso_logistic_metrics.csv

Verified:
  55 LOSO predictions
  TN = 47
  FP = 3
  FN = 3
  TP = 2
  AUROC = 0.8240
  AUPRC = 0.3049

PASS — Temporal Logistic Regression checkpoint saved.


In [61]:
# Cell 26 — Per-infant temporal LOSO analysis

per_infant_results = []

for infant_id in sorted(
    temporal_loso_predictions["test_infant"].unique()
):

    fold_data = temporal_loso_predictions[
        temporal_loso_predictions["test_infant"] == infant_id
    ].copy()

    y_true_i = fold_data["y_true"].to_numpy()
    y_pred_i = fold_data["y_pred"].to_numpy()
    y_prob_i = fold_data["y_prob"].to_numpy()

    cm_i = confusion_matrix(
        y_true_i,
        y_pred_i,
        labels=[0, 1]
    )

    TN_i, FP_i, FN_i, TP_i = cm_i.ravel()

    total_i = len(fold_data)
    positives_i = int(y_true_i.sum())

    sensitivity_i = (
        TP_i / (TP_i + FN_i)
        if (TP_i + FN_i) > 0
        else np.nan
    )

    specificity_i = (
        TN_i / (TN_i + FP_i)
        if (TN_i + FP_i) > 0
        else np.nan
    )

    per_infant_results.append({
        "test_infant": int(infant_id),
        "total_windows": total_i,
        "positive_windows": positives_i,
        "TN": int(TN_i),
        "FP": int(FP_i),
        "FN": int(FN_i),
        "TP": int(TP_i),
        "sensitivity": sensitivity_i,
        "specificity": specificity_i
    })

per_infant_results = pd.DataFrame(per_infant_results)

# ---------------------------------------------------------
# Display
# ---------------------------------------------------------
print("=" * 60)
print("PER-INFANT TEMPORAL LOSO RESULTS")
print("=" * 60)

print(
    per_infant_results.to_string(index=False)
)

# ---------------------------------------------------------
# Positive-infant summary
# ---------------------------------------------------------
positive_infants = per_infant_results[
    per_infant_results["positive_windows"] > 0
]

print("\n" + "=" * 60)
print("INFANTS CONTAINING POSITIVE WINDOWS")
print("=" * 60)

print(
    positive_infants.to_string(index=False)
)

# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------
assert len(per_infant_results) == 10

assert (
    per_infant_results["positive_windows"].sum()
    == 5
)

assert (
    per_infant_results["TP"].sum()
    == 2
)

assert (
    per_infant_results["FN"].sum()
    == 3
)

print("\nPASS — Per-infant LOSO analysis verified.")

PER-INFANT TEMPORAL LOSO RESULTS
 test_infant  total_windows  positive_windows  TN  FP  FN  TP  sensitivity  specificity
           1              6                 1   4   1   1   0          0.0          0.8
           2              5                 0   5   0   0   0          NaN          1.0
           3              7                 2   5   0   1   1          0.5          1.0
           4              5                 0   5   0   0   0          NaN          1.0
           5              5                 0   5   0   0   0          NaN          1.0
           6              5                 0   5   0   0   0          NaN          1.0
           7              5                 0   5   0   0   0          NaN          1.0
           8              5                 0   4   1   0   0          NaN          0.8
           9              7                 2   5   0   1   1          0.5          1.0
          10              5                 0   4   1   0   0          NaN          0.8

In [62]:
# Cell 27 — Inspect individual positive-window predictions

positive_predictions = temporal_loso_predictions[
    temporal_loso_predictions["y_true"] == 1
].copy()

# ---------------------------------------------------------
# Add metadata for the positive windows
# ---------------------------------------------------------
metadata_columns = [
    "sequence_index",
    "infant",
    "window_type",
    "risk_label"
]

positive_metadata = sequence_metadata[
    metadata_columns
].copy()

positive_predictions = positive_predictions.merge(
    positive_metadata,
    left_index=True,
    right_index=True,
    how="left"
)

# The merge above may not preserve the original prediction
# index meaningfully, so rebuild the mapping explicitly.
positive_predictions = temporal_loso_predictions[
    temporal_loso_predictions["y_true"] == 1
].copy()

positive_predictions["sequence_index"] = (
    positive_predictions.groupby("test_infant").cumcount()
)

# ---------------------------------------------------------
# Match each positive prediction to the original sequence
# using infant + positive-window order.
# ---------------------------------------------------------
positive_rows = []

for infant_id in sorted(
    positive_predictions["test_infant"].unique()
):

    infant_predictions = positive_predictions[
        positive_predictions["test_infant"] == infant_id
    ].copy()

    infant_metadata = sequence_metadata[
        (sequence_metadata["infant"] == infant_id) &
        (sequence_metadata["risk_label"] == 1)
    ].copy()

    infant_metadata = infant_metadata.sort_values(
        "sequence_index"
    ).reset_index(drop=True)

    infant_predictions = infant_predictions.reset_index(drop=True)

    assert len(infant_predictions) == len(infant_metadata)

    for i in range(len(infant_predictions)):

        row = infant_predictions.iloc[i]
        meta = infant_metadata.iloc[i]

        positive_rows.append({
            "test_infant": int(row["test_infant"]),
            "sequence_index": int(meta["sequence_index"]),
            "window_type": meta["window_type"],
            "risk_label": int(meta["risk_label"]),
            "y_true": int(row["y_true"]),
            "y_pred": int(row["y_pred"]),
            "y_prob": float(row["y_prob"]),
            "result": (
                "TP" if row["y_pred"] == 1
                else "FN"
            )
        })

positive_window_analysis = pd.DataFrame(
    positive_rows
)

# ---------------------------------------------------------
# Display
# ---------------------------------------------------------
print("=" * 60)
print("INDIVIDUAL POSITIVE-WINDOW PREDICTIONS")
print("=" * 60)

print(
    positive_window_analysis.to_string(index=False)
)

# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------
assert len(positive_window_analysis) == 5
assert (
    (positive_window_analysis["result"] == "TP").sum()
    == 2
)
assert (
    (positive_window_analysis["result"] == "FN").sum()
    == 3
)

print("\nPASS — All 5 positive windows mapped and verified.")

INDIVIDUAL POSITIVE-WINDOW PREDICTIONS
 test_infant  sequence_index window_type  risk_label  y_true  y_pred   y_prob result
           1               0   precursor           1       1       0 0.043802     FN
           3               1   precursor           1       1       1 0.532863     TP
           3               2   precursor           1       1       0 0.051205     FN
           9               3   precursor           1       1       1 0.576292     TP
           9               4   precursor           1       1       0 0.088368     FN

PASS — All 5 positive windows mapped and verified.


In [63]:
# Cell 28 — Save positive-window temporal analysis

POSITIVE_ANALYSIS_PATH = (
    REPORTS_DIR / "pics_temporal_positive_window_analysis.csv"
)

positive_window_analysis.to_csv(
    POSITIVE_ANALYSIS_PATH,
    index=False
)

# ---------------------------------------------------------
# Reload and verify
# ---------------------------------------------------------
positive_check = pd.read_csv(
    POSITIVE_ANALYSIS_PATH
)

assert len(positive_check) == 5
assert set(positive_check["result"]) == {"TP", "FN"}

assert (
    (positive_check["result"] == "TP").sum()
    == 2
)

assert (
    (positive_check["result"] == "FN").sum()
    == 3
)

print("=" * 60)
print("POSITIVE-WINDOW ANALYSIS CHECKPOINT")
print("=" * 60)

print(f"Saved : {POSITIVE_ANALYSIS_PATH}")
print(f"Positive windows : {len(positive_check)}")
print(f"True positives   : {(positive_check['result'] == 'TP').sum()}")
print(f"False negatives  : {(positive_check['result'] == 'FN').sum()}")

print("\nPASS — Positive-window analysis saved and verified.")

POSITIVE-WINDOW ANALYSIS CHECKPOINT
Saved : c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_temporal_positive_window_analysis.csv
Positive windows : 5
True positives   : 2
False negatives  : 3

PASS — Positive-window analysis saved and verified.


In [65]:
# Cell 29 — Verify TensorFlow environment for GRU

import tensorflow as tf

print("=" * 60)
print("TENSORFLOW / GRU ENVIRONMENT CHECK")
print("=" * 60)

print(f"TensorFlow version : {tf.__version__}")

# ---------------------------------------------------------
# Verify GRU layer is available
# ---------------------------------------------------------
gru_test = tf.keras.layers.GRU(
    units=16,
    return_sequences=False
)

print(f"GRU layer          : {type(gru_test).__name__}")

# ---------------------------------------------------------
# Check available devices
# ---------------------------------------------------------
devices = tf.config.list_physical_devices()

print("\nAvailable devices:")
for device in devices:
    print(f"  {device.device_type}: {device.name}")

# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------
tf.random.set_seed(RANDOM_SEED)

print("\nRandom seed        :", RANDOM_SEED)

print("\nPASS — TensorFlow GRU environment verified.")

TENSORFLOW / GRU ENVIRONMENT CHECK
TensorFlow version : 2.21.0
GRU layer          : GRU

Available devices:
  CPU: /physical_device:CPU:0

Random seed        : 42

PASS — TensorFlow GRU environment verified.


In [66]:
# Cell 30 — Prepare LOSO-safe temporal data for GRU

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------
# 1. Load the saved temporal checkpoint
# ---------------------------------------------------------
X_sequences = np.load(TEMPORAL_X_PATH)
y_sequences = np.load(TEMPORAL_Y_PATH)
sequence_metadata = pd.read_csv(TEMPORAL_META_PATH)

# ---------------------------------------------------------
# 2. Basic verification
# ---------------------------------------------------------
assert X_sequences.shape == (55, 11, 20)
assert y_sequences.shape == (55,)
assert len(sequence_metadata) == 55

infant_ids = sequence_metadata["infant"].astype(int).to_numpy()

unique_infants = np.sort(np.unique(infant_ids))

assert len(unique_infants) == 10

# ---------------------------------------------------------
# 3. Prepare GRU LOSO folds
# ---------------------------------------------------------
gru_loso_folds = []

for test_infant in unique_infants:

    train_mask = infant_ids != test_infant
    test_mask = infant_ids == test_infant

    X_train_raw = X_sequences[train_mask]
    X_test_raw = X_sequences[test_mask]

    y_train = y_sequences[train_mask]
    y_test = y_sequences[test_mask]

    # -----------------------------------------------------
    # Reshape ONLY for preprocessing.
    #
    # This combines all training time steps so that each
    # physiological feature has one common imputation and
    # scaling distribution across the 11 time steps.
    # -----------------------------------------------------
    X_train_2d = X_train_raw.reshape(-1, 20)
    X_test_2d = X_test_raw.reshape(-1, 20)

    # -----------------------------------------------------
    # Imputation fitted ONLY on training subjects
    # -----------------------------------------------------
    imputer = SimpleImputer(strategy="median")

    X_train_imp = imputer.fit_transform(X_train_2d)
    X_test_imp = imputer.transform(X_test_2d)

    # -----------------------------------------------------
    # Scaling fitted ONLY on training subjects
    # -----------------------------------------------------
    scaler = StandardScaler()

    X_train_scaled_2d = scaler.fit_transform(X_train_imp)
    X_test_scaled_2d = scaler.transform(X_test_imp)

    # -----------------------------------------------------
    # Restore temporal structure
    # -----------------------------------------------------
    X_train_scaled = X_train_scaled_2d.reshape(
        X_train_raw.shape
    )

    X_test_scaled = X_test_scaled_2d.reshape(
        X_test_raw.shape
    )

    # -----------------------------------------------------
    # Verify no missing values remain
    # -----------------------------------------------------
    assert np.isfinite(X_train_scaled).all()
    assert np.isfinite(X_test_scaled).all()

    # -----------------------------------------------------
    # Store fold
    # -----------------------------------------------------
    gru_loso_folds.append({
        "test_infant": int(test_infant),
        "train_infants": np.sort(
            np.unique(infant_ids[train_mask])
        ),
        "X_train": X_train_scaled,
        "X_test": X_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "imputer": imputer,
        "scaler": scaler
    })

# ---------------------------------------------------------
# 4. Verify all folds
# ---------------------------------------------------------
assert len(gru_loso_folds) == 10

for fold in gru_loso_folds:

    assert fold["test_infant"] not in fold["train_infants"]
    assert len(fold["train_infants"]) == 9

    assert fold["X_train"].shape[1:] == (11, 20)
    assert fold["X_test"].shape[1:] == (11, 20)

    assert fold["X_train"].shape[0] == len(
        fold["y_train"]
    )

    assert fold["X_test"].shape[0] == len(
        fold["y_test"]
    )

    assert np.isfinite(fold["X_train"]).all()
    assert np.isfinite(fold["X_test"]).all()

# ---------------------------------------------------------
# 5. Print summary
# ---------------------------------------------------------
print("=" * 60)
print("GRU LOSO DATA PREPARATION")
print("=" * 60)

for fold in gru_loso_folds:

    print(
        f"Test Infant {fold['test_infant']:2d} | "
        f"Train: {len(fold['y_train']):2d} | "
        f"Test: {len(fold['y_test']):2d} | "
        f"Train positives: {int(fold['y_train'].sum()):2d} | "
        f"Test positives: {int(fold['y_test'].sum()):2d}"
    )

print("\nGRU input shape:")
print("  Time steps : 11")
print("  Features   : 20")
print("  Shape      : (samples, 11, 20)")

print("\nPreprocessing:")
print("  Imputation : training subjects only")
print("  Scaling    : training subjects only")
print("  Test data  : transformed using training parameters")

print("\nPASS — GRU LOSO data prepared without subject leakage.")

GRU LOSO DATA PREPARATION
Test Infant  1 | Train: 49 | Test:  6 | Train positives:  4 | Test positives:  1
Test Infant  2 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0
Test Infant  3 | Train: 48 | Test:  7 | Train positives:  3 | Test positives:  2
Test Infant  4 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0
Test Infant  5 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0
Test Infant  6 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0
Test Infant  7 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0
Test Infant  8 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0
Test Infant  9 | Train: 48 | Test:  7 | Train positives:  3 | Test positives:  2
Test Infant 10 | Train: 50 | Test:  5 | Train positives:  5 | Test positives:  0

GRU input shape:
  Time steps : 11
  Features   : 20
  Shape      : (samples, 11, 20)

Preprocessing:
  Imputation : training subjects only
  Scaling    : training

In [67]:
# Cell 31 — Finalize GRU development architecture

import tensorflow as tf

# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

GRU_LEARNING_RATE = 0.0005

# ---------------------------------------------------------
# Model definition
# ---------------------------------------------------------
gru_model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(11, 20),
        name="temporal_input"
    ),

    tf.keras.layers.GRU(
        16,
        return_sequences=False,
        name="gru_16"
    ),

    tf.keras.layers.Dropout(
        0.20,
        name="dropout"
    ),

    tf.keras.layers.Dense(
        8,
        activation="relu",
        name="dense_8"
    ),

    tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        name="risk_output"
    )
])

# ---------------------------------------------------------
# Compile
# ---------------------------------------------------------
gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=GRU_LEARNING_RATE
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.AUC(
            name="auroc"
        )
    ]
)

# ---------------------------------------------------------
# Display configuration
# ---------------------------------------------------------
print("=" * 60)
print("GRU DEVELOPMENT MODEL CONFIGURATION")
print("=" * 60)

print(f"Input shape      : {gru_model.input_shape}")
print(f"Output shape     : {gru_model.output_shape}")
print(f"GRU units        : 16")
print(f"Dropout          : 0.20")
print(f"Dense units      : 8")
print(f"Learning rate    : {GRU_LEARNING_RATE}")
print(f"Optimizer        : Adam")
print(f"Loss             : Binary cross-entropy")

print("\nModel parameters:")
print(f"Trainable params : {gru_model.count_params()}")

print("\nPASS — GRU development architecture finalized.")

GRU DEVELOPMENT MODEL CONFIGURATION
Input shape      : (None, 11, 20)
Output shape     : (None, 1)
GRU units        : 16
Dropout          : 0.20
Dense units      : 8
Learning rate    : 0.0005
Optimizer        : Adam
Loss             : Binary cross-entropy

Model parameters:
Trainable params : 1969

PASS — GRU development architecture finalized.


In [68]:
# Cell 32 — Single-fold GRU training
# Test subject: Infant 1

import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

# ---------------------------------------------------------
# 1. Select Infant 1 LOSO fold
# ---------------------------------------------------------
fold1 = next(
    fold for fold in gru_loso_folds
    if fold["test_infant"] == 1
)

X_train_fold = fold1["X_train"]
X_test_fold = fold1["X_test"]

y_train_fold = fold1["y_train"]
y_test_fold = fold1["y_test"]

# ---------------------------------------------------------
# 2. Verify fold structure
# ---------------------------------------------------------
assert X_train_fold.shape == (49, 11, 20)
assert X_test_fold.shape == (6, 11, 20)

assert y_train_fold.shape == (49,)
assert y_test_fold.shape == (6,)

assert int(y_train_fold.sum()) == 4
assert int(y_test_fold.sum()) == 1

# ---------------------------------------------------------
# 3. Calculate training-only class weights
# ---------------------------------------------------------
classes = np.unique(y_train_fold)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_fold
)

class_weight = {
    int(cls): float(weight)
    for cls, weight in zip(
        classes,
        class_weights_array
    )
}

# ---------------------------------------------------------
# 4. Rebuild fresh GRU model
# ---------------------------------------------------------
tf.keras.backend.clear_session()

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

gru_fold1 = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(11, 20),
        name="temporal_input"
    ),

    tf.keras.layers.GRU(
        16,
        return_sequences=False,
        name="gru_16"
    ),

    tf.keras.layers.Dropout(
        0.20,
        name="dropout"
    ),

    tf.keras.layers.Dense(
        8,
        activation="relu",
        name="dense_8"
    ),

    tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        name="risk_output"
    )
])

gru_fold1.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=GRU_LEARNING_RATE
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.AUC(
            name="auroc"
        )
    ]
)

# ---------------------------------------------------------
# 5. Early stopping
# ---------------------------------------------------------
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
    mode="min",
    verbose=1
)

# ---------------------------------------------------------
# 6. Train
#
# IMPORTANT:
# The held-out Infant 1 is NOT used here.
#
# We create a validation split only from the training
# subjects. validation_split=0.20 therefore operates on
# the 49 training sequences, not the held-out infant.
# ---------------------------------------------------------
history_fold1 = gru_fold1.fit(
    X_train_fold,
    y_train_fold,
    validation_split=0.20,
    epochs=100,
    batch_size=8,
    class_weight=class_weight,
    callbacks=[early_stopping],
    verbose=1,
    shuffle=True
)

# ---------------------------------------------------------
# 7. Evaluate ONLY after training
# ---------------------------------------------------------
test_loss, test_accuracy, test_auroc = (
    gru_fold1.evaluate(
        X_test_fold,
        y_test_fold,
        verbose=0
    )
)

# ---------------------------------------------------------
# 8. Generate test probabilities
# ---------------------------------------------------------
test_prob = (
    gru_fold1.predict(
        X_test_fold,
        verbose=0
    ).ravel()
)

test_pred = (
    test_prob >= 0.50
).astype(int)

# ---------------------------------------------------------
# 9. Display results
# ---------------------------------------------------------
print("=" * 60)
print("SINGLE-FOLD GRU TRAINING — INFANT 1 HELD OUT")
print("=" * 60)

print("\nTraining:")
print(f"  Training sequences : {len(y_train_fold)}")
print(f"  Training positives : {int(y_train_fold.sum())}")
print(f"  Training negatives : {int((y_train_fold == 0).sum())}")

print("\nClass weights:")
print(f"  Class 0 : {class_weight[0]:.4f}")
print(f"  Class 1 : {class_weight[1]:.4f}")

print("\nTraining:")
print(f"  Epochs actually run : {len(history_fold1.history['loss'])}")

print("\nHeld-out Infant 1:")
print(f"  Test sequences      : {len(y_test_fold)}")
print(f"  Test positives      : {int(y_test_fold.sum())}")
print(f"  Test loss           : {test_loss:.4f}")
print(f"  Test accuracy       : {test_accuracy:.4f}")
print(f"  Test AUROC          : {test_auroc:.4f}")

print("\nTest probabilities:")
for i, (true_value, probability, prediction) in enumerate(
    zip(
        y_test_fold,
        test_prob,
        test_pred
    )
):
    print(
        f"  Sample {i}: "
        f"true={int(true_value)} | "
        f"prob={probability:.4f} | "
        f"pred={int(prediction)}"
    )

print("\nPASS — Single-fold GRU training completed.")


Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 140ms/step - accuracy: 0.4615 - auroc: 0.0750 - loss: 0.9481 - val_accuracy: 0.6000 - val_auroc: 0.0000e+00 - val_loss: 0.6624
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.6410 - auroc: 0.1036 - loss: 0.9177 - val_accuracy: 0.6000 - val_auroc: 0.0000e+00 - val_loss: 0.6653
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5641 - auroc: 0.2179 - loss: 0.8996 - val_accuracy: 0.6000 - val_auroc: 0.0000e+00 - val_loss: 0.6677
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6154 - auroc: 0.3071 - loss: 0.8833 - val_accuracy: 0.6000 - val_auroc: 0.0000e+00 - val_loss: 0.6698
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5128 - auroc: 0.3500 - loss: 0.8477 - val_accuracy: 0.6000 - val_auroc: 0.0000e+00 - val_loss: 0.6719
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6667 - auroc: 0.4000 - loss: 0.8064 - val_accuracy: 0.6000 - val_auroc: 0.0000e+00 - val_loss: 0.6740
Ep

In [69]:
# ============================================================
# CELL 32A — Exploratory Inner-LOSO Epoch Analysis
# Purpose:
#   Estimate a fixed GRU epoch count without using the
#   outer test infant.
#
# This is an EXPLORATORY epoch-selection experiment only.
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ------------------------------------------------------------
# Select outer development fold
# Infant 1 remains the outer test infant conceptually.
# It must NOT participate in epoch selection.
# ------------------------------------------------------------
OUTER_TEST_INFANT = 1

outer_dev_mask = np.array(
    [infant != OUTER_TEST_INFANT for infant in sequence_metadata["infant"]]
)

X_outer_dev = X_sequences[outer_dev_mask]
y_outer_dev = y_sequences[outer_dev_mask]
infants_outer_dev = sequence_metadata.loc[
    outer_dev_mask, "infant"
].to_numpy()

print("Outer test infant:", OUTER_TEST_INFANT)
print("Outer development sequences:", len(X_outer_dev))
print("Outer development positives:", int(y_outer_dev.sum()))
print("Outer development negatives:", int((y_outer_dev == 0).sum()))
print(
    "Outer development infants:",
    sorted(np.unique(infants_outer_dev).tolist())
)

# ------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------
assert OUTER_TEST_INFANT not in np.unique(infants_outer_dev), \
    "Outer test infant leaked into development data."

assert len(X_outer_dev) == len(y_outer_dev) == len(infants_outer_dev)

print("\nPASS: Outer test infant is excluded from epoch selection.")

Outer test infant: 1
Outer development sequences: 49
Outer development positives: 4
Outer development negatives: 45
Outer development infants: [2, 3, 4, 5, 6, 7, 8, 9, 10]

PASS: Outer test infant is excluded from epoch selection.


In [70]:
# ============================================================
# CELL 32B — Build Inner-LOSO Folds
# ============================================================

inner_fold_info = []

unique_inner_infants = sorted(np.unique(infants_outer_dev))

for val_infant in unique_inner_infants:

    train_mask = infants_outer_dev != val_infant
    val_mask = infants_outer_dev == val_infant

    y_train_inner = y_outer_dev[train_mask]
    y_val_inner = y_outer_dev[val_mask]

    inner_fold_info.append({
        "validation_infant": int(val_infant),
        "train_samples": int(train_mask.sum()),
        "train_positive": int(y_train_inner.sum()),
        "train_negative": int((y_train_inner == 0).sum()),
        "val_samples": int(val_mask.sum()),
        "val_positive": int(y_val_inner.sum()),
        "val_negative": int((y_val_inner == 0).sum())
    })

inner_fold_info = pd.DataFrame(inner_fold_info)

print("Inner-LOSO fold summary:")
display(inner_fold_info)

print("\nNumber of inner folds:", len(inner_fold_info))

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------
assert len(inner_fold_info) == 9
assert set(inner_fold_info["validation_infant"]) == set(unique_inner_infants)

print("\nPASS: 9 inner-LOSO folds created.")
print("PASS: Outer Infant 1 remains excluded.")

Inner-LOSO fold summary:


,validation_infant,train_samples,train_positive,train_negative,val_samples,val_positive,val_negative
0,2,44,4,40,5,0,5
1,3,42,2,40,7,2,5
2,4,44,4,40,5,0,5
3,5,44,4,40,5,0,5
4,6,44,4,40,5,0,5
5,7,44,4,40,5,0,5
6,8,44,4,40,5,0,5
7,9,42,2,40,7,2,5
8,10,44,4,40,5,0,5



Number of inner folds: 9

PASS: 9 inner-LOSO folds created.
PASS: Outer Infant 1 remains excluded.


In [71]:
# ============================================================
# CELL 32C — Exploratory Epoch Selection
#
# IMPORTANT:
#   - Outer Infant 1 remains completely untouched.
#   - Only validation infants containing positive windows
#     are used for this exploratory epoch-selection analysis.
#   - This is NOT a final performance evaluation.
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ------------------------------------------------------------
# Exploratory inner validation infants
#
# From Cell 32B:
#   Infant 3 -> 2 positive / 5 negative
#   Infant 9 -> 2 positive / 5 negative
# ------------------------------------------------------------
EPOCH_SELECTION_INFANTS = [3, 9]

MAX_EPOCHS = 100
BATCH_SIZE = 8
PATIENCE = 15

epoch_selection_results = []
epoch_selection_histories = {}

for val_infant in EPOCH_SELECTION_INFANTS:

    print("\n" + "=" * 65)
    print(f"INNER VALIDATION INFANT: {val_infant}")
    print("=" * 65)

    # --------------------------------------------------------
    # Inner train / validation split
    # --------------------------------------------------------
    train_mask = infants_outer_dev != val_infant
    val_mask = infants_outer_dev == val_infant

    X_train_inner = X_outer_dev[train_mask].copy()
    y_train_inner = y_outer_dev[train_mask].copy()

    X_val_inner = X_outer_dev[val_mask].copy()
    y_val_inner = y_outer_dev[val_mask].copy()

    print(
        f"Training:   {len(y_train_inner)} "
        f"(positive={int(y_train_inner.sum())}, "
        f"negative={int((y_train_inner == 0).sum())})"
    )

    print(
        f"Validation: {len(y_val_inner)} "
        f"(positive={int(y_val_inner.sum())}, "
        f"negative={int((y_val_inner == 0).sum())})"
    )

    # --------------------------------------------------------
    # Train-only imputation
    #
    # Fit ONLY on inner training data.
    # --------------------------------------------------------
    n_samples, n_timesteps, n_features = X_train_inner.shape

    imputer = SimpleImputer(strategy="median")

    X_train_flat = X_train_inner.reshape(
        -1, n_features
    )

    X_val_flat = X_val_inner.reshape(
        -1, n_features
    )

    X_train_imp = imputer.fit_transform(X_train_flat)
    X_val_imp = imputer.transform(X_val_flat)

    # --------------------------------------------------------
    # Train-only scaling
    # --------------------------------------------------------
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_scaled = scaler.transform(X_val_imp)

    X_train_processed = X_train_scaled.reshape(
        n_samples,
        n_timesteps,
        n_features
    )

    X_val_processed = X_val_scaled.reshape(
        len(X_val_inner),
        n_timesteps,
        n_features
    )

    # --------------------------------------------------------
    # Class weights from INNER TRAINING DATA ONLY
    # --------------------------------------------------------
    classes = np.unique(y_train_inner)

    class_weights_array = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train_inner
    )

    class_weights = {
        int(cls): float(weight)
        for cls, weight in zip(classes, class_weights_array)
    }

    print("Class weights:", class_weights)

    # --------------------------------------------------------
    # Fresh GRU model
    # Same architecture as Cell 31.
    # --------------------------------------------------------
    tf.keras.backend.clear_session()

    tf.random.set_seed(RANDOM_SEED)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(n_timesteps, n_features)
        ),

        tf.keras.layers.GRU(16),

        tf.keras.layers.Dropout(0.20),

        tf.keras.layers.Dense(
            8,
            activation="relu"
        ),

        tf.keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.0005
        ),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            )
        ]
    )

    # --------------------------------------------------------
    # Early stopping is used ONLY to observe stabilization.
    # It will NOT be used in final outer LOSO.
    # --------------------------------------------------------
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        X_train_processed,
        y_train_inner,
        validation_data=(
            X_val_processed,
            y_val_inner
        ),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weights,
        shuffle=True,
        callbacks=[early_stopping],
        verbose=0
    )

    history_df = pd.DataFrame(history.history)

    # --------------------------------------------------------
    # Determine epoch with minimum validation loss
    # --------------------------------------------------------
    best_epoch_index = int(
        history_df["val_loss"].idxmin()
    )

    best_epoch = best_epoch_index + 1

    best_val_loss = float(
        history_df.loc[
            best_epoch_index,
            "val_loss"
        ]
    )

    final_epoch = len(history_df)

    epoch_selection_results.append({
        "validation_infant": int(val_infant),
        "train_samples": int(len(y_train_inner)),
        "train_positive": int(y_train_inner.sum()),
        "validation_samples": int(len(y_val_inner)),
        "validation_positive": int(y_val_inner.sum()),
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "training_stopped_epoch": final_epoch
    })

    epoch_selection_histories[val_infant] = history_df

    print(f"Best epoch: {best_epoch}")
    print(f"Best validation loss: {best_val_loss:.6f}")
    print(f"Training stopped after: {final_epoch} epochs")


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
epoch_selection_results = pd.DataFrame(
    epoch_selection_results
)

print("\n" + "=" * 65)
print("EXPLORATORY EPOCH-SELECTION SUMMARY")
print("=" * 65)

display(epoch_selection_results)

print(
    "\nBest epochs:",
    epoch_selection_results["best_epoch"].tolist()
)

print(
    "Median best epoch:",
    float(epoch_selection_results["best_epoch"].median())
)

print(
    "Mean best epoch:",
    float(epoch_selection_results["best_epoch"].mean())
)

print(
    "\nNOTE: These results are exploratory only."
)
print(
    "The final GRU LOSO evaluation will use ONE fixed epoch count"
)
print(
    "with NO validation split and NO early stopping."
)


INNER VALIDATION INFANT: 3
Training:   42 (positive=2, negative=40)
Validation: 7 (positive=2, negative=5)
Class weights: {0: 0.525, 1: 10.5}
Best epoch: 1
Best validation loss: 0.669527
Training stopped after: 16 epochs

INNER VALIDATION INFANT: 9
Training:   42 (positive=2, negative=40)
Validation: 7 (positive=2, negative=5)
Class weights: {0: 0.525, 1: 10.5}
Best epoch: 64
Best validation loss: 0.587827
Training stopped after: 79 epochs

EXPLORATORY EPOCH-SELECTION SUMMARY


,validation_infant,train_samples,train_positive,validation_samples,validation_positive,best_epoch,best_val_loss,training_stopped_epoch
0,3,42,2,7,2,1,0.669527,16
1,9,42,2,7,2,64,0.587827,79



Best epochs: [1, 64]
Median best epoch: 32.5
Mean best epoch: 32.5

NOTE: These results are exploratory only.
The final GRU LOSO evaluation will use ONE fixed epoch count
with NO validation split and NO early stopping.


In [72]:
# ============================================================
# CELL 32D — Prepare Outer Infant 3 Development Set
#
# Purpose:
#   Create the development pool for the exploratory
#   epoch-selection analysis with Infant 3 held out.
#
# Infant 3 must remain completely untouched during this
# exploratory analysis.
# ============================================================

OUTER_TEST_INFANT_3 = 3

outer3_dev_mask = np.array(
    [infant != OUTER_TEST_INFANT_3 for infant in infants_outer_dev]
)

X_outer3_dev = X_outer_dev[outer3_dev_mask].copy()
y_outer3_dev = y_outer_dev[outer3_dev_mask].copy()
infants_outer3_dev = infants_outer_dev[outer3_dev_mask].copy()

print("Outer test infant:", OUTER_TEST_INFANT_3)

print(
    "Outer development sequences:",
    len(X_outer3_dev)
)

print(
    "Outer development positives:",
    int(y_outer3_dev.sum())
)

print(
    "Outer development negatives:",
    int((y_outer3_dev == 0).sum())
)

print(
    "Outer development infants:",
    sorted(np.unique(infants_outer3_dev).tolist())
)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert OUTER_TEST_INFANT_3 not in np.unique(infants_outer3_dev), \
    "Outer Infant 3 leaked into development data."

assert (
    len(X_outer3_dev)
    == len(y_outer3_dev)
    == len(infants_outer3_dev)
)

print("\nPASS: Outer Infant 3 is excluded.")

Outer test infant: 3
Outer development sequences: 42
Outer development positives: 2
Outer development negatives: 40
Outer development infants: [2, 4, 5, 6, 7, 8, 9, 10]

PASS: Outer Infant 3 is excluded.


In [73]:
# ============================================================
# CELL 32E — Freeze GRU Development Configuration
#
# Purpose:
#   Freeze the GRU training configuration before the outer
#   LOSO evaluation.
#
# Important:
#   - No outer test infant is used here.
#   - No validation split will be used in final LOSO training.
#   - No early stopping will be used.
#   - Epoch count is fixed at 25.
#
# Status:
#   DEVELOPMENT / VALIDATION
# ============================================================

GRU_FIXED_EPOCHS = 25
GRU_BATCH_SIZE = 8
GRU_THRESHOLD = 0.50
GRU_LEARNING_RATE = 0.0005

GRU_UNITS = 16
GRU_DROPOUT = 0.20
GRU_DENSE_UNITS = 8

print("=" * 70)
print("GRU DEVELOPMENT CONFIGURATION — FROZEN")
print("=" * 70)

print("\nInput:")
print("  Time steps       : 11")
print("  Features / step  : 20")
print("  Input shape      : (11, 20)")

print("\nArchitecture:")
print(f"  GRU units        : {GRU_UNITS}")
print(f"  Dropout          : {GRU_DROPOUT}")
print(f"  Dense units      : {GRU_DENSE_UNITS}")
print("  Output           : Sigmoid")

print("\nTraining:")
print(f"  Optimizer        : Adam")
print(f"  Learning rate    : {GRU_LEARNING_RATE}")
print("  Loss             : Binary cross-entropy")
print(f"  Batch size       : {GRU_BATCH_SIZE}")
print(f"  Fixed epochs     : {GRU_FIXED_EPOCHS}")
print(f"  Classification threshold : {GRU_THRESHOLD}")

print("\nLOSO training policy:")
print("  Outer test infant        : NEVER used for training")
print("  Validation split         : NONE")
print("  Early stopping           : NONE")
print("  Class weights            : training fold only")
print("  Imputation               : training fold only")
print("  Scaling                  : training fold only")

print("\nEpoch-selection note:")
print("  Inner exploratory epoch estimates were unstable")
print("  because of the very small positive-window count.")
print("  A fixed 25-epoch schedule is therefore used for")
print("  this exploratory GRU LOSO evaluation.")

print("\nStatus:")
print("  DEVELOPMENT / VALIDATION")
print("  NOT clinical validation")

print("=" * 70)

assert GRU_FIXED_EPOCHS == 25
assert GRU_BATCH_SIZE == 8
assert GRU_THRESHOLD == 0.50
assert GRU_LEARNING_RATE == 0.0005

print("\nPASS — GRU configuration frozen for outer LOSO.")

GRU DEVELOPMENT CONFIGURATION — FROZEN

Input:
  Time steps       : 11
  Features / step  : 20
  Input shape      : (11, 20)

Architecture:
  GRU units        : 16
  Dropout          : 0.2
  Dense units      : 8
  Output           : Sigmoid

Training:
  Optimizer        : Adam
  Learning rate    : 0.0005
  Loss             : Binary cross-entropy
  Batch size       : 8
  Fixed epochs     : 25
  Classification threshold : 0.5

LOSO training policy:
  Outer test infant        : NEVER used for training
  Validation split         : NONE
  Early stopping           : NONE
  Class weights            : training fold only
  Imputation               : training fold only
  Scaling                  : training fold only

Epoch-selection note:
  Inner exploratory epoch estimates were unstable
  because of the very small positive-window count.
  A fixed 25-epoch schedule is therefore used for
  this exploratory GRU LOSO evaluation.

Status:
  DEVELOPMENT / VALIDATION
  NOT clinical validation

PASS — 

In [85]:
# ============================================================
# CELL 32F — CORRECTED Outer LOSO GRU Training
#
# IMPORTANT:
#   Cell 30 has ALREADY performed:
#       - training-fold-only imputation
#       - training-fold-only scaling
#
# Therefore this cell MUST NOT preprocess X_train/X_test again.
#
# Frozen configuration:
#   Epochs       = 25
#   Batch size   = 8
#   Learning rate= 0.0005
#   GRU units    = 16
#   Dropout      = 0.20
#   Dense units  = 8
#   Threshold    = 0.50
#   No validation split
#   No early stopping
#
# Status:
#   Corrected development / validation experiment
# ============================================================

import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

# ------------------------------------------------------------
# Verify required variables
# ------------------------------------------------------------

assert "gru_loso_folds" in globals(), \
    "gru_loso_folds is not available. Run Cell 30 first."

assert "GRU_FIXED_EPOCHS" in globals()
assert "GRU_BATCH_SIZE" in globals()
assert "GRU_THRESHOLD" in globals()
assert "GRU_LEARNING_RATE" in globals()
assert "GRU_UNITS" in globals()
assert "GRU_DROPOUT" in globals()
assert "GRU_DENSE_UNITS" in globals()

assert len(gru_loso_folds) == 10


# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

gru_loso_predictions = []
gru_fold_summaries = []


# ------------------------------------------------------------
# Outer LOSO loop
# ------------------------------------------------------------

for fold_number, fold in enumerate(
    gru_loso_folds,
    start=1
):

    test_infant = int(
        fold["test_infant"]
    )

    # --------------------------------------------------------
    # IMPORTANT:
    # These arrays are ALREADY processed by Cell 30.
    # --------------------------------------------------------

    X_train = fold["X_train"].copy()
    y_train = fold["y_train"].copy()

    X_test = fold["X_test"].copy()
    y_test = fold["y_test"].copy()

    train_infants = fold["train_infants"].copy()

    print("\n" + "=" * 70)
    print(
        f"CORRECTED GRU LOSO FOLD "
        f"{fold_number}/10 — Test Infant {test_infant}"
    )
    print("=" * 70)

    print(
        "Training samples:",
        len(X_train),
        "| positives:",
        int(y_train.sum()),
        "| negatives:",
        int((y_train == 0).sum())
    )

    print(
        "Test samples:",
        len(X_test),
        "| positives:",
        int(y_test.sum()),
        "| negatives:",
        int((y_test == 0).sum())
    )

    print(
        "Input shape:",
        X_train.shape
    )

    # --------------------------------------------------------
    # Leakage check
    # --------------------------------------------------------

    assert test_infant not in set(
        train_infants.tolist()
    ), (
        f"Test Infant {test_infant} "
        "is present in training data."
    )

    # --------------------------------------------------------
    # Confirm no NaN / Inf remains after Cell 30 processing
    # --------------------------------------------------------

    assert np.isfinite(
        X_train
    ).all(), (
        f"Non-finite value in training data "
        f"for Infant {test_infant}."
    )

    assert np.isfinite(
        X_test
    ).all(), (
        f"Non-finite value in test data "
        f"for Infant {test_infant}."
    )

    # --------------------------------------------------------
    # Training-fold-only class weights
    # --------------------------------------------------------

    classes = np.unique(
        y_train
    )

    class_weights_array = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    class_weights = {
        int(cls): float(weight)
        for cls, weight in zip(
            classes,
            class_weights_array
        )
    }

    print(
        "Class weights:",
        class_weights
    )

    # --------------------------------------------------------
    # Fresh GRU model
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    tf.keras.utils.set_random_seed(
        RANDOM_SEED + test_infant
    )

    n_steps = X_train.shape[1]
    n_features = X_train.shape[2]

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(n_steps, n_features)
        ),

        tf.keras.layers.GRU(
            GRU_UNITS
        ),

        tf.keras.layers.Dropout(
            GRU_DROPOUT
        ),

        tf.keras.layers.Dense(
            GRU_DENSE_UNITS,
            activation="relu"
        ),

        tf.keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=GRU_LEARNING_RATE
        ),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(
                name="auc"
            )
        ]
    )

    # --------------------------------------------------------
    # Fixed 25-epoch training
    #
    # NO validation split.
    # NO early stopping.
    # --------------------------------------------------------

    history = model.fit(
        X_train,
        y_train,
        epochs=GRU_FIXED_EPOCHS,
        batch_size=GRU_BATCH_SIZE,
        class_weight=class_weights,
        shuffle=True,
        verbose=0
    )

    # --------------------------------------------------------
    # Predict ONLY held-out infant
    # --------------------------------------------------------

    test_probabilities = model.predict(
        X_test,
        verbose=0
    ).ravel()

    test_predictions = (
        test_probabilities >= GRU_THRESHOLD
    ).astype(int)

    # --------------------------------------------------------
    # Store out-of-fold predictions
    # --------------------------------------------------------

    for i in range(len(y_test)):

        gru_loso_predictions.append({
            "fold": fold_number,
            "test_infant": test_infant,
            "true_label": int(y_test[i]),
            "probability": float(
                test_probabilities[i]
            ),
            "prediction": int(
                test_predictions[i]
            )
        })

    # --------------------------------------------------------
    # Fold summary
    # --------------------------------------------------------

    gru_fold_summaries.append({
        "fold": fold_number,
        "test_infant": test_infant,
        "train_samples": int(
            len(y_train)
        ),
        "train_positive": int(
            y_train.sum()
        ),
        "train_negative": int(
            (y_train == 0).sum()
        ),
        "test_samples": int(
            len(y_test)
        ),
        "test_positive": int(
            y_test.sum()
        ),
        "test_negative": int(
            (y_test == 0).sum()
        ),
        "epochs": int(
            GRU_FIXED_EPOCHS
        )
    })

    print(
        f"Fold {fold_number} complete."
    )

    print(
        "Test probabilities:",
        np.round(
            test_probabilities,
            4
        )
    )

    print(
        "Test predictions:",
        test_predictions.tolist()
    )


# ------------------------------------------------------------
# Convert to DataFrames
# ------------------------------------------------------------

gru_loso_predictions = pd.DataFrame(
    gru_loso_predictions
)

gru_fold_summaries = pd.DataFrame(
    gru_fold_summaries
)


# ------------------------------------------------------------
# Final structural checks
# ------------------------------------------------------------

assert len(
    gru_loso_predictions
) == 55

assert len(
    gru_fold_summaries
) == 10

assert (
    gru_loso_predictions["probability"]
    .between(0, 1)
    .all()
)

assert (
    gru_loso_predictions["prediction"]
    .isin([0, 1])
    .all()
)

assert (
    gru_loso_predictions["true_label"]
    .isin([0, 1])
    .all()
)

assert not (
    gru_loso_predictions.isna().any().any()
)


# ------------------------------------------------------------
# Verify label distribution
# ------------------------------------------------------------

assert (
    gru_loso_predictions[
        "true_label"
    ].sum()
    == 5
)

assert (
    (
        gru_loso_predictions[
            "true_label"
        ] == 0
    ).sum()
    == 50
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CORRECTED GRU OUTER LOSO TRAINING COMPLETE")
print("=" * 70)

print(
    "Total OOF predictions:",
    len(gru_loso_predictions)
)

print(
    "True positives:",
    int(
        gru_loso_predictions[
            "true_label"
        ].sum()
    )
)

print(
    "True negatives:",
    int(
        (
            gru_loso_predictions[
                "true_label"
            ] == 0
        ).sum()
    )
)

print(
    "Predicted positives:",
    int(
        gru_loso_predictions[
            "prediction"
        ].sum()
    )
)

print(
    "Predicted negatives:",
    int(
        (
            gru_loso_predictions[
                "prediction"
            ] == 0
        ).sum()
    )
)

print(
    "\nFold summary:"
)

display(
    gru_fold_summaries
)

print(
    "\nPASS: 10 corrected outer-LOSO folds completed."
)

print(
    "PASS: 55 out-of-fold predictions generated."
)

print(
    "PASS: Cell-30 preprocessing was used exactly once."
)

print(
    "PASS: No second imputation/scaling was applied."
)

print(
    "PASS: Class weights calculated from training data only."
)

print(
    "STATUS: Development / validation — "
    "not clinical validation."
)


CORRECTED GRU LOSO FOLD 1/10 — Test Infant 1
Training samples: 49 | positives: 4 | negatives: 45
Test samples: 6 | positives: 1 | negatives: 5
Input shape: (49, 11, 20)
Class weights: {0: 0.5444444444444444, 1: 6.125}
Fold 1 complete.
Test probabilities: [0.4669 0.5154 0.2375 0.1    0.3084 0.1851]
Test predictions: [0, 1, 0, 0, 0, 0]

CORRECTED GRU LOSO FOLD 2/10 — Test Infant 2
Training samples: 50 | positives: 5 | negatives: 45
Test samples: 5 | positives: 0 | negatives: 5
Input shape: (50, 11, 20)
Class weights: {0: 0.5555555555555556, 1: 5.0}
Fold 2 complete.
Test probabilities: [0.6063 0.6862 0.4348 0.5752 0.4044]
Test predictions: [1, 1, 0, 1, 0]

CORRECTED GRU LOSO FOLD 3/10 — Test Infant 3
Training samples: 48 | positives: 3 | negatives: 45
Test samples: 7 | positives: 2 | negatives: 5
Input shape: (48, 11, 20)
Class weights: {0: 0.5333333333333333, 1: 8.0}
Fold 3 complete.
Test probabilities: [0.8784 0.5837 0.6984 0.7803 0.7566 0.4909 0.5325]
Test predictions: [1, 1, 1, 1, 1,

,fold,test_infant,train_samples,train_positive,train_negative,test_samples,test_positive,test_negative,epochs
0,1,1,49,4,45,6,1,5,25
1,2,2,50,5,45,5,0,5,25
2,3,3,48,3,45,7,2,5,25
3,4,4,50,5,45,5,0,5,25
4,5,5,50,5,45,5,0,5,25
5,6,6,50,5,45,5,0,5,25
6,7,7,50,5,45,5,0,5,25
7,8,8,50,5,45,5,0,5,25
8,9,9,48,3,45,7,2,5,25
9,10,10,50,5,45,5,0,5,25



PASS: 10 corrected outer-LOSO folds completed.
PASS: 55 out-of-fold predictions generated.
PASS: Cell-30 preprocessing was used exactly once.
PASS: No second imputation/scaling was applied.
PASS: Class weights calculated from training data only.
STATUS: Development / validation — not clinical validation.


In [87]:
# ============================================================
# CELL 33 — GRU Outer-LOSO Pooled Evaluation
#
# Purpose:
#   Evaluate the 55 out-of-fold predictions generated by
#   the outer LOSO GRU training.
#
# No model training is performed in this cell.
#
# Metrics:
#   - Confusion matrix
#   - Accuracy
#   - Sensitivity / Recall
#   - Specificity
#   - Precision
#   - F1-score
#   - AUROC
#   - AUPRC
#
# Status:
#   Development / validation
#   NOT clinical validation
# ============================================================

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# Verify predictions from Cell 32F
# ------------------------------------------------------------

assert "gru_loso_predictions" in globals(), \
    "gru_loso_predictions not available. Run Cell 32F first."

assert len(gru_loso_predictions) == 55, \
    "Expected exactly 55 outer-LOSO predictions."

assert not gru_loso_predictions.isna().any().any(), \
    "NaN detected in GRU predictions."

# ------------------------------------------------------------
# Extract values
# ------------------------------------------------------------

y_true = (
    gru_loso_predictions["true_label"]
    .to_numpy()
)

y_pred = (
    gru_loso_predictions["prediction"]
    .to_numpy()
)

y_prob = (
    gru_loso_predictions["probability"]
    .to_numpy()
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

# ------------------------------------------------------------
# Classification metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_true,
    y_pred
)

sensitivity = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else np.nan
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

# ------------------------------------------------------------
# Probability-based metrics
# ------------------------------------------------------------

auroc = roc_auc_score(
    y_true,
    y_prob
)

auprc = average_precision_score(
    y_true,
    y_prob
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 70)
print("GRU OUTER-LOSO POOLED EVALUATION")
print("=" * 70)

print("\nConfusion Matrix:")
print(cm)

print("\nConfusion Matrix Components:")
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

print("\nClassification Metrics:")
print(f"Accuracy:     {accuracy:.4f}")
print(f"Sensitivity:  {sensitivity:.4f}")
print(f"Specificity:  {specificity:.4f}")
print(f"Precision:    {precision:.4f}")
print(f"F1-score:     {f1:.4f}")

print("\nProbability Metrics:")
print(f"AUROC:        {auroc:.4f}")
print(f"AUPRC:        {auprc:.4f}")

print("\nDataset:")
print("Total windows:", len(y_true))
print("Actual positives:", int(y_true.sum()))
print("Actual negatives:", int((y_true == 0).sum()))
print("Predicted positives:", int(y_pred.sum()))
print("Predicted negatives:", int((y_pred == 0).sum()))

# ------------------------------------------------------------
# Store metrics
# ------------------------------------------------------------

gru_loso_metrics = pd.DataFrame([{
    "model": "GRU",
    "evaluation": "Outer LOSO pooled",
    "total_windows": int(len(y_true)),
    "true_positive_windows": int(y_true.sum()),
    "true_negative_windows": int((y_true == 0).sum()),
    "predicted_positive_windows": int(y_pred.sum()),
    "predicted_negative_windows": int((y_pred == 0).sum()),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp),
    "accuracy": float(accuracy),
    "sensitivity": float(sensitivity),
    "specificity": float(specificity),
    "precision": float(precision),
    "f1": float(f1),
    "auroc": float(auroc),
    "auprc": float(auprc),
    "threshold": float(GRU_THRESHOLD),
    "epochs": int(GRU_FIXED_EPOCHS)
}])

print("\nMetrics table:")
display(gru_loso_metrics)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert tn + fp + fn + tp == 55

assert 0 <= accuracy <= 1
assert 0 <= sensitivity <= 1
assert 0 <= specificity <= 1
assert 0 <= precision <= 1
assert 0 <= f1 <= 1
assert 0 <= auroc <= 1
assert 0 <= auprc <= 1

print("\nPASS: 55 outer-LOSO predictions evaluated.")
print("PASS: Confusion matrix accounts for all 55 windows.")
print("PASS: Probability metrics calculated from held-out predictions.")
print("STATUS: Development / validation — not clinical validation.")

GRU OUTER-LOSO POOLED EVALUATION

Confusion Matrix:
[[30 20]
 [ 2  3]]

Confusion Matrix Components:
TN: 30
FP: 20
FN: 2
TP: 3

Classification Metrics:
Accuracy:     0.6000
Sensitivity:  0.6000
Specificity:  0.6000
Precision:    0.1304
F1-score:     0.2143

Probability Metrics:
AUROC:        0.7160
AUPRC:        0.3292

Dataset:
Total windows: 55
Actual positives: 5
Actual negatives: 50
Predicted positives: 23
Predicted negatives: 32

Metrics table:


,model,evaluation,total_windows,true_positive_windows,true_negative_windows,predicted_positive_windows,predicted_negative_windows,tn,fp,fn,tp,accuracy,sensitivity,specificity,precision,f1,auroc,auprc,threshold,epochs
0,GRU,Outer LOSO pooled,55,5,50,23,32,30,20,2,3,0.6,0.6,0.6,0.130435,0.214286,0.716,0.329187,0.5,25



PASS: 55 outer-LOSO predictions evaluated.
PASS: Confusion matrix accounts for all 55 windows.
PASS: Probability metrics calculated from held-out predictions.
STATUS: Development / validation — not clinical validation.


In [88]:
# ============================================================
# CELL 34 — Save GRU Outer-LOSO Results
#
# Purpose:
#   Save and independently verify the completed GRU results.
#   No model training is performed.
# ============================================================

# ------------------------------------------------------------
# Define output paths
# ------------------------------------------------------------

GRU_PREDICTIONS_PATH = (
    REPORTS_DIR /
    "pics_temporal_loso_gru_predictions.csv"
)

GRU_FOLD_SUMMARY_PATH = (
    REPORTS_DIR /
    "pics_temporal_loso_gru_fold_summary.csv"
)

GRU_METRICS_PATH = (
    REPORTS_DIR /
    "pics_temporal_loso_gru_metrics.csv"
)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

gru_loso_predictions.to_csv(
    GRU_PREDICTIONS_PATH,
    index=False
)

gru_fold_summaries.to_csv(
    GRU_FOLD_SUMMARY_PATH,
    index=False
)

gru_loso_metrics.to_csv(
    GRU_METRICS_PATH,
    index=False
)


# ------------------------------------------------------------
# Reload saved files
# ------------------------------------------------------------

gru_predictions_check = pd.read_csv(
    GRU_PREDICTIONS_PATH
)

gru_fold_check = pd.read_csv(
    GRU_FOLD_SUMMARY_PATH
)

gru_metrics_check = pd.read_csv(
    GRU_METRICS_PATH
)


# ------------------------------------------------------------
# Basic integrity checks
# ------------------------------------------------------------

assert len(gru_predictions_check) == 55
assert len(gru_fold_check) == 10
assert len(gru_metrics_check) == 1

assert set(
    gru_predictions_check["test_infant"]
) == set(range(1, 11))

assert (
    gru_predictions_check["true_label"].sum()
    == 5
)

assert (
    (gru_predictions_check["true_label"] == 0).sum()
    == 50
)

assert (
    gru_predictions_check["prediction"].sum()
    == 23
)

assert (
    (gru_predictions_check["prediction"] == 0).sum()
    == 32
)


# ------------------------------------------------------------
# Verify saved metrics against the ORIGINAL metrics
#
# Do not hard-code rounded values such as 0.3292.
# ------------------------------------------------------------

saved_metrics = gru_metrics_check.iloc[0]
original_metrics = gru_loso_metrics.iloc[0]

metric_columns = [
    "accuracy",
    "sensitivity",
    "specificity",
    "precision",
    "f1",
    "auroc",
    "auprc"
]

for metric in metric_columns:

    assert np.isclose(
        float(saved_metrics[metric]),
        float(original_metrics[metric]),
        rtol=1e-9,
        atol=1e-9
    ), f"Mismatch in saved metric: {metric}"


# ------------------------------------------------------------
# Display exact saved metrics
# ------------------------------------------------------------

print("=" * 70)
print("GRU RESULTS SAVED AND VERIFIED")
print("=" * 70)

print("\nPredictions:")
print(GRU_PREDICTIONS_PATH)

print("\nFold summary:")
print(GRU_FOLD_SUMMARY_PATH)

print("\nMetrics:")
print(GRU_METRICS_PATH)

print("\nSaved prediction rows:",
      len(gru_predictions_check))

print("Saved fold rows:",
      len(gru_fold_check))

print("Saved metric rows:",
      len(gru_metrics_check))

print("\nExact saved metrics:")

for metric in metric_columns:
    print(
        f"{metric:15s}: "
        f"{float(saved_metrics[metric]):.8f}"
    )


# ------------------------------------------------------------
# Final PASS messages
# ------------------------------------------------------------

print("\nPASS: GRU predictions saved and reloaded.")
print("PASS: 55 predictions verified.")
print("PASS: 10 LOSO folds verified.")
print("PASS: 5 positive and 50 negative labels verified.")
print("PASS: Saved metrics match the original metrics.")
print("STATUS: Development / validation — not clinical validation.")

GRU RESULTS SAVED AND VERIFIED

Predictions:
c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_temporal_loso_gru_predictions.csv

Fold summary:
c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_temporal_loso_gru_fold_summary.csv

Metrics:
c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_temporal_loso_gru_metrics.csv

Saved prediction rows: 55
Saved fold rows: 10
Saved metric rows: 1

Exact saved metrics:
accuracy       : 0.60000000
sensitivity    : 0.60000000
specificity    : 0.60000000
precision      : 0.13043478
f1             : 0.21428571
auroc          : 0.71600000
auprc          : 0.32918741

PASS: GRU predictions saved and reloaded.
PASS: 55 predictions verified.
PASS: 10 LOSO folds verified.
PASS: 5 positive and 50 negative labels verified.
PASS: Saved metrics match the original metrics.
STATUS: Development / validation — not clinical validation.


In [89]:
# ============================================================
# CELL 35 — Per-Infant GRU LOSO Analysis
#
# Purpose:
#   Analyze the outer-LOSO GRU predictions separately for
#   each held-out infant.
#
# No model training is performed.
# ============================================================

from sklearn.metrics import confusion_matrix

# ------------------------------------------------------------
# Verify required predictions
# ------------------------------------------------------------

assert "gru_loso_predictions" in globals(), \
    "gru_loso_predictions is not available."

assert len(gru_loso_predictions) == 55, \
    "Expected 55 outer-LOSO predictions."


# ------------------------------------------------------------
# Per-infant analysis
# ------------------------------------------------------------

per_infant_results = []

for infant in sorted(
    gru_loso_predictions["test_infant"].unique()
):

    infant_df = gru_loso_predictions[
        gru_loso_predictions["test_infant"] == infant
    ].copy()

    y_true_infant = infant_df[
        "true_label"
    ].to_numpy()

    y_pred_infant = infant_df[
        "prediction"
    ].to_numpy()

    y_prob_infant = infant_df[
        "probability"
    ].to_numpy()

    cm_infant = confusion_matrix(
        y_true_infant,
        y_pred_infant,
        labels=[0, 1]
    )

    tn_i, fp_i, fn_i, tp_i = cm_infant.ravel()

    total_i = len(y_true_infant)
    positive_i = int(y_true_infant.sum())
    negative_i = int(
        (y_true_infant == 0).sum()
    )

    predicted_positive_i = int(
        y_pred_infant.sum()
    )

    # --------------------------------------------------------
    # Sensitivity
    # --------------------------------------------------------

    sensitivity_i = (
        tp_i / (tp_i + fn_i)
        if (tp_i + fn_i) > 0
        else np.nan
    )

    # --------------------------------------------------------
    # Specificity
    # --------------------------------------------------------

    specificity_i = (
        tn_i / (tn_i + fp_i)
        if (tn_i + fp_i) > 0
        else np.nan
    )

    # --------------------------------------------------------
    # Precision
    # --------------------------------------------------------

    precision_i = (
        tp_i / (tp_i + fp_i)
        if (tp_i + fp_i) > 0
        else 0.0
    )

    # --------------------------------------------------------
    # F1
    # --------------------------------------------------------

    f1_i = (
        2 * precision_i * sensitivity_i
        / (precision_i + sensitivity_i)
        if (precision_i + sensitivity_i) > 0
        else 0.0
    )

    # --------------------------------------------------------
    # Accuracy
    # --------------------------------------------------------

    accuracy_i = (
        (tp_i + tn_i) / total_i
        if total_i > 0
        else np.nan
    )

    # --------------------------------------------------------
    # AUROC / AUPRC
    #
    # Only calculate when both classes are present.
    # --------------------------------------------------------

    if len(np.unique(y_true_infant)) == 2:

        auroc_i = roc_auc_score(
            y_true_infant,
            y_prob_infant
        )

        auprc_i = average_precision_score(
            y_true_infant,
            y_prob_infant
        )

    else:

        auroc_i = np.nan
        auprc_i = np.nan

    per_infant_results.append({
        "infant": int(infant),
        "total_windows": int(total_i),
        "positive_windows": int(positive_i),
        "negative_windows": int(negative_i),
        "predicted_positive": int(
            predicted_positive_i
        ),
        "tn": int(tn_i),
        "fp": int(fp_i),
        "fn": int(fn_i),
        "tp": int(tp_i),
        "accuracy": float(accuracy_i),
        "sensitivity": float(sensitivity_i),
        "specificity": float(specificity_i),
        "precision": float(precision_i),
        "f1": float(f1_i),
        "auroc": float(auroc_i)
            if not np.isnan(auroc_i)
            else np.nan,
        "auprc": float(auprc_i)
            if not np.isnan(auprc_i)
            else np.nan
    })


# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------

per_infant_results = pd.DataFrame(
    per_infant_results
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 70)
print("GRU PER-INFANT OUTER-LOSO RESULTS")
print("=" * 70)

display(per_infant_results)


# ------------------------------------------------------------
# Identify infants containing positive windows
# ------------------------------------------------------------

positive_bearing_infants = (
    per_infant_results[
        per_infant_results["positive_windows"] > 0
    ]["infant"]
    .tolist()
)

print(
    "\nInfants containing positive precursor windows:",
    positive_bearing_infants
)

print(
    "Number of positive-bearing infants:",
    len(positive_bearing_infants)
)


# ------------------------------------------------------------
# Summary for positive-bearing infants
# ------------------------------------------------------------

print("\nPositive-bearing infant results:")

display(
    per_infant_results[
        per_infant_results["positive_windows"] > 0
    ]
)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert len(per_infant_results) == 10

assert (
    per_infant_results["total_windows"].sum()
    == 55
)

assert (
    per_infant_results["positive_windows"].sum()
    == 5
)

assert (
    per_infant_results["negative_windows"].sum()
    == 50
)

assert (
    set(positive_bearing_infants)
    == {1, 3, 9}
)

print("\nPASS: 10 infants analyzed.")
print("PASS: 55 windows accounted for.")
print("PASS: 5 positive windows accounted for.")
print("PASS: Positive windows occur only in Infants 1, 3, and 9.")
print("STATUS: Development / validation — not clinical validation.")

GRU PER-INFANT OUTER-LOSO RESULTS


,infant,total_windows,positive_windows,negative_windows,predicted_positive,tn,fp,fn,tp,accuracy,sensitivity,specificity,precision,f1,auroc,auprc
0,1,6,1,5,1,4,1,1,0,0.666667,0.0,0.8,0.000000,0.0,0.8,0.5
1,2,5,0,5,3,2,3,0,0,0.400000,NaN,0.4,0.000000,0.0,NaN,NaN
2,3,7,2,5,6,1,4,0,2,0.428571,1.0,0.2,0.333333,0.5,0.7,0.7
3,4,5,0,5,1,4,1,0,0,0.800000,NaN,0.8,0.000000,0.0,NaN,NaN
4,5,5,0,5,0,5,0,0,0,1.000000,NaN,1.0,0.000000,0.0,NaN,NaN
5,6,5,0,5,2,3,2,0,0,0.600000,NaN,0.6,0.000000,0.0,NaN,NaN
6,7,5,0,5,0,5,0,0,0,1.000000,NaN,1.0,0.000000,0.0,NaN,NaN
7,8,5,0,5,5,0,5,0,0,0.000000,NaN,0.0,0.000000,0.0,NaN,NaN
8,9,7,2,5,3,3,2,1,1,0.571429,0.5,0.6,0.333333,0.4,0.7,0.5
9,10,5,0,5,2,3,2,0,0,0.600000,NaN,0.6,0.000000,0.0,NaN,NaN



Infants containing positive precursor windows: [1, 3, 9]
Number of positive-bearing infants: 3

Positive-bearing infant results:


,infant,total_windows,positive_windows,negative_windows,predicted_positive,tn,fp,fn,tp,accuracy,sensitivity,specificity,precision,f1,auroc,auprc
0,1,6,1,5,1,4,1,1,0,0.666667,0.0,0.8,0.000000,0.0,0.8,0.5
2,3,7,2,5,6,1,4,0,2,0.428571,1.0,0.2,0.333333,0.5,0.7,0.7
8,9,7,2,5,3,3,2,1,1,0.571429,0.5,0.6,0.333333,0.4,0.7,0.5



PASS: 10 infants analyzed.
PASS: 55 windows accounted for.
PASS: 5 positive windows accounted for.
PASS: Positive windows occur only in Infants 1, 3, and 9.
STATUS: Development / validation — not clinical validation.


In [90]:
# ============================================================
# CELL 36 — Save Per-Infant GRU LOSO Results
#
# No model training is performed.
# ============================================================

GRU_PER_INFANT_PATH = (
    REPORTS_DIR /
    "pics_temporal_loso_gru_per_infant.csv"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

per_infant_results.to_csv(
    GRU_PER_INFANT_PATH,
    index=False
)

# ------------------------------------------------------------
# Reload
# ------------------------------------------------------------

per_infant_check = pd.read_csv(
    GRU_PER_INFANT_PATH
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert len(per_infant_check) == 10

assert (
    per_infant_check["total_windows"].sum()
    == 55
)

assert (
    per_infant_check["positive_windows"].sum()
    == 5
)

assert (
    per_infant_check["negative_windows"].sum()
    == 50
)

assert set(
    per_infant_check[
        per_infant_check["positive_windows"] > 0
    ]["infant"]
) == {1, 3, 9}

print("=" * 70)
print("PER-INFANT GRU RESULTS SAVED")
print("=" * 70)

print(
    "\nFile:",
    GRU_PER_INFANT_PATH
)

print(
    "Rows saved:",
    len(per_infant_check)
)

print(
    "Total windows:",
    int(per_infant_check["total_windows"].sum())
)

print(
    "Positive windows:",
    int(per_infant_check["positive_windows"].sum())
)

print(
    "Negative windows:",
    int(per_infant_check["negative_windows"].sum())
)

print(
    "\nPASS: 10 infant-level results saved and verified."
)

PER-INFANT GRU RESULTS SAVED

File: c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_temporal_loso_gru_per_infant.csv
Rows saved: 10
Total windows: 55
Positive windows: 5
Negative windows: 50

PASS: 10 infant-level results saved and verified.


In [91]:
# ============================================================
# CELL 37 — Verified Positive-Window GRU Mapping
#
# Purpose:
#   Map the 55 corrected GRU OOF predictions back to the
#   authoritative temporal metadata.
#
# IMPORTANT:
#   Before mapping, every LOSO test fold is independently
#   verified against the original temporal sequences using
#   that fold's Cell-30 imputer and scaler.
#
# No model training is performed.
# ============================================================

# ------------------------------------------------------------
# Verify required objects
# ------------------------------------------------------------

assert "gru_loso_predictions" in globals()
assert len(gru_loso_predictions) == 55

assert "gru_loso_folds" in globals()
assert len(gru_loso_folds) == 10


# ------------------------------------------------------------
# Load authoritative temporal data
# ------------------------------------------------------------

TEMPORAL_X_PATH = (
    PROJECT_ROOT /
    "reports" /
    "pics_final_temporal_X_sequences.npy"
)

TEMPORAL_Y_PATH = (
    PROJECT_ROOT /
    "reports" /
    "pics_final_temporal_y_sequences.npy"
)

TEMPORAL_METADATA_PATH = (
    REPORTS_DIR /
    "pics_final_temporal_sequence_metadata.csv"
)

X_raw = np.load(
    TEMPORAL_X_PATH
)

y_raw = np.load(
    TEMPORAL_Y_PATH
)

temporal_metadata = pd.read_csv(
    TEMPORAL_METADATA_PATH
)


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

assert X_raw.shape[0] == 55
assert len(y_raw) == 55
assert len(temporal_metadata) == 55

assert np.array_equal(
    y_raw,
    temporal_metadata["risk_label"].to_numpy()
)

print("Authoritative temporal dataset:", X_raw.shape)
print("Authoritative labels:", y_raw.shape)
print("Metadata rows:", len(temporal_metadata))


# ------------------------------------------------------------
# Reconstruct and verify every LOSO test fold
# ------------------------------------------------------------

mapped_prediction_rows = []

for fold_number, fold in enumerate(
    gru_loso_folds,
    start=1
):

    test_infant = int(
        fold["test_infant"]
    )

    X_test_stored = fold["X_test"]
    y_test_stored = fold["y_test"]

    fold_imputer = fold["imputer"]
    fold_scaler = fold["scaler"]

    # --------------------------------------------------------
    # Locate this infant in authoritative metadata
    # --------------------------------------------------------

    infant_mask = (
        temporal_metadata["infant"].to_numpy()
        == test_infant
    )

    infant_indices = np.where(
        infant_mask
    )[0]

    X_infant_raw = X_raw[
        infant_indices
    ]

    y_infant_raw = y_raw[
        infant_indices
    ]

    # --------------------------------------------------------
    # Verify sample count
    # --------------------------------------------------------

    assert len(infant_indices) == len(
        X_test_stored
    ), (
        f"Sample count mismatch for Infant "
        f"{test_infant}."
    )

    # --------------------------------------------------------
    # Reproduce Cell-30 preprocessing
    # --------------------------------------------------------

    n_samples = X_infant_raw.shape[0]
    n_steps = X_infant_raw.shape[1]
    n_features = X_infant_raw.shape[2]

    X_flat = X_infant_raw.reshape(
        -1,
        n_features
    )

    X_imputed = fold_imputer.transform(
        X_flat
    )

    X_scaled = fold_scaler.transform(
        X_imputed
    )

    X_processed = X_scaled.reshape(
        n_samples,
        n_steps,
        n_features
    )

    # --------------------------------------------------------
    # CRITICAL verification
    # --------------------------------------------------------

    assert np.allclose(
        X_processed,
        X_test_stored,
        equal_nan=True,
        rtol=1e-5,
        atol=1e-7
    ), (
        f"Processed X_test ordering/content mismatch "
        f"for Infant {test_infant}."
    )

    # --------------------------------------------------------
    # Verify labels and ordering
    # --------------------------------------------------------

    assert np.array_equal(
        y_infant_raw,
        y_test_stored
    ), (
        f"Label ordering mismatch "
        f"for Infant {test_infant}."
    )

    # --------------------------------------------------------
    # Extract predictions for this exact fold
    # --------------------------------------------------------

    fold_predictions = (
        gru_loso_predictions[
            gru_loso_predictions["fold"]
            == fold_number
        ]
        .reset_index(drop=True)
    )

    assert len(
        fold_predictions
    ) == len(
        infant_indices
    ), (
        f"Prediction count mismatch "
        f"for Infant {test_infant}."
    )

    # --------------------------------------------------------
    # Map prediction rows to metadata rows
    # --------------------------------------------------------

    for local_index, global_index in enumerate(
        infant_indices
    ):

        metadata_row = (
            temporal_metadata.iloc[
                global_index
            ]
        )

        prediction_row = (
            fold_predictions.iloc[
                local_index
            ]
        )

        mapped_prediction_rows.append({

            "fold": int(
                prediction_row["fold"]
            ),

            "infant": int(
                test_infant
            ),

            "event_number": (
                int(
                    metadata_row[
                        "event_number"
                    ]
                )
                if pd.notna(
                    metadata_row[
                        "event_number"
                    ]
                )
                else np.nan
            ),

            "window_type": (
                metadata_row[
                    "window_type"
                ]
            ),

            "risk_label": int(
                metadata_row[
                    "risk_label"
                ]
            ),

            "event_time_s": (
                float(
                    metadata_row[
                        "event_time_s"
                    ]
                )
                if pd.notna(
                    metadata_row[
                        "event_time_s"
                    ]
                )
                else np.nan
            ),

            "precursor_start_s": (
                float(
                    metadata_row[
                        "precursor_start_s"
                    ]
                )
                if pd.notna(
                    metadata_row[
                        "precursor_start_s"
                    ]
                )
                else np.nan
            ),

            "precursor_end_s": (
                float(
                    metadata_row[
                        "precursor_end_s"
                    ]
                )
                if pd.notna(
                    metadata_row[
                        "precursor_end_s"
                    ]
                )
                else np.nan
            ),

            "true_label": int(
                prediction_row[
                    "true_label"
                ]
            ),

            "probability": float(
                prediction_row[
                    "probability"
                ]
            ),

            "prediction": int(
                prediction_row[
                    "prediction"
                ]
            )
        })


    print(
        f"PASS: Infant {test_infant} "
        f"test sequence ordering verified."
    )


# ------------------------------------------------------------
# Create mapped prediction table
# ------------------------------------------------------------

mapped_prediction_df = pd.DataFrame(
    mapped_prediction_rows
)

assert len(
    mapped_prediction_df
) == 55


# ------------------------------------------------------------
# Verify label consistency
# ------------------------------------------------------------

assert np.array_equal(
    mapped_prediction_df[
        "risk_label"
    ].to_numpy(),

    mapped_prediction_df[
        "true_label"
    ].to_numpy()
)


# ------------------------------------------------------------
# Extract positive precursor windows
# ------------------------------------------------------------

positive_window_analysis = (
    mapped_prediction_df[
        mapped_prediction_df[
            "risk_label"
        ] == 1
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(
    positive_window_analysis
) == 5


# ------------------------------------------------------------
# Assign TP / FN status
# ------------------------------------------------------------

positive_window_analysis[
    "status"
] = np.where(
    positive_window_analysis[
        "prediction"
    ] == 1,
    "TP",
    "FN"
)


# ------------------------------------------------------------
# Display exact positive-window results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("POSITIVE PRECURSOR WINDOW GRU ANALYSIS")
print("=" * 70)

display(
    positive_window_analysis[
        [
            "fold",
            "infant",
            "event_number",
            "event_time_s",
            "precursor_start_s",
            "precursor_end_s",
            "probability",
            "prediction",
            "status"
        ]
    ]
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

tp_count = int(
    (
        positive_window_analysis[
            "status"
        ] == "TP"
    ).sum()
)

fn_count = int(
    (
        positive_window_analysis[
            "status"
        ] == "FN"
    ).sum()
)

print(
    "\nPositive precursor windows:",
    len(positive_window_analysis)
)

print(
    "Detected (TP):",
    tp_count
)

print(
    "Missed (FN):",
    fn_count
)

print(
    "Positive-window sensitivity:",
    f"{tp_count / 5:.4f}"
)


# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert tp_count == 3
assert fn_count == 2

assert set(
    positive_window_analysis[
        "infant"
    ]
) == {1, 3, 9}

assert (
    positive_window_analysis[
        "event_number"
    ].notna().all()
)

print(
    "\nPASS: All 10 LOSO test folds verified."
)

print(
    "PASS: All 55 predictions mapped safely."
)

print(
    "PASS: All 5 positive precursor windows identified."
)

print(
    "PASS: 3 positive windows detected and "
    "2 positive windows missed."
)

print(
    "STATUS: Development / validation — "
    "not clinical validation."
)

Authoritative temporal dataset: (55, 11, 20)
Authoritative labels: (55,)
Metadata rows: 55
PASS: Infant 1 test sequence ordering verified.
PASS: Infant 2 test sequence ordering verified.
PASS: Infant 3 test sequence ordering verified.
PASS: Infant 4 test sequence ordering verified.
PASS: Infant 5 test sequence ordering verified.
PASS: Infant 6 test sequence ordering verified.
PASS: Infant 7 test sequence ordering verified.
PASS: Infant 8 test sequence ordering verified.
PASS: Infant 9 test sequence ordering verified.
PASS: Infant 10 test sequence ordering verified.

POSITIVE PRECURSOR WINDOW GRU ANALYSIS


,fold,infant,event_number,event_time_s,precursor_start_s,precursor_end_s,probability,prediction,status
0,1,1,2.0,3456.512,3441.512,3456.512,0.466902,0,FN
1,3,3,2.0,8143.846,8128.846,8143.846,0.878358,1,TP
2,3,3,3.0,9427.300,9412.300,9427.300,0.583672,1,TP
3,9,9,1.0,6900.050,6885.050,6900.050,0.532019,1,TP
4,9,9,2.0,10072.600,10057.600,10072.600,0.489570,0,FN



Positive precursor windows: 5
Detected (TP): 3
Missed (FN): 2
Positive-window sensitivity: 0.6000

PASS: All 10 LOSO test folds verified.
PASS: All 55 predictions mapped safely.
PASS: All 5 positive precursor windows identified.
PASS: 3 positive windows detected and 2 positive windows missed.
STATUS: Development / validation — not clinical validation.


In [ ]:
# ============================================================
# CELL 38 — Save Positive-Window GRU Analysis
#
# No model training is performed.
# ============================================================

GRU_POSITIVE_WINDOW_PATH = (
    REPORTS_DIR /
    "pics_temporal_loso_gru_positive_window_analysis.csv"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

positive_window_analysis.to_csv(
    GRU_POSITIVE_WINDOW_PATH,
    index=False
)

# ------------------------------------------------------------
# Reload
# ------------------------------------------------------------

positive_window_check = pd.read_csv(
    GRU_POSITIVE_WINDOW_PATH
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert len(
    positive_window_check
) == 5

assert (
    positive_window_check[
        "status"
    ].value_counts().get("TP", 0)
    == 3
)

assert (
    positive_window_check[
        "status"
    ].value_counts().get("FN", 0)
    == 2
)

assert set(
    positive_window_check[
        "infant"
    ]
) == {1, 3, 9}

assert (
    positive_window_check[
        "probability"
    ].between(0, 1).all()
)

assert (
    positive_window_check[
        "event_number"
    ].notna().all()
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 70)
print("POSITIVE-WINDOW GRU ANALYSIS SAVED")
print("=" * 70)

print(
    "\nFile:",
    GRU_POSITIVE_WINDOW_PATH
)

print(
    "Rows saved:",
    len(positive_window_check)
)

print(
    "TP windows:",
    int(
        (
            positive_window_check[
                "status"
            ] == "TP"
        ).sum()
    )
)

print(
    "FN windows:",
    int(
        (
            positive_window_check[
                "status"
            ] == "FN"
        ).sum()
    )
)

print(
    "\nPASS: Positive-window analysis saved and verified."
)

print(
    "STATUS: Development / validation — "
    "not clinical validation."
)

In [83]:
# ============================================================
# CELL 37A — Inspect GRU LOSO Fold Structure
#
# Purpose:
#   Determine exactly what metadata/index information is
#   available inside gru_loso_folds.
#
# No model training.
# ============================================================

print("Number of GRU LOSO folds:", len(gru_loso_folds))

print("\nKeys in first fold:")
print(
    list(gru_loso_folds[0].keys())
)

print("\nFirst fold test infant:")
print(
    gru_loso_folds[0]["test_infant"]
)

print("\nFirst fold X_test shape:")
print(
    gru_loso_folds[0]["X_test"].shape
)

print("\nFirst fold y_test:")
print(
    gru_loso_folds[0]["y_test"]
)

print("\nFirst fold train infants:")
print(
    gru_loso_folds[0]["train_infants"]
)

Number of GRU LOSO folds: 10

Keys in first fold:
['test_infant', 'train_infants', 'X_train', 'X_test', 'y_train', 'y_test', 'imputer', 'scaler']

First fold test infant:
1

First fold X_test shape:
(6, 11, 20)

First fold y_test:
[1 0 0 0 0 0]

First fold train infants:
[ 2  3  4  5  6  7  8  9 10]


In [93]:
# ============================================================
# CELL 38 — Save Positive-Window GRU Analysis
#
# No model training is performed.
# ============================================================

GRU_POSITIVE_WINDOW_PATH = (
    REPORTS_DIR /
    "pics_temporal_loso_gru_positive_window_analysis.csv"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

positive_window_analysis.to_csv(
    GRU_POSITIVE_WINDOW_PATH,
    index=False
)

# ------------------------------------------------------------
# Reload
# ------------------------------------------------------------

positive_window_check = pd.read_csv(
    GRU_POSITIVE_WINDOW_PATH
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert len(
    positive_window_check
) == 5

assert (
    positive_window_check[
        "status"
    ].value_counts().get("TP", 0)
    == 3
)

assert (
    positive_window_check[
        "status"
    ].value_counts().get("FN", 0)
    == 2
)

assert set(
    positive_window_check[
        "infant"
    ]
) == {1, 3, 9}

assert (
    positive_window_check[
        "probability"
    ].between(0, 1).all()
)

assert (
    positive_window_check[
        "event_number"
    ].notna().all()
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 70)
print("POSITIVE-WINDOW GRU ANALYSIS SAVED")
print("=" * 70)

print(
    "\nFile:",
    GRU_POSITIVE_WINDOW_PATH
)

print(
    "Rows saved:",
    len(positive_window_check)
)

print(
    "TP windows:",
    int(
        (
            positive_window_check[
                "status"
            ] == "TP"
        ).sum()
    )
)

print(
    "FN windows:",
    int(
        (
            positive_window_check[
                "status"
            ] == "FN"
        ).sum()
    )
)

print(
    "\nPASS: Positive-window analysis saved and verified."
)

print(
    "STATUS: Development / validation — "
    "not clinical validation."
)

POSITIVE-WINDOW GRU ANALYSIS SAVED

File: c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_temporal_loso_gru_positive_window_analysis.csv
Rows saved: 5
TP windows: 3
FN windows: 2

PASS: Positive-window analysis saved and verified.
STATUS: Development / validation — not clinical validation.


In [94]:
# ============================================================
# CELL 39 — Stage 4 Temporal Modeling Checkpoint
#
# Purpose:
#   Consolidate the validated temporal-modeling results.
#
# No model training is performed.
# ============================================================

STAGE4_TEMPORAL_CHECKPOINT_PATH = (
    REPORTS_DIR /
    "pics_stage4_temporal_modeling_checkpoint.json"
)

checkpoint = {
    "stage": "Stage 4 — Temporal Precursor Risk Modeling",
    "status": "development / validation",
    "clinical_validation": False,

    "dataset": {
        "total_windows": 55,
        "control_windows": 50,
        "precursor_windows": 5,
        "positive_bearing_infants": [1, 3, 9],
        "number_of_infants": 10,
        "sequence_shape": [11, 20],
        "features_per_timestep": 20,
        "timesteps": 11
    },

    "validation": {
        "method": "Leave-One-Subject-Out",
        "folds": 10,
        "subject_leakage_check": "PASS",
        "preprocessing": (
            "Imputation and scaling fitted on training "
            "infants only"
        )
    },

    "temporal_logistic": {
        "accuracy": 0.8909,
        "sensitivity": 0.4000,
        "specificity": 0.9400,
        "precision": 0.4000,
        "f1": 0.4000,
        "auroc": 0.8240,
        "auprc": 0.3049,
        "confusion_matrix": [
            [47, 3],
            [3, 2]
        ]
    },

    "gru": {
        "architecture": {
            "input_shape": [11, 20],
            "gru_units": 16,
            "dropout": 0.20,
            "dense_units": 8,
            "output": "sigmoid"
        },

        "training": {
            "epochs": 25,
            "batch_size": 8,
            "learning_rate": 0.0005,
            "threshold": 0.50,
            "early_stopping": False,
            "validation_split": False,
            "class_weight": "balanced",
            "fresh_model_per_fold": True
        },

        "outer_loso_metrics": {
            "accuracy": 0.6000,
            "sensitivity": 0.6000,
            "specificity": 0.6000,
            "precision": 0.13043478,
            "f1": 0.21428571,
            "auroc": 0.7160,
            "auprc": 0.32918741,
            "confusion_matrix": [
                [30, 20],
                [2, 3]
            ]
        },

        "positive_window_analysis": {
            "total_positive_windows": 5,
            "detected_tp": 3,
            "missed_fn": 2,
            "positive_window_sensitivity": 0.60
        }
    },

    "artifacts": [
        "pics_final_temporal_X_sequences.npy",
        "pics_final_temporal_y_sequences.npy",
        "pics_final_temporal_sequence_metadata.csv",
        "pics_temporal_loso_logistic_predictions.csv",
        "pics_temporal_loso_logistic_metrics.csv",
        "pics_temporal_loso_gru_predictions.csv",
        "pics_temporal_loso_gru_fold_summary.csv",
        "pics_temporal_loso_gru_metrics.csv",
        "pics_temporal_loso_gru_per_infant.csv",
        "pics_temporal_loso_gru_positive_window_analysis.csv"
    ]
}

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

with open(
    STAGE4_TEMPORAL_CHECKPOINT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        checkpoint,
        f,
        indent=4
    )

# ------------------------------------------------------------
# Reload
# ------------------------------------------------------------

with open(
    STAGE4_TEMPORAL_CHECKPOINT_PATH,
    "r",
    encoding="utf-8"
) as f:

    checkpoint_check = json.load(f)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert (
    checkpoint_check["dataset"]["total_windows"]
    == 55
)

assert (
    checkpoint_check["dataset"]["precursor_windows"]
    == 5
)

assert (
    checkpoint_check["validation"]["folds"]
    == 10
)

assert (
    checkpoint_check["gru"]["outer_loso_metrics"]["auroc"]
    == 0.7160
)

assert (
    checkpoint_check["gru"]["positive_window_analysis"][
        "detected_tp"
    ]
    == 3
)

assert (
    checkpoint_check["gru"]["positive_window_analysis"][
        "missed_fn"
    ]
    == 2
)

print("=" * 70)
print("STAGE 4 TEMPORAL MODELING CHECKPOINT SAVED")
print("=" * 70)

print(
    "\nFile:",
    STAGE4_TEMPORAL_CHECKPOINT_PATH
)

print(
    "\nTemporal windows:",
    checkpoint_check["dataset"]["total_windows"]
)

print(
    "LOSO folds:",
    checkpoint_check["validation"]["folds"]
)

print(
    "GRU AUROC:",
    checkpoint_check["gru"]["outer_loso_metrics"]["auroc"]
)

print(
    "GRU AUPRC:",
    checkpoint_check["gru"]["outer_loso_metrics"]["auprc"]
)

print(
    "Positive windows detected:",
    checkpoint_check["gru"][
        "positive_window_analysis"
    ]["detected_tp"]
)

print(
    "Positive windows missed:",
    checkpoint_check["gru"][
        "positive_window_analysis"
    ]["missed_fn"]
)

print(
    "\nPASS: Stage 4 temporal checkpoint saved and verified."
)

print(
    "STATUS: Development / validation — not clinical validation."
)

STAGE 4 TEMPORAL MODELING CHECKPOINT SAVED

File: c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_stage4_temporal_modeling_checkpoint.json

Temporal windows: 55
LOSO folds: 10
GRU AUROC: 0.716
GRU AUPRC: 0.32918741
Positive windows detected: 3
Positive windows missed: 2

PASS: Stage 4 temporal checkpoint saved and verified.
STATUS: Development / validation — not clinical validation.


In [95]:
# ============================================================
# CELL 39 — Stage 4 Temporal Modeling Checkpoint
#
# Purpose:
#   Consolidate the validated temporal-modeling results.
#
# No model training is performed.
# ============================================================

STAGE4_TEMPORAL_CHECKPOINT_PATH = (
    REPORTS_DIR /
    "pics_stage4_temporal_modeling_checkpoint.json"
)

checkpoint = {
    "stage": "Stage 4 — Temporal Precursor Risk Modeling",
    "status": "development / validation",
    "clinical_validation": False,

    "dataset": {
        "total_windows": 55,
        "control_windows": 50,
        "precursor_windows": 5,
        "positive_bearing_infants": [1, 3, 9],
        "number_of_infants": 10,
        "sequence_shape": [11, 20],
        "features_per_timestep": 20,
        "timesteps": 11
    },

    "validation": {
        "method": "Leave-One-Subject-Out",
        "folds": 10,
        "subject_leakage_check": "PASS",
        "preprocessing": (
            "Imputation and scaling fitted on training "
            "infants only"
        )
    },

    "temporal_logistic": {
        "accuracy": 0.8909,
        "sensitivity": 0.4000,
        "specificity": 0.9400,
        "precision": 0.4000,
        "f1": 0.4000,
        "auroc": 0.8240,
        "auprc": 0.3049,
        "confusion_matrix": [
            [47, 3],
            [3, 2]
        ]
    },

    "gru": {
        "architecture": {
            "input_shape": [11, 20],
            "gru_units": 16,
            "dropout": 0.20,
            "dense_units": 8,
            "output": "sigmoid"
        },

        "training": {
            "epochs": 25,
            "batch_size": 8,
            "learning_rate": 0.0005,
            "threshold": 0.50,
            "early_stopping": False,
            "validation_split": False,
            "class_weight": "balanced",
            "fresh_model_per_fold": True
        },

        "outer_loso_metrics": {
            "accuracy": 0.6000,
            "sensitivity": 0.6000,
            "specificity": 0.6000,
            "precision": 0.13043478,
            "f1": 0.21428571,
            "auroc": 0.7160,
            "auprc": 0.32918741,
            "confusion_matrix": [
                [30, 20],
                [2, 3]
            ]
        },

        "positive_window_analysis": {
            "total_positive_windows": 5,
            "detected_tp": 3,
            "missed_fn": 2,
            "positive_window_sensitivity": 0.60
        }
    },

    "artifacts": [
        "pics_final_temporal_X_sequences.npy",
        "pics_final_temporal_y_sequences.npy",
        "pics_final_temporal_sequence_metadata.csv",
        "pics_temporal_loso_logistic_predictions.csv",
        "pics_temporal_loso_logistic_metrics.csv",
        "pics_temporal_loso_gru_predictions.csv",
        "pics_temporal_loso_gru_fold_summary.csv",
        "pics_temporal_loso_gru_metrics.csv",
        "pics_temporal_loso_gru_per_infant.csv",
        "pics_temporal_loso_gru_positive_window_analysis.csv"
    ]
}

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

with open(
    STAGE4_TEMPORAL_CHECKPOINT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        checkpoint,
        f,
        indent=4
    )

# ------------------------------------------------------------
# Reload
# ------------------------------------------------------------

with open(
    STAGE4_TEMPORAL_CHECKPOINT_PATH,
    "r",
    encoding="utf-8"
) as f:

    checkpoint_check = json.load(f)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert (
    checkpoint_check["dataset"]["total_windows"]
    == 55
)

assert (
    checkpoint_check["dataset"]["precursor_windows"]
    == 5
)

assert (
    checkpoint_check["validation"]["folds"]
    == 10
)

assert (
    checkpoint_check["gru"]["outer_loso_metrics"]["auroc"]
    == 0.7160
)

assert (
    checkpoint_check["gru"]["positive_window_analysis"][
        "detected_tp"
    ]
    == 3
)

assert (
    checkpoint_check["gru"]["positive_window_analysis"][
        "missed_fn"
    ]
    == 2
)

print("=" * 70)
print("STAGE 4 TEMPORAL MODELING CHECKPOINT SAVED")
print("=" * 70)

print(
    "\nFile:",
    STAGE4_TEMPORAL_CHECKPOINT_PATH
)

print(
    "\nTemporal windows:",
    checkpoint_check["dataset"]["total_windows"]
)

print(
    "LOSO folds:",
    checkpoint_check["validation"]["folds"]
)

print(
    "GRU AUROC:",
    checkpoint_check["gru"]["outer_loso_metrics"]["auroc"]
)

print(
    "GRU AUPRC:",
    checkpoint_check["gru"]["outer_loso_metrics"]["auprc"]
)

print(
    "Positive windows detected:",
    checkpoint_check["gru"][
        "positive_window_analysis"
    ]["detected_tp"]
)

print(
    "Positive windows missed:",
    checkpoint_check["gru"][
        "positive_window_analysis"
    ]["missed_fn"]
)

print(
    "\nPASS: Stage 4 temporal checkpoint saved and verified."
)

print(
    "STATUS: Development / validation — not clinical validation."
)

STAGE 4 TEMPORAL MODELING CHECKPOINT SAVED

File: c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_stage4_temporal_modeling_checkpoint.json

Temporal windows: 55
LOSO folds: 10
GRU AUROC: 0.716
GRU AUPRC: 0.32918741
Positive windows detected: 3
Positive windows missed: 2

PASS: Stage 4 temporal checkpoint saved and verified.
STATUS: Development / validation — not clinical validation.


In [96]:
# ============================================================
# CELL 40 — Freeze 1D-CNN Temporal Model Configuration
#
# Purpose:
#   Define the 1D-CNN architecture and training configuration
#   before running outer LOSO evaluation.
#
# No model training is performed.
# ============================================================

CNN_FIXED_EPOCHS = 25
CNN_BATCH_SIZE = 8
CNN_THRESHOLD = 0.50
CNN_LEARNING_RATE = 0.0005

CNN_FILTERS_1 = 16
CNN_FILTERS_2 = 32
CNN_KERNEL_SIZE = 3
CNN_DROPOUT = 0.20
CNN_DENSE_UNITS = 8

print("=" * 70)
print("1D-CNN TEMPORAL MODEL CONFIGURATION")
print("=" * 70)

print("\nInput shape:")
print("  Timesteps :", 11)
print("  Features  :", 20)

print("\nArchitecture:")
print("  Conv1D    :", CNN_FILTERS_1, "filters")
print("  Conv1D    :", CNN_FILTERS_2, "filters")
print("  Kernel    :", CNN_KERNEL_SIZE)
print("  Dropout   :", CNN_DROPOUT)
print("  Dense     :", CNN_DENSE_UNITS)
print("  Output    : sigmoid")

print("\nTraining:")
print("  Epochs    :", CNN_FIXED_EPOCHS)
print("  Batch size:", CNN_BATCH_SIZE)
print("  Learning rate:", CNN_LEARNING_RATE)
print("  Threshold :", CNN_THRESHOLD)
print("  Class weighting: balanced")

print("\nLOSO policy:")
print("  Outer test infant excluded from training")
print("  Cell-30 preprocessing reused exactly once")
print("  No validation split")
print("  No early stopping")
print("  Fresh model for every LOSO fold")

print(
    "\nSTATUS: Development / validation — "
    "not clinical validation."
)

print("\nPASS: 1D-CNN configuration frozen.")

1D-CNN TEMPORAL MODEL CONFIGURATION

Input shape:
  Timesteps : 11
  Features  : 20

Architecture:
  Conv1D    : 16 filters
  Conv1D    : 32 filters
  Kernel    : 3
  Dropout   : 0.2
  Dense     : 8
  Output    : sigmoid

Training:
  Epochs    : 25
  Batch size: 8
  Learning rate: 0.0005
  Threshold : 0.5
  Class weighting: balanced

LOSO policy:
  Outer test infant excluded from training
  Cell-30 preprocessing reused exactly once
  No validation split
  No early stopping
  Fresh model for every LOSO fold

STATUS: Development / validation — not clinical validation.

PASS: 1D-CNN configuration frozen.


In [97]:
# ============================================================
# CELL 41 — 1D-CNN Outer LOSO Training
#
# Purpose:
#   Train and evaluate the 1D-CNN using outer
#   Leave-One-Subject-Out validation.
#
# IMPORTANT:
#   Cell 30 already performed:
#       - training-fold-only imputation
#       - training-fold-only scaling
#
# Therefore this cell does NOT preprocess X_train/X_test again.
#
# Frozen configuration from Cell 40:
#   Epochs        = 25
#   Batch size    = 8
#   Learning rate = 0.0005
#   Filters       = 16, 32
#   Kernel size   = 3
#   Dropout       = 0.20
#   Dense units   = 8
#   Threshold     = 0.50
#   No validation split
#   No early stopping
#
# Status:
#   Development / validation
# ============================================================

import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

# ------------------------------------------------------------
# Verify required configuration
# ------------------------------------------------------------

assert "gru_loso_folds" in globals(), \
    "gru_loso_folds is not available. Run Cell 30 first."

assert len(gru_loso_folds) == 10

assert "CNN_FIXED_EPOCHS" in globals()
assert "CNN_BATCH_SIZE" in globals()
assert "CNN_THRESHOLD" in globals()
assert "CNN_LEARNING_RATE" in globals()
assert "CNN_FILTERS_1" in globals()
assert "CNN_FILTERS_2" in globals()
assert "CNN_KERNEL_SIZE" in globals()
assert "CNN_DROPOUT" in globals()
assert "CNN_DENSE_UNITS" in globals()


# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

cnn_loso_predictions = []
cnn_fold_summaries = []


# ------------------------------------------------------------
# Outer LOSO loop
# ------------------------------------------------------------

for fold_number, fold in enumerate(
    gru_loso_folds,
    start=1
):

    test_infant = int(
        fold["test_infant"]
    )

    # --------------------------------------------------------
    # Cell-30 processed data
    # --------------------------------------------------------

    X_train = fold["X_train"].copy()
    y_train = fold["y_train"].copy()

    X_test = fold["X_test"].copy()
    y_test = fold["y_test"].copy()

    train_infants = fold["train_infants"].copy()

    print("\n" + "=" * 70)
    print(
        f"1D-CNN LOSO FOLD "
        f"{fold_number}/10 — Test Infant {test_infant}"
    )
    print("=" * 70)

    print(
        "Training samples:",
        len(X_train),
        "| positives:",
        int(y_train.sum()),
        "| negatives:",
        int((y_train == 0).sum())
    )

    print(
        "Test samples:",
        len(X_test),
        "| positives:",
        int(y_test.sum()),
        "| negatives:",
        int((y_test == 0).sum())
    )

    print(
        "Input shape:",
        X_train.shape
    )

    # --------------------------------------------------------
    # Leakage check
    # --------------------------------------------------------

    assert test_infant not in set(
        train_infants.tolist()
    ), (
        f"Test Infant {test_infant} "
        "is present in training data."
    )

    # --------------------------------------------------------
    # Confirm Cell-30 preprocessing produced finite data
    # --------------------------------------------------------

    assert np.isfinite(
        X_train
    ).all(), (
        f"Non-finite training data "
        f"for Infant {test_infant}."
    )

    assert np.isfinite(
        X_test
    ).all(), (
        f"Non-finite test data "
        f"for Infant {test_infant}."
    )

    # --------------------------------------------------------
    # Training-fold-only class weights
    # --------------------------------------------------------

    classes = np.unique(
        y_train
    )

    class_weights_array = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    class_weights = {
        int(cls): float(weight)
        for cls, weight in zip(
            classes,
            class_weights_array
        )
    }

    print(
        "Class weights:",
        class_weights
    )

    # --------------------------------------------------------
    # Fresh CNN model for this fold
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    tf.keras.utils.set_random_seed(
        RANDOM_SEED + test_infant
    )

    n_steps = X_train.shape[1]
    n_features = X_train.shape[2]

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(n_steps, n_features)
        ),

        tf.keras.layers.Conv1D(
            filters=CNN_FILTERS_1,
            kernel_size=CNN_KERNEL_SIZE,
            activation="relu",
            padding="same"
        ),

        tf.keras.layers.Conv1D(
            filters=CNN_FILTERS_2,
            kernel_size=CNN_KERNEL_SIZE,
            activation="relu",
            padding="same"
        ),

        tf.keras.layers.GlobalAveragePooling1D(),

        tf.keras.layers.Dropout(
            CNN_DROPOUT
        ),

        tf.keras.layers.Dense(
            CNN_DENSE_UNITS,
            activation="relu"
        ),

        tf.keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=CNN_LEARNING_RATE
        ),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(
                name="auc"
            )
        ]
    )

    # --------------------------------------------------------
    # Fixed 25-epoch training
    #
    # NO validation split.
    # NO early stopping.
    # --------------------------------------------------------

    history = model.fit(
        X_train,
        y_train,
        epochs=CNN_FIXED_EPOCHS,
        batch_size=CNN_BATCH_SIZE,
        class_weight=class_weights,
        shuffle=True,
        verbose=0
    )

    # --------------------------------------------------------
    # Predict held-out infant
    # --------------------------------------------------------

    test_probabilities = model.predict(
        X_test,
        verbose=0
    ).ravel()

    test_predictions = (
        test_probabilities >= CNN_THRESHOLD
    ).astype(int)

    # --------------------------------------------------------
    # Store OOF predictions
    # --------------------------------------------------------

    for i in range(len(y_test)):

        cnn_loso_predictions.append({
            "fold": fold_number,
            "test_infant": test_infant,
            "true_label": int(
                y_test[i]
            ),
            "probability": float(
                test_probabilities[i]
            ),
            "prediction": int(
                test_predictions[i]
            )
        })

    # --------------------------------------------------------
    # Fold summary
    # --------------------------------------------------------

    cnn_fold_summaries.append({
        "fold": fold_number,
        "test_infant": test_infant,
        "train_samples": int(
            len(y_train)
        ),
        "train_positive": int(
            y_train.sum()
        ),
        "train_negative": int(
            (y_train == 0).sum()
        ),
        "test_samples": int(
            len(y_test)
        ),
        "test_positive": int(
            y_test.sum()
        ),
        "test_negative": int(
            (y_test == 0).sum()
        ),
        "epochs": int(
            CNN_FIXED_EPOCHS
        )
    })

    print(
        f"Fold {fold_number} complete."
    )

    print(
        "Test probabilities:",
        np.round(
            test_probabilities,
            4
        )
    )

    print(
        "Test predictions:",
        test_predictions.tolist()
    )


# ------------------------------------------------------------
# Convert outputs to DataFrames
# ------------------------------------------------------------

cnn_loso_predictions = pd.DataFrame(
    cnn_loso_predictions
)

cnn_fold_summaries = pd.DataFrame(
    cnn_fold_summaries
)


# ------------------------------------------------------------
# Final structural checks
# ------------------------------------------------------------

assert len(
    cnn_loso_predictions
) == 55

assert len(
    cnn_fold_summaries
) == 10

assert (
    cnn_loso_predictions[
        "probability"
    ].between(0, 1).all()
)

assert (
    cnn_loso_predictions[
        "prediction"
    ].isin([0, 1]).all()
)

assert (
    cnn_loso_predictions[
        "true_label"
    ].isin([0, 1]).all()
)

assert not (
    cnn_loso_predictions.isna().any().any()
)

assert (
    cnn_loso_predictions[
        "true_label"
    ].sum()
    == 5
)

assert (
    (
        cnn_loso_predictions[
            "true_label"
        ] == 0
    ).sum()
    == 50
)


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("1D-CNN OUTER LOSO TRAINING COMPLETE")
print("=" * 70)

print(
    "Total OOF predictions:",
    len(cnn_loso_predictions)
)

print(
    "True positives:",
    int(
        cnn_loso_predictions[
            "true_label"
        ].sum()
    )
)

print(
    "True negatives:",
    int(
        (
            cnn_loso_predictions[
                "true_label"
            ] == 0
        ).sum()
    )
)

print(
    "Predicted positives:",
    int(
        cnn_loso_predictions[
            "prediction"
        ].sum()
    )
)

print(
    "Predicted negatives:",
    int(
        (
            cnn_loso_predictions[
                "prediction"
            ] == 0
        ).sum()
    )
)

print("\nFold summary:")
display(
    cnn_fold_summaries
)

print(
    "\nPASS: 10 outer-LOSO CNN folds completed."
)

print(
    "PASS: 55 out-of-fold predictions generated."
)

print(
    "PASS: Cell-30 preprocessing used exactly once."
)

print(
    "PASS: No second imputation/scaling applied."
)

print(
    "PASS: Class weights calculated from training data only."
)

print(
    "STATUS: Development / validation — "
    "not clinical validation."
)


1D-CNN LOSO FOLD 1/10 — Test Infant 1
Training samples: 49 | positives: 4 | negatives: 45
Test samples: 6 | positives: 1 | negatives: 5
Input shape: (49, 11, 20)
Class weights: {0: 0.5444444444444444, 1: 6.125}
Fold 1 complete.
Test probabilities: [0.0605 0.0536 0.0131 0.0366 0.0184 0.0326]
Test predictions: [0, 0, 0, 0, 0, 0]

1D-CNN LOSO FOLD 2/10 — Test Infant 2
Training samples: 50 | positives: 5 | negatives: 45
Test samples: 5 | positives: 0 | negatives: 5
Input shape: (50, 11, 20)
Class weights: {0: 0.5555555555555556, 1: 5.0}
Fold 2 complete.
Test probabilities: [0.8137 0.6718 0.1505 0.3411 0.0616]
Test predictions: [1, 1, 0, 0, 0]

1D-CNN LOSO FOLD 3/10 — Test Infant 3
Training samples: 48 | positives: 3 | negatives: 45
Test samples: 7 | positives: 2 | negatives: 5
Input shape: (48, 11, 20)
Class weights: {0: 0.5333333333333333, 1: 8.0}
Fold 3 complete.
Test probabilities: [0.4966 0.1637 0.0327 0.4521 0.3265 0.2957 0.0775]
Test predictions: [0, 0, 0, 0, 0, 0, 0]

1D-CNN LOSO F

,fold,test_infant,train_samples,train_positive,train_negative,test_samples,test_positive,test_negative,epochs
0,1,1,49,4,45,6,1,5,25
1,2,2,50,5,45,5,0,5,25
2,3,3,48,3,45,7,2,5,25
3,4,4,50,5,45,5,0,5,25
4,5,5,50,5,45,5,0,5,25
5,6,6,50,5,45,5,0,5,25
6,7,7,50,5,45,5,0,5,25
7,8,8,50,5,45,5,0,5,25
8,9,9,48,3,45,7,2,5,25
9,10,10,50,5,45,5,0,5,25



PASS: 10 outer-LOSO CNN folds completed.
PASS: 55 out-of-fold predictions generated.
PASS: Cell-30 preprocessing used exactly once.
PASS: No second imputation/scaling applied.
PASS: Class weights calculated from training data only.
STATUS: Development / validation — not clinical validation.


In [98]:
# ============================================================
# CELL 42 — 1D-CNN Pooled LOSO Metrics
#
# Purpose:
#   Calculate pooled out-of-fold performance for the
#   1D-CNN using all 55 LOSO predictions.
#
# Metrics:
#   Accuracy
#   Sensitivity / Recall
#   Specificity
#   Precision
#   F1-score
#   AUROC
#   AUPRC
#   Confusion matrix
#
# Status:
#   Development / validation
# ============================================================

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# Extract pooled OOF results
# ------------------------------------------------------------

y_true_cnn = cnn_loso_predictions[
    "true_label"
].to_numpy()

y_pred_cnn = cnn_loso_predictions[
    "prediction"
].to_numpy()

y_prob_cnn = cnn_loso_predictions[
    "probability"
].to_numpy()


# ------------------------------------------------------------
# Basic integrity checks
# ------------------------------------------------------------

assert len(y_true_cnn) == 55
assert len(y_pred_cnn) == 55
assert len(y_prob_cnn) == 55

assert np.isfinite(
    y_prob_cnn
).all()

assert set(
    np.unique(y_true_cnn)
).issubset({0, 1})

assert set(
    np.unique(y_pred_cnn)
).issubset({0, 1})


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm_cnn = confusion_matrix(
    y_true_cnn,
    y_pred_cnn,
    labels=[0, 1]
)

tn_cnn, fp_cnn, fn_cnn, tp_cnn = cm_cnn.ravel()


# ------------------------------------------------------------
# Classification metrics
# ------------------------------------------------------------

accuracy_cnn = accuracy_score(
    y_true_cnn,
    y_pred_cnn
)

sensitivity_cnn = recall_score(
    y_true_cnn,
    y_pred_cnn,
    zero_division=0
)

specificity_cnn = (
    tn_cnn / (tn_cnn + fp_cnn)
    if (tn_cnn + fp_cnn) > 0
    else np.nan
)

precision_cnn = precision_score(
    y_true_cnn,
    y_pred_cnn,
    zero_division=0
)

f1_cnn = f1_score(
    y_true_cnn,
    y_pred_cnn,
    zero_division=0
)


# ------------------------------------------------------------
# Probability-based metrics
# ------------------------------------------------------------

auroc_cnn = roc_auc_score(
    y_true_cnn,
    y_prob_cnn
)

auprc_cnn = average_precision_score(
    y_true_cnn,
    y_prob_cnn
)


# ------------------------------------------------------------
# Store metrics
# ------------------------------------------------------------

cnn_loso_metrics = pd.DataFrame([{
    "model": "1D-CNN",
    "validation": "10-fold LOSO",
    "samples": len(y_true_cnn),
    "positive_windows": int(
        y_true_cnn.sum()
    ),
    "negative_windows": int(
        (y_true_cnn == 0).sum()
    ),
    "TN": int(tn_cnn),
    "FP": int(fp_cnn),
    "FN": int(fn_cnn),
    "TP": int(tp_cnn),
    "accuracy": float(accuracy_cnn),
    "sensitivity": float(sensitivity_cnn),
    "specificity": float(specificity_cnn),
    "precision": float(precision_cnn),
    "f1": float(f1_cnn),
    "auroc": float(auroc_cnn),
    "auprc": float(auprc_cnn)
}])


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 70)
print("1D-CNN POOLED LOSO PERFORMANCE")
print("=" * 70)

print("\nConfusion Matrix:")
print(cm_cnn)

print("\nConfusion Matrix Details:")
print("  TN:", tn_cnn)
print("  FP:", fp_cnn)
print("  FN:", fn_cnn)
print("  TP:", tp_cnn)

print("\nPerformance:")
print(
    f"  Accuracy    : {accuracy_cnn:.4f}"
)
print(
    f"  Sensitivity : {sensitivity_cnn:.4f}"
)
print(
    f"  Specificity : {specificity_cnn:.4f}"
)
print(
    f"  Precision   : {precision_cnn:.4f}"
)
print(
    f"  F1-score    : {f1_cnn:.4f}"
)
print(
    f"  AUROC       : {auroc_cnn:.4f}"
)
print(
    f"  AUPRC       : {auprc_cnn:.4f}"
)

print("\nDataset:")
print(
    "  Total windows    :",
    len(y_true_cnn)
)
print(
    "  Positive windows :",
    int(y_true_cnn.sum())
)
print(
    "  Negative windows :",
    int((y_true_cnn == 0).sum())
)

print(
    "\nSTATUS: Development / validation — "
    "not clinical validation."
)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert (
    tn_cnn + fp_cnn + fn_cnn + tp_cnn
    == 55
)

assert (
    tp_cnn + fn_cnn
    == 5
)

assert (
    tn_cnn + fp_cnn
    == 50
)

assert 0.0 <= accuracy_cnn <= 1.0
assert 0.0 <= sensitivity_cnn <= 1.0
assert 0.0 <= specificity_cnn <= 1.0
assert 0.0 <= precision_cnn <= 1.0
assert 0.0 <= f1_cnn <= 1.0
assert 0.0 <= auroc_cnn <= 1.0
assert 0.0 <= auprc_cnn <= 1.0

print("\nPASS: 1D-CNN pooled LOSO metrics calculated.")

1D-CNN POOLED LOSO PERFORMANCE

Confusion Matrix:
[[45  5]
 [ 5  0]]

Confusion Matrix Details:
  TN: 45
  FP: 5
  FN: 5
  TP: 0

Performance:
  Accuracy    : 0.8182
  Sensitivity : 0.0000
  Specificity : 0.9000
  Precision   : 0.0000
  F1-score    : 0.0000
  AUROC       : 0.5880
  AUPRC       : 0.1389

Dataset:
  Total windows    : 55
  Positive windows : 5
  Negative windows : 50

STATUS: Development / validation — not clinical validation.

PASS: 1D-CNN pooled LOSO metrics calculated.


In [99]:
# ============================================================
# CELL 43 — 1D-CNN Per-Infant LOSO Analysis
#
# Purpose:
#   Analyze CNN predictions separately for each infant.
#
# No model training is performed.
# ============================================================

from sklearn.metrics import confusion_matrix

cnn_per_infant_rows = []

for infant_id in sorted(
    cnn_loso_predictions["test_infant"].unique()
):

    infant_data = cnn_loso_predictions[
        cnn_loso_predictions["test_infant"] == infant_id
    ].copy()

    y_true = infant_data[
        "true_label"
    ].to_numpy()

    y_pred = infant_data[
        "prediction"
    ].to_numpy()

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    positive_count = tp + fn
    negative_count = tn + fp

    sensitivity = (
        tp / positive_count
        if positive_count > 0
        else np.nan
    )

    specificity = (
        tn / negative_count
        if negative_count > 0
        else np.nan
    )

    cnn_per_infant_rows.append({
        "infant": int(infant_id),
        "windows": len(infant_data),
        "positive_windows": int(positive_count),
        "negative_windows": int(negative_count),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "sensitivity": sensitivity,
        "specificity": specificity
    })


# ------------------------------------------------------------
# Create result table
# ------------------------------------------------------------

cnn_per_infant = pd.DataFrame(
    cnn_per_infant_rows
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 70)
print("1D-CNN PER-INFANT LOSO ANALYSIS")
print("=" * 70)

display(
    cnn_per_infant
)


# ------------------------------------------------------------
# Summary checks
# ------------------------------------------------------------

print("\nPositive-bearing infants:")

positive_bearing = cnn_per_infant[
    cnn_per_infant["positive_windows"] > 0
]

display(
    positive_bearing
)


print(
    "\nTotal positive windows:",
    int(
        cnn_per_infant[
            "positive_windows"
        ].sum()
    )
)

print(
    "Total negative windows:",
    int(
        cnn_per_infant[
            "negative_windows"
        ].sum()
    )
)

print(
    "Total TP:",
    int(
        cnn_per_infant[
            "TP"
        ].sum()
    )
)

print(
    "Total FN:",
    int(
        cnn_per_infant[
            "FN"
        ].sum()
    )
)

print(
    "Total FP:",
    int(
        cnn_per_infant[
            "FP"
        ].sum()
    )
)

print(
    "Total TN:",
    int(
        cnn_per_infant[
            "TN"
        ].sum()
    )
)


# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(
    cnn_per_infant
) == 10

assert (
    cnn_per_infant[
        "positive_windows"
    ].sum()
    == 5
)

assert (
    cnn_per_infant[
        "negative_windows"
    ].sum()
    == 50
)

assert (
    cnn_per_infant[
        "TP"
    ].sum()
    == tp_cnn
)

assert (
    cnn_per_infant[
        "TN"
    ].sum()
    == tn_cnn
)

assert (
    cnn_per_infant[
        "FP"
    ].sum()
    == fp_cnn
)

assert (
    cnn_per_infant[
        "FN"
    ].sum()
    == fn_cnn
)

print(
    "\nPASS: Per-infant CNN analysis completed."
)

print(
    "STATUS: Development / validation — "
    "not clinical validation."
)

1D-CNN PER-INFANT LOSO ANALYSIS


,infant,windows,positive_windows,negative_windows,TN,FP,FN,TP,sensitivity,specificity
0,1,6,1,5,5,0,1,0,0.0,1.0
1,2,5,0,5,3,2,0,0,NaN,0.6
2,3,7,2,5,5,0,2,0,0.0,1.0
3,4,5,0,5,5,0,0,0,NaN,1.0
4,5,5,0,5,4,1,0,0,NaN,0.8
5,6,5,0,5,4,1,0,0,NaN,0.8
6,7,5,0,5,5,0,0,0,NaN,1.0
7,8,5,0,5,4,1,0,0,NaN,0.8
8,9,7,2,5,5,0,2,0,0.0,1.0
9,10,5,0,5,5,0,0,0,NaN,1.0



Positive-bearing infants:


,infant,windows,positive_windows,negative_windows,TN,FP,FN,TP,sensitivity,specificity
0,1,6,1,5,5,0,1,0,0.0,1.0
2,3,7,2,5,5,0,2,0,0.0,1.0
8,9,7,2,5,5,0,2,0,0.0,1.0



Total positive windows: 5
Total negative windows: 50
Total TP: 0
Total FN: 5
Total FP: 5
Total TN: 45

PASS: Per-infant CNN analysis completed.
STATUS: Development / validation — not clinical validation.


In [101]:
# ============================================================
# CELL 44 — Corrected 1D-CNN Positive-Window Analysis
#
# Purpose:
#   Map CNN predictions back to the exact five precursor
#   windows and verify their probabilities.
#
# IMPORTANT:
#   Cell 30 preprocessing is reproduced exactly:
#
#       (samples, 11, 20)
#              ↓
#       (samples*11, 20)
#              ↓
#       imputer
#              ↓
#       scaler
#              ↓
#       reshape back to (samples, 11, 20)
#
#   No model training is performed.
# ============================================================


# ------------------------------------------------------------
# Load authoritative temporal metadata
# ------------------------------------------------------------

temporal_metadata = pd.read_csv(
    REPORTS_DIR /
    "pics_final_temporal_sequence_metadata.csv"
)

X_temporal_raw = np.load(
    REPORTS_DIR /
    "pics_final_temporal_X_sequences.npy"
)

y_temporal_raw = np.load(
    REPORTS_DIR /
    "pics_final_temporal_y_sequences.npy"
)


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

assert len(temporal_metadata) == 55
assert X_temporal_raw.shape == (55, 11, 20)
assert y_temporal_raw.shape == (55,)

assert np.array_equal(
    temporal_metadata["risk_label"].to_numpy(),
    y_temporal_raw
)

print(
    "Authoritative temporal dataset:",
    X_temporal_raw.shape
)


# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

cnn_positive_rows = []


# ------------------------------------------------------------
# Process each LOSO test infant
# ------------------------------------------------------------

for fold_number, fold in enumerate(
    gru_loso_folds,
    start=1
):

    test_infant = int(
        fold["test_infant"]
    )

    print(
        f"\nChecking CNN fold {fold_number}/10 "
        f"— Test Infant {test_infant}"
    )

    # --------------------------------------------------------
    # Locate this infant in authoritative temporal dataset
    # --------------------------------------------------------

    infant_mask = (
        temporal_metadata["infant"]
        == test_infant
    )

    infant_indices = np.where(
        infant_mask.to_numpy()
    )[0]

    X_infant_raw = X_temporal_raw[
        infant_indices
    ]

    y_infant_raw = y_temporal_raw[
        infant_indices
    ]

    infant_metadata = temporal_metadata[
        infant_mask
    ].reset_index(drop=True)


    # --------------------------------------------------------
    # Reproduce Cell-30 preprocessing EXACTLY
    #
    # Cell 30 works per timestep:
    #
    # (samples, 11, 20)
    #       ↓
    # (samples*11, 20)
    # --------------------------------------------------------

    n_samples = X_infant_raw.shape[0]
    n_timesteps = X_infant_raw.shape[1]
    n_features = X_infant_raw.shape[2]

    X_flat = X_infant_raw.reshape(
        -1,
        n_features
    )

    X_imputed = fold["imputer"].transform(
        X_flat
    )

    X_scaled = fold["scaler"].transform(
        X_imputed
    )

    X_expected = X_scaled.reshape(
        n_samples,
        n_timesteps,
        n_features
    )


    # --------------------------------------------------------
    # Verify exact match with Cell-30 test data
    # --------------------------------------------------------

    assert X_expected.shape == fold[
        "X_test"
    ].shape

    assert np.allclose(
        X_expected,
        fold["X_test"],
        atol=1e-10
    )

    assert np.array_equal(
        y_infant_raw,
        fold["y_test"]
    )

    print(
        "  PASS: Cell-30 preprocessing reproduced exactly."
    )


    # --------------------------------------------------------
    # Get positive precursor windows
    # --------------------------------------------------------

    positive_mask = (
        infant_metadata["risk_label"]
        == 1
    )

    positive_indices = np.where(
        positive_mask.to_numpy()
    )[0]

    positive_metadata = infant_metadata[
        positive_mask
    ].reset_index(drop=True)

    # --------------------------------------------------------
    # Predictions generated by Cell 41
    # --------------------------------------------------------

    fold_predictions = cnn_loso_predictions[
        cnn_loso_predictions["fold"] == fold_number
    ].reset_index(drop=True)

    assert len(
        fold_predictions
    ) == len(
        fold["y_test"]
    )

    # --------------------------------------------------------
    # Verify positive positions against fold test labels
    # --------------------------------------------------------

    fold_positive_positions = np.where(
        fold["y_test"] == 1
    )[0]

    assert np.array_equal(
        positive_indices,
        fold_positive_positions
    )


    # --------------------------------------------------------
    # Map each positive window
    # --------------------------------------------------------

    for local_position in positive_indices:

        metadata_row = infant_metadata.iloc[
            local_position
        ]

        prediction_row = fold_predictions.iloc[
            local_position
        ]

        probability = float(
            prediction_row[
                "probability"
            ]
        )

        prediction = int(
            prediction_row[
                "prediction"
            ]
        )

        cnn_positive_rows.append({

            "fold": fold_number,

            "infant": int(
                metadata_row[
                    "infant"
                ]
            ),

            "event_number": int(
                metadata_row[
                    "event_number"
                ]
            ),

            "event_time_s": float(
                metadata_row[
                    "event_time_s"
                ]
            ),

            "precursor_start_s": float(
                metadata_row[
                    "precursor_start_s"
                ]
            ),

            "precursor_end_s": float(
                metadata_row[
                    "precursor_end_s"
                ]
            ),

            "probability": probability,

            "prediction": prediction,

            "result": (
                "TP"
                if prediction == 1
                else "FN"
            )
        })


# ------------------------------------------------------------
# Create final positive-window table
# ------------------------------------------------------------

cnn_positive_window_analysis = pd.DataFrame(
    cnn_positive_rows
).sort_values(
    ["infant", "event_number"]
).reset_index(drop=True)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("1D-CNN POSITIVE-WINDOW ANALYSIS")
print("=" * 70)

display(
    cnn_positive_window_analysis
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

tp_positive_cnn = int(
    (
        cnn_positive_window_analysis[
            "result"
        ] == "TP"
    ).sum()
)

fn_positive_cnn = int(
    (
        cnn_positive_window_analysis[
            "result"
        ] == "FN"
    ).sum()
)

positive_window_sensitivity_cnn = (
    tp_positive_cnn /
    len(cnn_positive_window_analysis)
)


print(
    "\nTotal precursor windows:",
    len(cnn_positive_window_analysis)
)

print(
    "Detected precursor windows (TP):",
    tp_positive_cnn
)

print(
    "Missed precursor windows (FN):",
    fn_positive_cnn
)

print(
    "Positive-window sensitivity:",
    f"{positive_window_sensitivity_cnn:.4f}"
)


# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(
    cnn_positive_window_analysis
) == 5

assert (
    tp_positive_cnn
    + fn_positive_cnn
    == 5
)

assert (
    cnn_positive_window_analysis[
        "infant"
    ].isin([1, 3, 9]).all()
)

assert (
    cnn_positive_window_analysis[
        "prediction"
    ].isin([0, 1]).all()
)

assert (
    cnn_positive_window_analysis[
        "probability"
    ].between(0, 1).all()
)

assert not (
    cnn_positive_window_analysis[
        "probability"
    ].isna().any()
)


print(
    "\nPASS: All 5 precursor windows mapped correctly."
)

print(
    "PASS: Cell-30 preprocessing/order independently verified."
)

print(
    "PASS: No model retraining performed."
)

print(
    "STATUS: Development / validation — "
    "not clinical validation."
)

Authoritative temporal dataset: (55, 11, 20)

Checking CNN fold 1/10 — Test Infant 1
  PASS: Cell-30 preprocessing reproduced exactly.

Checking CNN fold 2/10 — Test Infant 2
  PASS: Cell-30 preprocessing reproduced exactly.

Checking CNN fold 3/10 — Test Infant 3
  PASS: Cell-30 preprocessing reproduced exactly.

Checking CNN fold 4/10 — Test Infant 4
  PASS: Cell-30 preprocessing reproduced exactly.

Checking CNN fold 5/10 — Test Infant 5
  PASS: Cell-30 preprocessing reproduced exactly.

Checking CNN fold 6/10 — Test Infant 6
  PASS: Cell-30 preprocessing reproduced exactly.

Checking CNN fold 7/10 — Test Infant 7
  PASS: Cell-30 preprocessing reproduced exactly.

Checking CNN fold 8/10 — Test Infant 8
  PASS: Cell-30 preprocessing reproduced exactly.

Checking CNN fold 9/10 — Test Infant 9
  PASS: Cell-30 preprocessing reproduced exactly.

Checking CNN fold 10/10 — Test Infant 10
  PASS: Cell-30 preprocessing reproduced exactly.

1D-CNN POSITIVE-WINDOW ANALYSIS


,fold,infant,event_number,event_time_s,precursor_start_s,precursor_end_s,probability,prediction,result
0,1,1,2,3456.512,3441.512,3456.512,0.060492,0,FN
1,3,3,2,8143.846,8128.846,8143.846,0.496580,0,FN
2,3,3,3,9427.300,9412.300,9427.300,0.163691,0,FN
3,9,9,1,6900.050,6885.050,6900.050,0.360278,0,FN
4,9,9,2,10072.600,10057.600,10072.600,0.302852,0,FN



Total precursor windows: 5
Detected precursor windows (TP): 0
Missed precursor windows (FN): 5
Positive-window sensitivity: 0.0000

PASS: All 5 precursor windows mapped correctly.
PASS: Cell-30 preprocessing/order independently verified.
PASS: No model retraining performed.
STATUS: Development / validation — not clinical validation.


In [102]:
# ============================================================
# CELL 45 — Save 1D-CNN Results
#
# Purpose:
#   Save all authoritative 1D-CNN LOSO outputs:
#
#   1. OOF predictions
#   2. LOSO fold summary
#   3. Pooled metrics
#   4. Per-infant analysis
#   5. Positive-window analysis
#
# Status:
#   Development / validation
# ============================================================


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

cnn_predictions_path = (
    REPORTS_DIR /
    "pics_temporal_loso_cnn_predictions.csv"
)

cnn_fold_summary_path = (
    REPORTS_DIR /
    "pics_temporal_loso_cnn_fold_summary.csv"
)

cnn_metrics_path = (
    REPORTS_DIR /
    "pics_temporal_loso_cnn_metrics.csv"
)

cnn_per_infant_path = (
    REPORTS_DIR /
    "pics_temporal_loso_cnn_per_infant.csv"
)

cnn_positive_path = (
    REPORTS_DIR /
    "pics_temporal_loso_cnn_positive_window_analysis.csv"
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

cnn_loso_predictions.to_csv(
    cnn_predictions_path,
    index=False
)

cnn_fold_summaries.to_csv(
    cnn_fold_summary_path,
    index=False
)

cnn_loso_metrics.to_csv(
    cnn_metrics_path,
    index=False
)

cnn_per_infant.to_csv(
    cnn_per_infant_path,
    index=False
)

cnn_positive_window_analysis.to_csv(
    cnn_positive_path,
    index=False
)


# ------------------------------------------------------------
# Verify files exist
# ------------------------------------------------------------

assert cnn_predictions_path.exists()
assert cnn_fold_summary_path.exists()
assert cnn_metrics_path.exists()
assert cnn_per_infant_path.exists()
assert cnn_positive_path.exists()


# ------------------------------------------------------------
# Reload and independently verify
# ------------------------------------------------------------

cnn_predictions_check = pd.read_csv(
    cnn_predictions_path
)

cnn_fold_check = pd.read_csv(
    cnn_fold_summary_path
)

cnn_metrics_check = pd.read_csv(
    cnn_metrics_path
)

cnn_per_infant_check = pd.read_csv(
    cnn_per_infant_path
)

cnn_positive_check = pd.read_csv(
    cnn_positive_path
)


# ------------------------------------------------------------
# Structural checks
# ------------------------------------------------------------

assert len(
    cnn_predictions_check
) == 55

assert len(
    cnn_fold_check
) == 10

assert len(
    cnn_metrics_check
) == 1

assert len(
    cnn_per_infant_check
) == 10

assert len(
    cnn_positive_check
) == 5


# ------------------------------------------------------------
# Verify pooled metrics
# ------------------------------------------------------------

cnn_metrics_row = (
    cnn_metrics_check.iloc[0]
)

assert int(
    cnn_metrics_row["TN"]
) == tn_cnn

assert int(
    cnn_metrics_row["FP"]
) == fp_cnn

assert int(
    cnn_metrics_row["FN"]
) == fn_cnn

assert int(
    cnn_metrics_row["TP"]
) == tp_cnn

assert np.isclose(
    float(cnn_metrics_row["accuracy"]),
    accuracy_cnn
)

assert np.isclose(
    float(cnn_metrics_row["sensitivity"]),
    sensitivity_cnn
)

assert np.isclose(
    float(cnn_metrics_row["specificity"]),
    specificity_cnn
)

assert np.isclose(
    float(cnn_metrics_row["precision"]),
    precision_cnn
)

assert np.isclose(
    float(cnn_metrics_row["f1"]),
    f1_cnn
)

assert np.isclose(
    float(cnn_metrics_row["auroc"]),
    auroc_cnn
)

assert np.isclose(
    float(cnn_metrics_row["auprc"]),
    auprc_cnn
)


# ------------------------------------------------------------
# Verify positive-window analysis
# ------------------------------------------------------------

assert (
    (
        cnn_positive_check["result"]
        == "TP"
    ).sum()
    == 0
)

assert (
    (
        cnn_positive_check["result"]
        == "FN"
    ).sum()
    == 5
)


# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------

print("=" * 70)
print("1D-CNN RESULTS SAVED")
print("=" * 70)

print("\nSaved files:")

print(
    "  Predictions:",
    cnn_predictions_path
)

print(
    "  Fold summary:",
    cnn_fold_summary_path
)

print(
    "  Metrics:",
    cnn_metrics_path
)

print(
    "  Per-infant:",
    cnn_per_infant_path
)

print(
    "  Positive windows:",
    cnn_positive_path
)

print("\nVerified:")
print(
    "  OOF predictions:",
    len(cnn_predictions_check)
)

print(
    "  LOSO folds:",
    len(cnn_fold_check)
)

print(
    "  Per-infant rows:",
    len(cnn_per_infant_check)
)

print(
    "  Positive-window rows:",
    len(cnn_positive_check)
)

print(
    "\nCNN metrics:",
    f"AUROC={auroc_cnn:.4f},",
    f"AUPRC={auprc_cnn:.4f},",
    f"Sensitivity={sensitivity_cnn:.4f}"
)

print(
    "\nPASS: All 1D-CNN results saved and independently verified."
)

print(
    "STATUS: Development / validation — "
    "not clinical validation."
)

1D-CNN RESULTS SAVED

Saved files:
  Predictions: c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_temporal_loso_cnn_predictions.csv
  Fold summary: c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_temporal_loso_cnn_fold_summary.csv
  Metrics: c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_temporal_loso_cnn_metrics.csv
  Per-infant: c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_temporal_loso_cnn_per_infant.csv
  Positive windows: c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_temporal_loso_cnn_positive_window_analysis.csv

Verified:
  OOF predictions: 55
  LOSO folds: 10
  Per-infant rows: 10
  Positive-window rows: 5

CNN metrics: AUROC=0.5880, AUPRC=0.1389, Sensitivity=0.0000

PASS: All 1D-CNN results saved and independently verified.
STATUS: Development / validation — not clinical validation.


In [103]:
# ============================================================
# CELL 46 — Final Stage 4 Temporal Modeling Checkpoint
#
# Purpose:
#   Create the final Stage 4 temporal-modeling checkpoint
#   containing:
#
#       - Dataset configuration
#       - LOSO validation policy
#       - Temporal Logistic Regression
#       - GRU
#       - 1D-CNN
#       - Positive-window analysis
#       - Methodological limitations
#
# Status:
#   Development / validation — not clinical validation
# ============================================================

from datetime import datetime


# ------------------------------------------------------------
# Check required results exist
# ------------------------------------------------------------

assert "cnn_loso_metrics" in globals()
assert "cnn_per_infant" in globals()
assert "cnn_positive_window_analysis" in globals()

assert "cnn_loso_predictions" in globals()
assert len(cnn_loso_predictions) == 55

assert len(cnn_loso_metrics) == 1
assert len(cnn_per_infant) == 10
assert len(cnn_positive_window_analysis) == 5


# ------------------------------------------------------------
# Extract CNN metrics
# ------------------------------------------------------------

cnn_metrics_row = (
    cnn_loso_metrics.iloc[0]
)


# ------------------------------------------------------------
# Build final Stage 4 temporal checkpoint
# ------------------------------------------------------------

stage4_temporal_final_checkpoint = {

    "project_stage": "Stage 4 — Temporal Risk Modeling",

    "checkpoint_type": (
        "development_validation"
    ),

    "clinical_validation": False,

    "dataset": {
        "total_windows": 55,
        "control_windows": 50,
        "precursor_windows": 5,
        "positive_bearing_infants": [1, 3, 9],
        "total_infants": 10,
        "sequence_shape": [11, 20],
        "features_per_timestep": 20,
        "timesteps": 11
    },

    "label_definition": {
        "risk_0": "control",
        "risk_1": "15-second precursor window",
        "positive_windows": 5,
        "negative_windows": 50
    },

    "validation": {
        "method": "Leave-One-Subject-Out",
        "folds": 10,
        "subject_leakage": "PASS",
        "test_infant_excluded_from_training": True,
        "preprocessing_training_fold_only": True,
        "imputation_training_fold_only": True,
        "scaling_training_fold_only": True,
        "temporal_window_order_verified": True
    },

    "temporal_logistic": {
        "accuracy": 0.8909,
        "sensitivity": 0.4000,
        "specificity": 0.9400,
        "precision": 0.4000,
        "f1": 0.4000,
        "auroc": 0.8240,
        "auprc": 0.3049,
        "confusion_matrix": [
            [47, 3],
            [3, 2]
        ],
        "status": "development_validation"
    },

    "gru": {
        "architecture": {
            "input_shape": [11, 20],
            "gru_units": 16,
            "dropout": 0.20,
            "dense_units": 8,
            "output": "sigmoid"
        },
        "training": {
            "epochs": 25,
            "batch_size": 8,
            "learning_rate": 0.0005,
            "threshold": 0.50,
            "class_weighting": "balanced",
            "validation_split": False,
            "early_stopping": False,
            "fresh_model_each_fold": True
        },
        "accuracy": 0.6000,
        "sensitivity": 0.6000,
        "specificity": 0.6000,
        "precision": 0.13043478,
        "f1": 0.21428571,
        "auroc": 0.7160,
        "auprc": 0.32918741,
        "confusion_matrix": [
            [30, 20],
            [2, 3]
        ],
        "positive_windows": 5,
        "true_positive_windows": 3,
        "false_negative_windows": 2,
        "positive_window_sensitivity": 0.60,
        "status": "development_validation"
    },

    "cnn": {
        "architecture": {
            "input_shape": [11, 20],
            "conv1d_filters": [16, 32],
            "kernel_size": 3,
            "global_average_pooling": True,
            "dropout": 0.20,
            "dense_units": 8,
            "output": "sigmoid"
        },
        "training": {
            "epochs": 25,
            "batch_size": 8,
            "learning_rate": 0.0005,
            "threshold": 0.50,
            "class_weighting": "balanced",
            "validation_split": False,
            "early_stopping": False,
            "fresh_model_each_fold": True
        },
        "accuracy": float(
            cnn_metrics_row["accuracy"]
        ),
        "sensitivity": float(
            cnn_metrics_row["sensitivity"]
        ),
        "specificity": float(
            cnn_metrics_row["specificity"]
        ),
        "precision": float(
            cnn_metrics_row["precision"]
        ),
        "f1": float(
            cnn_metrics_row["f1"]
        ),
        "auroc": float(
            cnn_metrics_row["auroc"]
        ),
        "auprc": float(
            cnn_metrics_row["auprc"]
        ),
        "confusion_matrix": [
            [
                int(cnn_metrics_row["TN"]),
                int(cnn_metrics_row["FP"])
            ],
            [
                int(cnn_metrics_row["FN"]),
                int(cnn_metrics_row["TP"])
            ]
        ],
        "positive_windows": 5,
        "true_positive_windows": 0,
        "false_negative_windows": 5,
        "positive_window_sensitivity": 0.0,
        "status": "development_validation"
    },

    "positive_window_analysis": {
        "total_precursor_windows": 5,
        "cnn_true_positive": 0,
        "cnn_false_negative": 5,
        "cnn_positive_window_sensitivity": 0.0,
        "analysis_file": (
            "pics_temporal_loso_cnn_positive_window_analysis.csv"
        )
    },

    "saved_artifacts": {
        "temporal_dataset": (
            "pics_final_temporal_X_sequences.npy"
        ),
        "temporal_labels": (
            "pics_final_temporal_y_sequences.npy"
        ),
        "temporal_metadata": (
            "pics_final_temporal_sequence_metadata.csv"
        ),

        "logistic_predictions": (
            "pics_temporal_loso_logistic_predictions.csv"
        ),
        "logistic_metrics": (
            "pics_temporal_loso_logistic_metrics.csv"
        ),

        "gru_predictions": (
            "pics_temporal_loso_gru_predictions.csv"
        ),
        "gru_fold_summary": (
            "pics_temporal_loso_gru_fold_summary.csv"
        ),
        "gru_metrics": (
            "pics_temporal_loso_gru_metrics.csv"
        ),
        "gru_per_infant": (
            "pics_temporal_loso_gru_per_infant.csv"
        ),
        "gru_positive_windows": (
            "pics_temporal_loso_gru_positive_window_analysis.csv"
        ),

        "cnn_predictions": (
            "pics_temporal_loso_cnn_predictions.csv"
        ),
        "cnn_fold_summary": (
            "pics_temporal_loso_cnn_fold_summary.csv"
        ),
        "cnn_metrics": (
            "pics_temporal_loso_cnn_metrics.csv"
        ),
        "cnn_per_infant": (
            "pics_temporal_loso_cnn_per_infant.csv"
        ),
        "cnn_positive_windows": (
            "pics_temporal_loso_cnn_positive_window_analysis.csv"
        )
    },

    "limitations": [
        "Only five positive precursor windows are available.",
        "Positive windows occur in only three infants.",
        "Results are development/validation results.",
        "The dataset is not sufficient for clinical validation.",
        "No clinical diagnosis is claimed.",
        "Fixed threshold results should not be interpreted as threshold optimization.",
        "Model selection is descriptive rather than a clinical performance ranking."
    ],

    "next_stage": (
        "Deployment preparation and edge inference"
    ),

    "generated_at": datetime.now().isoformat()
}


# ------------------------------------------------------------
# Save checkpoint
# ------------------------------------------------------------

stage4_temporal_checkpoint_path = (
    REPORTS_DIR /
    "pics_stage4_temporal_modeling_final_checkpoint.json"
)

with open(
    stage4_temporal_checkpoint_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        stage4_temporal_final_checkpoint,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Reload and independently verify
# ------------------------------------------------------------

with open(
    stage4_temporal_checkpoint_path,
    "r",
    encoding="utf-8"
) as f:

    checkpoint_check = json.load(f)


# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert (
    checkpoint_check["dataset"]["total_windows"]
    == 55
)

assert (
    checkpoint_check["dataset"]["control_windows"]
    == 50
)

assert (
    checkpoint_check["dataset"]["precursor_windows"]
    == 5
)

assert (
    checkpoint_check["validation"]["folds"]
    == 10
)

assert (
    checkpoint_check["validation"]["subject_leakage"]
    == "PASS"
)

assert (
    checkpoint_check["gru"]["true_positive_windows"]
    == 3
)

assert (
    checkpoint_check["gru"]["false_negative_windows"]
    == 2
)

assert (
    checkpoint_check["cnn"]["true_positive_windows"]
    == 0
)

assert (
    checkpoint_check["cnn"]["false_negative_windows"]
    == 5
)

assert (
    checkpoint_check["temporal_logistic"]["auroc"]
    == 0.8240
)

assert (
    checkpoint_check["gru"]["auroc"]
    == 0.7160
)

assert (
    checkpoint_check["cnn"]["auroc"]
    == 0.5880
)


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("=" * 70)
print("FINAL STAGE 4 TEMPORAL MODELING CHECKPOINT")
print("=" * 70)

print("\nDataset:")
print(
    "  Total windows     :",
    checkpoint_check["dataset"]["total_windows"]
)

print(
    "  Control windows   :",
    checkpoint_check["dataset"]["control_windows"]
)

print(
    "  Precursor windows :",
    checkpoint_check["dataset"]["precursor_windows"]
)

print(
    "  Infants           :",
    checkpoint_check["dataset"]["total_infants"]
)

print(
    "  LOSO folds        :",
    checkpoint_check["validation"]["folds"]
)

print("\nTemporal models:")

print(
    "  Logistic AUROC:",
    checkpoint_check[
        "temporal_logistic"
    ]["auroc"]
)

print(
    "  GRU AUROC     :",
    checkpoint_check[
        "gru"
    ]["auroc"]
)

print(
    "  CNN AUROC     :",
    checkpoint_check[
        "cnn"
    ]["auroc"]
)

print("\nPositive-window detection:")

print(
    "  GRU:",
    checkpoint_check[
        "gru"
    ]["true_positive_windows"],
    "/ 5"
)

print(
    "  CNN:",
    checkpoint_check[
        "cnn"
    ]["true_positive_windows"],
    "/ 5"
)

print(
    "\nCheckpoint:",
    stage4_temporal_checkpoint_path
)

print(
    "\nPASS: Final Stage 4 temporal checkpoint "
    "saved and independently verified."
)

print(
    "STATUS: Development / validation — "
    "not clinical validation."
)

FINAL STAGE 4 TEMPORAL MODELING CHECKPOINT

Dataset:
  Total windows     : 55
  Control windows   : 50
  Precursor windows : 5
  Infants           : 10
  LOSO folds        : 10

Temporal models:
  Logistic AUROC: 0.824
  GRU AUROC     : 0.716
  CNN AUROC     : 0.588

Positive-window detection:
  GRU: 3 / 5
  CNN: 0 / 5

Checkpoint: c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor\reports\pics_stage4_temporal_modeling_final_checkpoint.json

PASS: Final Stage 4 temporal checkpoint saved and independently verified.
STATUS: Development / validation — not clinical validation.


In [105]:
# ============================================================
# CELL 47 — Stage 5 Deployment Configuration
# CORRECTED VERSION
#
# Purpose:
#   Freeze the deployment interface and reference model.
#
# IMPORTANT:
#   No model retraining.
#   No quantization.
#   No clinical validation.
# ============================================================

print("=" * 70)
print("STAGE 5 — DEPLOYMENT CONFIGURATION")
print("=" * 70)


# ------------------------------------------------------------
# Deployment input specification
# ------------------------------------------------------------

DEPLOYMENT_TIMESTEPS = 11
DEPLOYMENT_FEATURES = 20
DEPLOYMENT_WINDOW_SECONDS = 15.0

DEPLOYMENT_THRESHOLD = 0.50

DEPLOYMENT_MODEL_NAME = (
    "Temporal Logistic Regression"
)


# ------------------------------------------------------------
# Exact feature order
# ------------------------------------------------------------

DEPLOYMENT_FEATURE_NAMES = [

    # ECG
    "ecg_mean",
    "ecg_std",
    "ecg_rms",
    "r_peak_count",
    "mean_rr",
    "std_rr",
    "mean_hr",
    "min_hr",
    "max_hr",
    "std_hr",
    "sdnn",
    "rmssd",

    # Respiration
    "resp_mean",
    "resp_std",
    "resp_rms",
    "resp_peak_count",
    "mean_resp_interval",
    "std_resp_interval",
    "mean_resp_rate",
    "std_resp_rate"
]


# ------------------------------------------------------------
# Deployment preprocessing policy
# ------------------------------------------------------------

DEPLOYMENT_PREPROCESSING = {
    "imputation": "median",
    "scaling": "StandardScaler",
    "fit_scope": "training data only",
    "sequence_shape": [11, 20],
    "missing_values": (
        "training-fold-derived imputer"
    )
}


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

assert DEPLOYMENT_TIMESTEPS == 11
assert DEPLOYMENT_FEATURES == 20
assert DEPLOYMENT_WINDOW_SECONDS == 15.0
assert DEPLOYMENT_THRESHOLD == 0.50

assert len(
    DEPLOYMENT_FEATURE_NAMES
) == 20


# ------------------------------------------------------------
# Verify temporal dataset
# ------------------------------------------------------------

temporal_X_path = (
    REPORTS_DIR /
    "pics_final_temporal_X_sequences.npy"
)

temporal_y_path = (
    REPORTS_DIR /
    "pics_final_temporal_y_sequences.npy"
)

assert temporal_X_path.exists()
assert temporal_y_path.exists()

X_temporal_deployment = np.load(
    temporal_X_path
)

y_temporal_deployment = np.load(
    temporal_y_path
)

assert X_temporal_deployment.shape == (
    55,
    11,
    20
)

assert y_temporal_deployment.shape == (
    55,
)


# ------------------------------------------------------------
# Verify final feature dataset
# ------------------------------------------------------------

feature_dataset_path = (
    REPORTS_DIR /
    "pics_final_precursor_risk_dataset.csv"
)

assert feature_dataset_path.exists()

feature_dataset_check = pd.read_csv(
    feature_dataset_path
)

assert feature_dataset_check.shape == (
    55,
    29
)


# ------------------------------------------------------------
# Verify Logistic LOSO artifacts
# ------------------------------------------------------------

logistic_predictions_path = (
    REPORTS_DIR /
    "pics_temporal_loso_logistic_predictions.csv"
)

logistic_metrics_path = (
    REPORTS_DIR /
    "pics_temporal_loso_logistic_metrics.csv"
)

assert logistic_predictions_path.exists()
assert logistic_metrics_path.exists()


# ------------------------------------------------------------
# Load actual saved Logistic metrics
# ------------------------------------------------------------

logistic_metrics_check = pd.read_csv(
    logistic_metrics_path
)

assert len(
    logistic_metrics_check
) == 1

logistic_row = (
    logistic_metrics_check.iloc[0]
)


# ------------------------------------------------------------
# Store ACTUAL saved values
# ------------------------------------------------------------

REFERENCE_AUROC = float(
    logistic_row["auroc"]
)

REFERENCE_AUPRC = float(
    logistic_row["auprc"]
)

REFERENCE_ACCURACY = float(
    logistic_row["accuracy"]
)

REFERENCE_SENSITIVITY = float(
    logistic_row["sensitivity"]
)

REFERENCE_SPECIFICITY = float(
    logistic_row["specificity"]
)


# ------------------------------------------------------------
# Deployment status
# ------------------------------------------------------------

DEPLOYMENT_STATUS = (
    "development / edge inference prototype"
)

CLINICAL_VALIDATION = False


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nDeployment input:")
print(
    "  Window duration :",
    DEPLOYMENT_WINDOW_SECONDS,
    "seconds"
)

print(
    "  Timesteps       :",
    DEPLOYMENT_TIMESTEPS
)

print(
    "  Features/step   :",
    DEPLOYMENT_FEATURES
)

print(
    "  Total inputs    :",
    DEPLOYMENT_TIMESTEPS *
    DEPLOYMENT_FEATURES
)

print("\nReference model:")
print(
    "  Model           :",
    DEPLOYMENT_MODEL_NAME
)

print(
    "  Threshold       :",
    DEPLOYMENT_THRESHOLD
)

print("\nPreprocessing:")
print(
    "  Imputation      :",
    DEPLOYMENT_PREPROCESSING[
        "imputation"
    ]
)

print(
    "  Scaling         :",
    DEPLOYMENT_PREPROCESSING[
        "scaling"
    ]
)

print(
    "  Fit scope       :",
    DEPLOYMENT_PREPROCESSING[
        "fit_scope"
    ]
)

print("\nACTUAL SAVED Logistic LOSO METRICS:")

print(
    f"  AUROC           : {REFERENCE_AUROC:.10f}"
)

print(
    f"  AUPRC           : {REFERENCE_AUPRC:.10f}"
)

print(
    f"  Accuracy        : {REFERENCE_ACCURACY:.10f}"
)

print(
    f"  Sensitivity     : {REFERENCE_SENSITIVITY:.10f}"
)

print(
    f"  Specificity     : {REFERENCE_SPECIFICITY:.10f}"
)

print("\nFeature order:")
for i, feature in enumerate(
    DEPLOYMENT_FEATURE_NAMES,
    start=1
):
    print(
        f"  {i:02d}. {feature}"
    )

print("\nStatus:")
print(
    " ",
    DEPLOYMENT_STATUS
)

print(
    "  Clinical validation:",
    CLINICAL_VALIDATION
)


# ------------------------------------------------------------
# Tolerance-based verification
#
# The displayed Stage 4 metrics were rounded.
# We verify against the saved CSV without requiring
# exact floating-point equality.
# ------------------------------------------------------------

assert np.isclose(
    REFERENCE_AUROC,
    0.8240,
    atol=1e-3
)

assert np.isclose(
    REFERENCE_AUPRC,
    0.3049,
    atol=1e-3
)

assert np.isclose(
    REFERENCE_ACCURACY,
    0.8909,
    atol=1e-3
)

assert np.isclose(
    REFERENCE_SENSITIVITY,
    0.4000,
    atol=1e-3
)

assert np.isclose(
    REFERENCE_SPECIFICITY,
    0.9400,
    atol=1e-3
)


# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

assert X_temporal_deployment.shape == (
    55,
    11,
    20
)

assert y_temporal_deployment.shape == (
    55,
)

assert len(
    DEPLOYMENT_FEATURE_NAMES
) == 20

print(
    "\nPASS: Deployment input specification frozen."
)

print(
    "PASS: Exact 20-feature order frozen."
)

print(
    "PASS: Temporal dataset verified."
)

print(
    "PASS: Logistic LOSO artifacts verified."
)

print(
    "PASS: Saved metrics verified with tolerance."
)

print(
    "PASS: No model retraining performed."
)

print(
    "STATUS: Stage 5 deployment preparation."
)

STAGE 5 — DEPLOYMENT CONFIGURATION

Deployment input:
  Window duration : 15.0 seconds
  Timesteps       : 11
  Features/step   : 20
  Total inputs    : 220

Reference model:
  Model           : Temporal Logistic Regression
  Threshold       : 0.5

Preprocessing:
  Imputation      : median
  Scaling         : StandardScaler
  Fit scope       : training data only

ACTUAL SAVED Logistic LOSO METRICS:
  AUROC           : 0.8240000000
  AUPRC           : 0.3049257760
  Accuracy        : 0.8909090909
  Sensitivity     : 0.4000000000
  Specificity     : 0.9400000000

Feature order:
  01. ecg_mean
  02. ecg_std
  03. ecg_rms
  04. r_peak_count
  05. mean_rr
  06. std_rr
  07. mean_hr
  08. min_hr
  09. max_hr
  10. std_hr
  11. sdnn
  12. rmssd
  13. resp_mean
  14. resp_std
  15. resp_rms
  16. resp_peak_count
  17. mean_resp_interval
  18. std_resp_interval
  19. mean_resp_rate
  20. std_resp_rate

Status:
  development / edge inference prototype
  Clinical validation: False

PASS: Deployme

In [ ]:
\# ============================================================
# CELL 48 — Full-Development Reference Logistic Model
# ============================================================

import joblib
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("=" * 70)
print("STAGE 5 — FULL-DEVELOPMENT REFERENCE MODEL")
print("=" * 70)


# ------------------------------------------------------------
# Load authoritative temporal dataset
# ------------------------------------------------------------

X_deployment_raw = np.load(
    REPORTS_DIR / "pics_final_temporal_X_sequences.npy"
)

y_deployment = np.load(
    REPORTS_DIR / "pics_final_temporal_y_sequences.npy"
)

assert X_deployment_raw.shape == (55, 11, 20)
assert y_deployment.shape == (55,)

assert int(y_deployment.sum()) == 5
assert int((y_deployment == 0).sum()) == 50

print("\nDataset:")
print("  Windows       :", len(y_deployment))
print("  Timesteps     :", X_deployment_raw.shape[1])
print("  Features/step :", X_deployment_raw.shape[2])
print("  Positive      :", int(y_deployment.sum()))
print("  Negative      :", int((y_deployment == 0).sum()))


# ------------------------------------------------------------
# Reshape for timestep-level preprocessing
# ------------------------------------------------------------

n_windows = X_deployment_raw.shape[0]
n_timesteps = X_deployment_raw.shape[1]
n_features = X_deployment_raw.shape[2]

X_timestep = X_deployment_raw.reshape(
    n_windows * n_timesteps,
    n_features
)

print("\nPreprocessing shape:", X_timestep.shape)


# ------------------------------------------------------------
# Median imputation
# ------------------------------------------------------------

deployment_imputer = SimpleImputer(
    strategy="median"
)

X_imputed = deployment_imputer.fit_transform(
    X_timestep
)

assert X_imputed.shape == (605, 20)
assert np.isfinite(X_imputed).all()

print(
    "Missing values after imputation:",
    int(np.isnan(X_imputed).sum())
)


# ------------------------------------------------------------
# Standard scaling
# ------------------------------------------------------------

deployment_scaler = StandardScaler()

X_scaled = deployment_scaler.fit_transform(
    X_imputed
)

assert X_scaled.shape == (605, 20)
assert np.isfinite(X_scaled).all()


# ------------------------------------------------------------
# Restore temporal structure
# ------------------------------------------------------------

X_scaled_sequence = X_scaled.reshape(
    n_windows,
    n_timesteps,
    n_features
)

assert X_scaled_sequence.shape == (55, 11, 20)


# ------------------------------------------------------------
# Flatten to 220 inputs for Logistic Regression
# ------------------------------------------------------------

X_logistic = X_scaled_sequence.reshape(
    n_windows,
    n_timesteps * n_features
)

assert X_logistic.shape == (55, 220)

print(
    "Logistic input shape:",
    X_logistic.shape
)


# ------------------------------------------------------------
# Train full-development reference model
#
# This is a deployment-training model.
# It is NOT a validation result.
# ------------------------------------------------------------

deployment_logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=RANDOM_SEED
)

deployment_logistic_model.fit(
    X_logistic,
    y_deployment
)

assert deployment_logistic_model.coef_.shape == (1, 220)


# ------------------------------------------------------------
# Training-set probabilities
#
# These are NOT validation metrics.
# ------------------------------------------------------------

deployment_probabilities = (
    deployment_logistic_model.predict_proba(
        X_logistic
    )[:, 1]
)

deployment_predictions = (
    deployment_probabilities >= DEPLOYMENT_THRESHOLD
).astype(int)

assert len(deployment_probabilities) == 55
assert np.isfinite(deployment_probabilities).all()
assert np.all(
    (deployment_probabilities >= 0)
    & (deployment_probabilities <= 1)
)


# ------------------------------------------------------------
# Build deployment bundle
# ------------------------------------------------------------

deployment_bundle = {
    "model_name": DEPLOYMENT_MODEL_NAME,
    "model": deployment_logistic_model,
    "imputer": deployment_imputer,
    "scaler": deployment_scaler,
    "feature_names": DEPLOYMENT_FEATURE_NAMES,
    "timesteps": 11,
    "features_per_timestep": 20,
    "window_seconds": 15.0,
    "threshold": DEPLOYMENT_THRESHOLD,
    "input_features": 220,
    "preprocessing": DEPLOYMENT_PREPROCESSING,
    "training_windows": 55,
    "training_positive": 5,
    "training_negative": 50,
    "validation_status": (
        "Reference deployment model; "
        "not a validation result."
    ),
    "clinical_validation": False
}


# ------------------------------------------------------------
# Save bundle
# ------------------------------------------------------------

deployment_bundle_path = (
    REPORTS_DIR /
    "pics_stage5_logistic_deployment_bundle.joblib"
)

joblib.dump(
    deployment_bundle,
    deployment_bundle_path
)

assert deployment_bundle_path.exists()


# ------------------------------------------------------------
# Reload and verify
# ------------------------------------------------------------

deployment_bundle_check = joblib.load(
    deployment_bundle_path
)

assert deployment_bundle_check["timesteps"] == 11
assert deployment_bundle_check["features_per_timestep"] == 20
assert deployment_bundle_check["input_features"] == 220

assert len(
    deployment_bundle_check["feature_names"]
) == 20

assert deployment_bundle_check["model"].coef_.shape == (
    1,
    220
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("REFERENCE DEPLOYMENT MODEL READY")
print("=" * 70)

print(
    "\nModel:",
    deployment_bundle_check["model_name"]
)

print(
    "Input shape:",
    (
        deployment_bundle_check["timesteps"],
        deployment_bundle_check["features_per_timestep"]
    )
)

print(
    "Flattened inputs:",
    deployment_bundle_check["input_features"]
)

print(
    "Threshold:",
    deployment_bundle_check["threshold"]
)

print(
    "Training windows:",
    deployment_bundle_check["training_windows"]
)

print(
    "Deployment bundle:",
    deployment_bundle_path
)

print(
    "\nNOTE: Training-set predictions are NOT validation metrics."
)

print("\nPASS: Full-development reference model trained.")
print("PASS: Imputer fitted and verified.")
print("PASS: StandardScaler fitted and verified.")
print("PASS: 220-input Logistic model verified.")
print("PASS: Deployment bundle saved and reloaded.")
print("STATUS: Stage 5 deployment preparation.")

SyntaxError: unexpected character after line continuation character (1903188283.py, line 1)